<a href="https://colab.research.google.com/github/hjchoipt-byte/olympic-medal-outcome-ml/blob/main/Olympic_Medal_ML_Public_Reproducibility_Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Predicting Olympic Medal Outcomes Using Multimodal Machine Learning

## Public Reproducibility Code

This notebook contains the computational workflow used for data preprocessing, feature screening, forward feature selection, nested cross-validation, SHAP interpretation, and supplementary sensitivity analyses.

The athlete-level dataset is not publicly distributed because of institutional restrictions concerning athlete health data.

# 1.Environment and Global Configuration

This section installs the required packages, imports the analysis libraries, defines the file paths, and specifies the analysis settings reported in the manuscript and Supplementary Materials.

### 1.1.Google Drive and Package Installation

Install the required Python packages and mount Google Drive.

In [ ]:
# ================================================================
# 1.1 Google Drive and Package Installation
# ================================================================

from google.colab import drive
drive.mount("/content/drive")

!pip install -q pandas numpy scipy scikit-learn imbalanced-learn statsmodels \
    matplotlib shap catboost xgboost lightgbm openpyxl xlsxwriter

##1.2.Library Imports

Import all libraries required for data preprocessing, feature selection, machine learning, model interpretation, statistical analyses, and visualization.

In [ ]:
# ================================================================
# 1.2 Library Imports
# ================================================================

from __future__ import annotations

import json, math, os, pickle, random, warnings
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
import statsmodels.api as sm

from scipy.stats import chi2_contingency, fisher_exact, norm
from statsmodels.stats.outliers_influence import variance_inflation_factor

from sklearn.base import clone
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, brier_score_loss,
    classification_report, confusion_matrix, f1_score,
    precision_recall_curve, precision_score, recall_score,
    roc_auc_score, roc_curve
)
from sklearn.model_selection import (
    RandomizedSearchCV, RepeatedStratifiedKFold,
    StratifiedKFold, cross_val_predict
)
from sklearn.pipeline import Pipeline as SklearnPipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbalancedPipeline

from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore")

print("All required libraries were imported successfully.")

##1.3.Reproducibility Settings

Define the random seed to ensure reproducible analyses.

In [ ]:
# ================================================================
# 1.3 Reproducibility Settings
# ================================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

print(f"Random seed: {SEED}")

##1.4.File Paths and Output Directories

Specify the input dataset and create the directory structure used to save all analysis outputs.

In [ ]:
# ================================================================
# 1.4 File Paths and Output Directories
# ================================================================

BASE_DIR = Path("/content/drive/MyDrive/PhD_degree_ML")
DATA_PATH = BASE_DIR / "International_Sports_Data.csv"
OUTPUT_DIR = BASE_DIR / "Olympic_Medal_ML_Public_Code_Results"

OUTPUT_FOLDERS = {
    "configuration": OUTPUT_DIR / "00_Configuration",
    "descriptive": OUTPUT_DIR / "01_Descriptive_Analyses",
    "vif": OUTPUT_DIR / "02_VIF_Screening",
    "ffs": OUTPUT_DIR / "03_Forward_Feature_Selection",
    "shap": OUTPUT_DIR / "04_SHAP_Analyses",
    "nested_cv": OUTPUT_DIR / "05_Main_Nested_CV",
    "complete_case": OUTPUT_DIR / "06_Complete_Case_Sensitivity",
    "no_sd": OUTPUT_DIR / "07_Sports_Discipline_Excluded",
    "supplementary": OUTPUT_DIR / "08_Supplementary_Analyses",
    "figures": OUTPUT_DIR / "09_Figures",
    "models": OUTPUT_DIR / "10_Saved_Models",
    "tables": OUTPUT_DIR / "11_Consolidated_Tables"
}

for folder in OUTPUT_FOLDERS.values():
    folder.mkdir(parents=True, exist_ok=True)

print(f"Data path: {DATA_PATH}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Data file exists: {DATA_PATH.exists()}")

##1.5.Outcomes, Algorithms, and Selected Models

Define the binary medal outcomes, candidate machine-learning algorithms, and the final models reported in the manuscript.

In [ ]:
# ================================================================
# 1.5 Outcomes, Algorithms, and Selected Models
# ================================================================

TARGETS = ["GMA", "SMA", "BMA", "OMA"]
ALGORITHMS = ["CatBoost", "XGBoost", "LightGBM"]

SELECTED_MODELS = {
    "GMA": "CatBoost",
    "SMA": "CatBoost",
    "BMA": "CatBoost",
    "OMA": "XGBoost"
}

print("Targets:", TARGETS)
print("Algorithms:", ALGORITHMS)
print("Selected outcome-specific models:", SELECTED_MODELS)

##1.6.Cross-Validation and Sensitivity Settings

Specify the cross-validation design, SMOTE settings, calibration settings, and sensitivity-analysis parameters.

In [ ]:
# ================================================================
# 1.6 Cross-Validation and Sensitivity Settings
# ================================================================

FFS_N_SPLITS, FFS_N_REPEATS = 5, 10
OUTER_CV_SPLITS, INNER_CV_SPLITS = 5, 3

SMOTE_K_NEIGHBORS = 3
FIXED_CLASSIFICATION_THRESHOLD = 0.50
VIF_CUTOFF = 10.0
BOOTSTRAP_REPETITIONS = 2000
CALIBRATION_BINS = 10
TMPS_CUTOFF = 14
THEORETICAL_TMPS_MAXIMUM = 21

print(f"FFS: {FFS_N_SPLITS}-fold CV × {FFS_N_REPEATS} repeats")
print(f"Nested CV: {OUTER_CV_SPLITS} outer folds × {INNER_CV_SPLITS} inner folds")
print(f"SMOTE k-neighbors: {SMOTE_K_NEIGHBORS}")
print(f"VIF cutoff: {VIF_CUTOFF}")
print(f"Bootstrap repetitions: {BOOTSTRAP_REPETITIONS}")

##1.7.Cross-Validation and Sensitivity Settings

Specify the cross-validation design, SMOTE settings, calibration settings, and sensitivity-analysis parameters.

In [ ]:
# ================================================================
# 1.7 Manuscript-Aligned Analysis Structure
# ================================================================

ANALYSIS_STRUCTURE = {
    "Feature screening": "Iterative VIF screening before FFS",
    "FFS": "Independent repeated stratified 5-fold CV with 10 repeats",
    "SHAP": "FFS peak subset refitted on the full available dataset",
    "Main nested CV": "Independent evaluation using the post-VIF candidate feature set",
    "Complete case": "Nested-CV sensitivity for the selected outcome-specific models",
    "No-SD analysis": "Remove SD, repeat FFS, and evaluate the no-SD peak subset"
}

for analysis, description in ANALYSIS_STRUCTURE.items():
    print(f"{analysis}: {description}")

##1.8.Movement Performance Score Variables

Define the Movement Performance Score (MPS) variables used throughout the analyses.

In [ ]:
# ================================================================
# 1.8 Movement Performance Score Variables
# ================================================================

MPS_FEATURES = [
    "DSMPS", "HSMPS", "ILMPS", "SMPS",
    "SLRMPS", "TSPMPS", "RSMPS", "TMPS"
]

print(f"Number of MPS variables: {len(MPS_FEATURES)}")
print(MPS_FEATURES)

##1.9.Candidate Predictor Variables

Specify all candidate predictor variables before multicollinearity screening.

In [ ]:
# ================================================================
# 1.9 Candidate Predictor Variables
# ================================================================

DEMOGRAPHIC_FEATURES = ["Gender", "Age", "Height", "Weight", "BMI"]
SPORT_FEATURES = ["SD"]
PHYSIOTHERAPY_COUNT_FEATURES = ["TA", "TC", "TAcq", "TCnt", "TotC"]
TREATMENT_PURPOSE_FEATURES = ["PP", "RP", "IMP"]
TREATMENT_CAUSE_FEATURES = ["COI", "CPI", "CFI", "CCI", "CNI", "CEI"]

TREATED_AREA_FEATURES = [
    "Femoral", "Neck", "Back", "Shoulder", "Lumbar", "Pelvis", "Arm",
    "Elbow", "Wrist", "Hand", "Finger", "Hip", "Hip Joint", "Knee",
    "Popliteus", "Tibia", "Achilles", "Ankle", "Foot", "Toe", "Chest",
    "Side", "Abdomen", "Calf", "Hamstring"
]

TAPED_AREA_FEATURES = [
    "T-Femoral", "T-Neck", "T-Back", "T-Shoulder", "T-Lumbar",
    "T-Pelvis", "T-Arm", "T-Elbow", "T-Wrist", "T-Hand", "T-Finger",
    "T-Hip", "T-Hip joint", "T-Knee", "T-Popliteus", "T-Tibia",
    "T-Achilles", "T-Ankle", "T-Foot", "T-Toe", "T-Chest", "T-Side",
    "T-Abdomen", "T-Calf", "T-Hamstring"
]

CANDIDATE_FEATURES = (
    DEMOGRAPHIC_FEATURES + SPORT_FEATURES + PHYSIOTHERAPY_COUNT_FEATURES +
    TREATMENT_PURPOSE_FEATURES + TREATMENT_CAUSE_FEATURES +
    TREATED_AREA_FEATURES + TAPED_AREA_FEATURES + MPS_FEATURES
)

CANDIDATE_FEATURES = list(dict.fromkeys(CANDIDATE_FEATURES))

print(f"Specified candidate feature count: {len(CANDIDATE_FEATURES)}")

##1.10.Algorithm-Specific Hyperparameter Search Spaces

RandomizedSearchCV evaluates 16 randomly sampled configurations for CatBoost and LightGBM. All four available XGBoost configurations are evaluated because its finite search space contains fewer than 16 unique combinations.

In [ ]:
# ================================================================
# 1.10 Algorithm-Specific Hyperparameter Search Spaces
# ================================================================

SEARCH_SPACES = {
    "CatBoost": {
        "model__iterations": [200, 300],
        "model__depth": [4, 6],
        "model__learning_rate": [0.05, 0.10],
        "model__subsample": [0.80, 1.00],
        "model__l2_leaf_reg": [1.0, 3.0, 5.0],
        "model__random_strength": [1.0, 2.0],
        "model__bagging_temperature": [0.0, 1.0]
    },
    "XGBoost": {
        "model__n_estimators": [200, 300],
        "model__max_depth": [3],
        "model__learning_rate": [0.05, 0.10],
        "model__subsample": [0.80],
        "model__colsample_bytree": [0.80]
    },
    "LightGBM": {
        "model__n_estimators": [200, 300],
        "model__learning_rate": [0.05, 0.10],
        "model__num_leaves": [31, 63],
        "model__max_depth": [-1, 6],
        "model__subsample": [0.80],
        "model__colsample_bytree": [0.80],
        "model__min_child_samples": [20, 40],
        "model__reg_lambda": [0.0, 1.0],
        "model__reg_alpha": [0.0, 0.1]
    }
}

##1.11.Candidate Predictor Variables

Specify all candidate predictor variables before multicollinearity screening.

In [ ]:
# ================================================================
# 1.11 Search-Space Validation
# ================================================================

def count_parameter_combinations(parameter_space: dict) -> int:
    return int(np.prod([len(values) for values in parameter_space.values()]))

SEARCH_SPACE_SUMMARY = pd.DataFrame([
    {
        "Algorithm": algorithm,
        "Total_Combinations": count_parameter_combinations(SEARCH_SPACES[algorithm]),
        "Number_Evaluated": min(16, count_parameter_combinations(SEARCH_SPACES[algorithm]))
    }
    for algorithm in ALGORITHMS
])

display(SEARCH_SPACE_SUMMARY)

##1.12.Fixed Model Settings for Forward Feature Selection

Define the fixed model configurations used during forward feature selection.

In [ ]:
# ================================================================
# 1.12 Fixed Model Settings for Forward Feature Selection
# ================================================================

FFS_MODEL_PARAMETERS = {
    "CatBoost": {
        "loss_function": "Logloss", "eval_metric": "AUC",
        "iterations": 300, "learning_rate": 0.05, "depth": 6,
        "l2_leaf_reg": 3.0, "subsample": 0.80, "rsm": 0.80,
        "random_seed": SEED, "verbose": 0, "allow_writing_files": False
    },
    "XGBoost": {
        "n_estimators": 100, "objective": "binary:logistic",
        "eval_metric": "logloss", "random_state": SEED,
        "n_jobs": -1, "tree_method": "hist"
    },
    "LightGBM": {
        "objective": "binary", "n_estimators": 200,
        "learning_rate": 0.05, "num_leaves": 31,
        "subsample": 0.80, "colsample_bytree": 0.80,
        "reg_lambda": 1.0, "random_state": SEED,
        "n_jobs": -1, "verbosity": -1
    }
}

print("Fixed FFS model configurations were defined.")

##1.13.Save Analysis Configuration

Export the analysis configuration to a JSON file to improve reproducibility.

In [ ]:
# ================================================================
# 1.13 Save Analysis Configuration
# ================================================================

ANALYSIS_CONFIGURATION = {
    "random_seed": SEED,
    "targets": TARGETS,
    "algorithms": ALGORITHMS,
    "selected_models": SELECTED_MODELS,
    "ffs_cv": f"{FFS_N_SPLITS}-fold × {FFS_N_REPEATS} repeats",
    "nested_cv": f"{OUTER_CV_SPLITS} outer × {INNER_CV_SPLITS} inner",
    "smote_k_neighbors": SMOTE_K_NEIGHBORS,
    "fixed_classification_threshold": FIXED_CLASSIFICATION_THRESHOLD,
    "vif_cutoff": VIF_CUTOFF,
    "bootstrap_repetitions": BOOTSTRAP_REPETITIONS,
    "tmps_cutoff": TMPS_CUTOFF,
    "tmps_theoretical_maximum": THEORETICAL_TMPS_MAXIMUM,
    "analysis_structure": ANALYSIS_STRUCTURE
}

CONFIGURATION_FILE = OUTPUT_FOLDERS["configuration"] / "analysis_configuration.json"

with open(CONFIGURATION_FILE, "w", encoding="utf-8") as file:
    json.dump(ANALYSIS_CONFIGURATION, file, indent=2, ensure_ascii=False)

print(f"Configuration saved: {CONFIGURATION_FILE}")

##1.14.Configuration Summary

Summarize the analysis configuration and verify that the notebook is ready for the next section.

In [ ]:
# ================================================================
# 1.14 Section Completion Check
# ================================================================

SECTION_1_CHECK = pd.DataFrame([
    {"Item": "Data file exists", "Value": DATA_PATH.exists()},
    {"Item": "Output directory exists", "Value": OUTPUT_DIR.exists()},
    {"Item": "Candidate feature count", "Value": len(CANDIDATE_FEATURES)},
    {"Item": "Outcome count", "Value": len(TARGETS)},
    {"Item": "Algorithm count", "Value": len(ALGORITHMS)},
    {"Item": "Configuration file saved", "Value": CONFIGURATION_FILE.exists()}
])

display(SECTION_1_CHECK)

# 2.Data Loading and Integrity Checks

This section loads the athlete-level dataset, validates the sample size and variable structure, creates binary medal outcomes when necessary, and checks predictor availability, outcome coding, duplicate records, and missing values.

##2.1.Dataset Loading

The original athlete-level CSV file is loaded without modifying the source file. Binary medal outcomes are retained when already available and are otherwise derived from the corresponding medal-count variables.

In [ ]:
# ================================================================
# 2.1 Load Dataset and Create Binary Medal Outcomes
# ================================================================

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)
df.columns = df.columns.astype(str).str.strip()

MEDAL_COUNT_COLUMNS = {"GMA": "GMC", "SMA": "SMC", "BMA": "BMC", "OMA": "OMC"}

for target, count_column in MEDAL_COUNT_COLUMNS.items():
    if target not in df.columns:
        if count_column not in df.columns:
            raise ValueError(f"Neither '{target}' nor '{count_column}' was found.")
        medal_count = pd.to_numeric(df[count_column], errors="coerce")
        df[target] = np.where(medal_count > 0, 1, 0)

for target in TARGETS:
    df[target] = pd.to_numeric(df[target], errors="coerce")

print(f"Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]:,} columns")
print(f"Source file: {DATA_PATH}")

##2.2.Dataset Structure Validation

The dataset structure is checked against the expected cohort size and the variables specified in the manuscript. Missing and unexpected variables are reported rather than silently ignored.


In [ ]:
# ================================================================
# 2.2 Validate Sample Size and Column Structure
# ================================================================

EXPECTED_SAMPLE_SIZE = 1011

duplicate_column_names = df.columns[df.columns.duplicated()].tolist()
available_candidate_features = [c for c in CANDIDATE_FEATURES if c in df.columns]
missing_candidate_features = [c for c in CANDIDATE_FEATURES if c not in df.columns]
available_targets = [c for c in TARGETS if c in df.columns]
missing_targets = [c for c in TARGETS if c not in df.columns]

DATA_STRUCTURE_CHECK = pd.DataFrame([
    {"Check": "Observed sample size", "Expected": EXPECTED_SAMPLE_SIZE, "Observed": len(df), "Passed": len(df) == EXPECTED_SAMPLE_SIZE},
    {"Check": "Duplicate column names", "Expected": 0, "Observed": len(duplicate_column_names), "Passed": len(duplicate_column_names) == 0},
    {"Check": "Required outcome columns", "Expected": len(TARGETS), "Observed": len(available_targets), "Passed": len(missing_targets) == 0},
    {"Check": "Specified candidate predictors", "Expected": len(CANDIDATE_FEATURES), "Observed": len(available_candidate_features), "Passed": len(missing_candidate_features) == 0}
])

display(DATA_STRUCTURE_CHECK)

if missing_targets:
    raise ValueError(f"Missing outcome columns: {missing_targets}")

if duplicate_column_names:
    raise ValueError(f"Duplicate column names detected: {duplicate_column_names}")

##2.3.Predictor Availability

Compare the predefined candidate predictor list with the variables contained in the dataset and identify missing or duplicated variables.

In [ ]:
# ================================================================
# 2.3 Report Predictor Availability
# ================================================================

FEATURE_AVAILABILITY = pd.DataFrame({
    "Feature": CANDIDATE_FEATURES,
    "Available_in_Dataset": [c in df.columns for c in CANDIDATE_FEATURES]
})

FEATURE_AVAILABILITY.to_csv(
    OUTPUT_FOLDERS["configuration"] / "candidate_feature_availability.csv",
    index=False
)

print(f"Available candidate features: {len(available_candidate_features)}")
print(f"Missing candidate features: {len(missing_candidate_features)}")

if missing_candidate_features:
    print("\nMissing candidate features:")
    print(missing_candidate_features)

if duplicate_column_names:
    print("\nDuplicate column names:")
    print(duplicate_column_names)

##2.4.Outcome Validation

Each medal outcome is verified as a binary variable. The numbers and proportions of athletes achieving gold, silver, bronze, and any medal are summarized for reproducibility.

In [ ]:
# ================================================================
# 2.4 Validate Binary Medal Outcomes
# ================================================================

OUTCOME_SUMMARY = []

for target in TARGETS:
    values = pd.to_numeric(df[target], errors="coerce")
    observed_values = sorted(values.dropna().unique().tolist())
    invalid_values = [value for value in observed_values if value not in [0, 1]]

    OUTCOME_SUMMARY.append({
        "Target": target,
        "Total_N": len(df),
        "Observed_n": int(values.notna().sum()),
        "Missing_n": int(values.isna().sum()),
        "Negative_n": int((values == 0).sum()),
        "Positive_n": int((values == 1).sum()),
        "Positive_Percent": 100 * (values == 1).sum() / values.notna().sum(),
        "Observed_Values": ", ".join(map(str, observed_values)),
        "Valid_Binary_Coding": len(invalid_values) == 0
    })

OUTCOME_SUMMARY_DF = pd.DataFrame(OUTCOME_SUMMARY)
display(OUTCOME_SUMMARY_DF)

if not OUTCOME_SUMMARY_DF["Valid_Binary_Coding"].all():
    invalid_targets = OUTCOME_SUMMARY_DF.loc[
        ~OUTCOME_SUMMARY_DF["Valid_Binary_Coding"], "Target"
    ].tolist()
    raise ValueError(f"Non-binary outcome coding detected: {invalid_targets}")

OUTCOME_SUMMARY_DF.to_csv(
    OUTPUT_FOLDERS["descriptive"] / "medal_outcome_summary.csv",
    index=False
)

##2.5.Record-Level and Missingness Checks

Potential duplicate rows, fully duplicated predictor profiles, variable data types, and overall missingness are documented. No observations are removed automatically at this stage.

In [ ]:
# ================================================================
# 2.5 Record-Level, Data-Type, and Missingness Checks
# ================================================================

analysis_columns = list(dict.fromkeys(available_candidate_features + TARGETS))
exact_duplicate_rows = int(df.duplicated().sum())
duplicate_analysis_profiles = int(df[analysis_columns].duplicated().sum())

DATA_TYPE_SUMMARY = pd.DataFrame({
    "Variable": analysis_columns,
    "Pandas_Dtype": [str(df[c].dtype) for c in analysis_columns],
    "Unique_Observed_Values": [df[c].nunique(dropna=True) for c in analysis_columns]
})

OVERALL_MISSINGNESS = pd.DataFrame({
    "Variable": analysis_columns,
    "Observed_n": [df[c].notna().sum() for c in analysis_columns],
    "Missing_n": [df[c].isna().sum() for c in analysis_columns],
    "Missing_Percent": [100 * df[c].isna().mean() for c in analysis_columns]
}).sort_values(["Missing_Percent", "Variable"], ascending=[False, True])

RECORD_CHECK = pd.DataFrame([
    {"Check": "Exact duplicate rows", "Count": exact_duplicate_rows},
    {"Check": "Duplicate analysis-variable profiles", "Count": duplicate_analysis_profiles},
    {"Check": "Rows with all candidate predictors missing", "Count": int(df[available_candidate_features].isna().all(axis=1).sum())},
    {"Check": "Rows with at least one missing outcome", "Count": int(df[TARGETS].isna().any(axis=1).sum())}
])

display(RECORD_CHECK)
display(OVERALL_MISSINGNESS.head(20))

DATA_TYPE_SUMMARY.to_csv(
    OUTPUT_FOLDERS["configuration"] / "variable_data_types.csv",
    index=False
)
OVERALL_MISSINGNESS.to_csv(
    OUTPUT_FOLDERS["descriptive"] / "overall_variable_missingness.csv",
    index=False
)
RECORD_CHECK.to_csv(
    OUTPUT_FOLDERS["configuration"] / "record_level_checks.csv",
    index=False
)

##2.6.Dataset Metadata Export

Save the validated dataset information and analysis metadata required for reproducibility.

In [ ]:
# ================================================================
# 2.6 Save Validated Dataset Metadata
# ================================================================

VALIDATED_DATA_METADATA = {
    "data_path": str(DATA_PATH),
    "observed_rows": int(df.shape[0]),
    "observed_columns": int(df.shape[1]),
    "expected_rows": EXPECTED_SAMPLE_SIZE,
    "sample_size_matches_manuscript": bool(len(df) == EXPECTED_SAMPLE_SIZE),
    "available_candidate_feature_count": len(available_candidate_features),
    "missing_candidate_features": missing_candidate_features,
    "available_targets": available_targets,
    "duplicate_column_names": duplicate_column_names,
    "exact_duplicate_rows": exact_duplicate_rows,
    "duplicate_analysis_profiles": duplicate_analysis_profiles
}

METADATA_FILE = OUTPUT_FOLDERS["configuration"] / "validated_dataset_metadata.json"

with open(METADATA_FILE, "w", encoding="utf-8") as file:
    json.dump(VALIDATED_DATA_METADATA, file, indent=2, ensure_ascii=False)

print(f"Dataset metadata saved: {METADATA_FILE}")

#3.Descriptive and Missing-Data Analyses

Summarize participant characteristics, medal distributions, movement performance scores, and missing-data patterns prior to machine-learning model development.

##3.1.Participant Characteristics

Summarize the demographic characteristics, anthropometric variables, physiotherapy utilization, and movement performance assessments for the full athlete cohort.

In [ ]:
# ================================================================
# 3.1 Participant Characteristics
# ================================================================

CONTINUOUS_CHARACTERISTICS = [
    "Age", "Height", "Weight", "BMI", "TA", "TC", "TAcq", "TCnt", "TotC", "TMPS"
]

continuous_rows = []
for variable in [c for c in CONTINUOUS_CHARACTERISTICS if c in df.columns]:
    values = pd.to_numeric(df[variable], errors="coerce")
    continuous_rows.append({
        "Variable": variable, "Total_N": len(df), "Observed_n": int(values.notna().sum()),
        "Missing_n": int(values.isna().sum()), "Mean": values.mean(), "SD": values.std(ddof=1),
        "Median": values.median(), "Q1": values.quantile(0.25), "Q3": values.quantile(0.75),
        "Minimum": values.min(), "Maximum": values.max()
    })

CONTINUOUS_CHARACTERISTICS_DF = pd.DataFrame(continuous_rows)

CATEGORICAL_CHARACTERISTICS = [c for c in ["Gender", "SD", "TR"] if c in df.columns]
categorical_tables = []

for variable in CATEGORICAL_CHARACTERISTICS:
    counts = (
        df[variable].fillna("Missing").astype(str).value_counts(dropna=False)
        .rename_axis("Category").reset_index(name="n")
    )
    counts.insert(0, "Variable", variable)
    counts["Percent"] = 100 * counts["n"] / len(df)
    categorical_tables.append(counts)

CATEGORICAL_CHARACTERISTICS_DF = (
    pd.concat(categorical_tables, ignore_index=True)
    if categorical_tables else pd.DataFrame(columns=["Variable", "Category", "n", "Percent"])
)

display(CONTINUOUS_CHARACTERISTICS_DF.round(3))
display(CATEGORICAL_CHARACTERISTICS_DF)

##3.2.Medal Outcome Distributions

Summarize the frequencies and proportions of gold, silver, bronze, and overall medal achievement.

In [ ]:
# ================================================================
# 3.2 Medal Outcome Distributions
# ================================================================

MEDAL_DISTRIBUTION_ROWS = []

for target in TARGETS:
    outcome = pd.to_numeric(df[target], errors="coerce")

    MEDAL_DISTRIBUTION_ROWS.append({
        "Target": target, "Total_N": len(df), "Observed_n": int(outcome.notna().sum()),
        "Missing_n": int(outcome.isna().sum()), "No_Medal_n": int((outcome == 0).sum()),
        "Medal_n": int((outcome == 1).sum()),
        "Medal_Percent": 100 * (outcome == 1).sum() / outcome.notna().sum()
    })

MEDAL_DISTRIBUTION_DF = pd.DataFrame(MEDAL_DISTRIBUTION_ROWS)
display(MEDAL_DISTRIBUTION_DF.round(3))

##3.3.Sex-Specific Medal Comparisons

Compare medal achievement between male and female athletes using chi-square tests and standardized residuals.

In [ ]:
# ================================================================
# 3.3 Medal Outcome Distributions
# ================================================================

MEDAL_DISTRIBUTION_ROWS = []

for target in TARGETS:
    outcome = pd.to_numeric(df[target], errors="coerce")

    MEDAL_DISTRIBUTION_ROWS.append({
        "Target": target, "Total_N": len(df), "Observed_n": int(outcome.notna().sum()),
        "Missing_n": int(outcome.isna().sum()), "No_Medal_n": int((outcome == 0).sum()),
        "Medal_n": int((outcome == 1).sum()),
        "Medal_Percent": 100 * (outcome == 1).sum() / outcome.notna().sum()
    })

MEDAL_DISTRIBUTION_DF = pd.DataFrame(MEDAL_DISTRIBUTION_ROWS)
display(MEDAL_DISTRIBUTION_DF.round(3))

##3.4.Missing-Data Summary

Summarize missing values for all predictor variables before data preprocessing.

In [ ]:
# ================================================================
# 3.4 Missing-Data Summary
# ================================================================

DESCRIPTIVE_FEATURES = [c for c in CANDIDATE_FEATURES if c in df.columns]

MISSING_DATA_SUMMARY_DF = pd.DataFrame({
    "Variable": DESCRIPTIVE_FEATURES,
    "Total_N": len(df),
    "Observed_n": [int(df[c].notna().sum()) for c in DESCRIPTIVE_FEATURES],
    "Missing_n": [int(df[c].isna().sum()) for c in DESCRIPTIVE_FEATURES],
    "Missing_Percent": [100 * df[c].isna().mean() for c in DESCRIPTIVE_FEATURES]
}).sort_values(["Missing_Percent", "Variable"], ascending=[False, True]).reset_index(drop=True)

display(MISSING_DATA_SUMMARY_DF.round(3))

##3.5.Movement Performance Score Missingness

Summarize missingness for each Movement Performance Score (MPS) component and the overall MPS assessment.

In [ ]:
# ================================================================
# 3.5 Movement Performance Score Missingness
# ================================================================

MPS_MISSINGNESS_ROWS = []

for variable in [c for c in MPS_FEATURES if c in df.columns]:
    values = pd.to_numeric(df[variable], errors="coerce")

    MPS_MISSINGNESS_ROWS.append({
        "MPS_Variable": variable, "Total_N": len(df),
        "Observed_n": int(values.notna().sum()), "Observed_Percent": 100 * values.notna().mean(),
        "Missing_n": int(values.isna().sum()), "Missing_Percent": 100 * values.isna().mean(),
        "Observed_Median": values.median(), "Observed_Q1": values.quantile(0.25),
        "Observed_Q3": values.quantile(0.75)
    })

MPS_MISSINGNESS_DF = pd.DataFrame(MPS_MISSINGNESS_ROWS)
display(MPS_MISSINGNESS_DF.round(3))

##3.6.Athlete-Level MPS Completeness

Summarize the number of athletes with complete and incomplete Movement Performance Score assessments.

In [ ]:
# ================================================================
# 3.6 Athlete-Level MPS Completeness
# ================================================================

AVAILABLE_MPS_FEATURES = [c for c in MPS_FEATURES if c in df.columns]
MPS_NUMERIC_DF = df[AVAILABLE_MPS_FEATURES].apply(pd.to_numeric, errors="coerce")
MPS_MISSING_COUNT = MPS_NUMERIC_DF.isna().sum(axis=1)

MPS_COMPLETENESS_DF = pd.DataFrame([{
    "Total_Athletes": len(df),
    "Complete_All_MPS_n": int((MPS_MISSING_COUNT == 0).sum()),
    "Complete_All_MPS_Percent": 100 * (MPS_MISSING_COUNT == 0).mean(),
    "At_Least_One_MPS_Missing_n": int((MPS_MISSING_COUNT > 0).sum()),
    "At_Least_One_MPS_Missing_Percent": 100 * (MPS_MISSING_COUNT > 0).mean(),
    "Median_Number_Missing_MPS": float(MPS_MISSING_COUNT.median()),
    "Maximum_Number_Missing_MPS": int(MPS_MISSING_COUNT.max())
}])

MPS_MISSING_COUNT_DISTRIBUTION_DF = (
    MPS_MISSING_COUNT.value_counts().sort_index()
    .rename_axis("Number_of_Missing_MPS_Variables").reset_index(name="Athletes_n")
)
MPS_MISSING_COUNT_DISTRIBUTION_DF["Percent"] = (
    100 * MPS_MISSING_COUNT_DISTRIBUTION_DF["Athletes_n"] / len(df)
)

display(MPS_COMPLETENESS_DF.round(3))
display(MPS_MISSING_COUNT_DISTRIBUTION_DF.round(3))

##3.7.Total Movement Performance Score Distribution

Summarize the distribution of the Total Movement Performance Score (TMPS), including the median, interquartile range, and maximum observed value.

In [ ]:
# ================================================================
# 3.7 Total Movement Performance Score Distribution
# ================================================================

TMPS_VALUES = pd.to_numeric(df["TMPS"], errors="coerce")
TMPS_OBSERVED = TMPS_VALUES.dropna()

TMPS_SUMMARY_DF = pd.DataFrame([{
    "Total_N": len(df), "Observed_TMPS_n": len(TMPS_OBSERVED),
    "Observed_TMPS_Percent": 100 * TMPS_VALUES.notna().mean(),
    "Missing_TMPS_n": int(TMPS_VALUES.isna().sum()),
    "Missing_TMPS_Percent": 100 * TMPS_VALUES.isna().mean(),
    "Mean": TMPS_OBSERVED.mean(), "SD": TMPS_OBSERVED.std(ddof=1),
    "Median": TMPS_OBSERVED.median(), "Q1": TMPS_OBSERVED.quantile(0.25),
    "Q3": TMPS_OBSERVED.quantile(0.75), "Minimum": TMPS_OBSERVED.min(),
    "Maximum": TMPS_OBSERVED.max(),
    "Maximum_21_n": int((TMPS_OBSERVED == THEORETICAL_TMPS_MAXIMUM).sum()),
    "Maximum_21_Percent": 100 * (TMPS_OBSERVED == THEORETICAL_TMPS_MAXIMUM).mean()
}])

TMPS_FREQUENCY_DF = (
    TMPS_OBSERVED.value_counts().sort_index()
    .rename_axis("TMPS_Score").reset_index(name="Athletes_n")
)
TMPS_FREQUENCY_DF["Percent"] = 100 * TMPS_FREQUENCY_DF["Athletes_n"] / len(TMPS_OBSERVED)
TMPS_FREQUENCY_DF["Cumulative_Percent"] = TMPS_FREQUENCY_DF["Percent"].cumsum()

plt.figure(figsize=(8, 5))
bins = np.arange(np.floor(TMPS_OBSERVED.min()) - 0.5, np.ceil(TMPS_OBSERVED.max()) + 1.5, 1)
plt.hist(TMPS_OBSERVED, bins=bins, edgecolor="black")
plt.axvline(TMPS_CUTOFF, linestyle="--", label=f"TMPS cutoff = {TMPS_CUTOFF}")
plt.axvline(THEORETICAL_TMPS_MAXIMUM, linestyle=":", label=f"Maximum = {THEORETICAL_TMPS_MAXIMUM}")
plt.xlabel("Total Movement Performance Score")
plt.ylabel("Number of athletes")
plt.title("Distribution of Total Movement Performance Scores")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_FOLDERS["figures"] / "TMPS_distribution.png", dpi=300)
plt.close()

display(TMPS_SUMMARY_DF.round(3))
display(TMPS_FREQUENCY_DF.round(3))

##3.8.TMPS Binary Classification Summary

Summarize athletes classified using the prespecified Total Movement Performance Score cutoff of 14.

In [ ]:
# ================================================================
# 3.8 TMPS Binary Classification Summary
# ================================================================

TMPS_BINARY = pd.Series(
    np.where(TMPS_VALUES.notna(), (TMPS_VALUES <= TMPS_CUTOFF).astype(int), np.nan),
    index=df.index,
    name="TMPS_14_or_Less"
)

low_tmps_n = int((TMPS_BINARY == 1).sum())
high_tmps_n = int((TMPS_BINARY == 0).sum())
observed_tmps_n = int(TMPS_BINARY.notna().sum())

TMPS_BINARY_SUMMARY_DF = pd.DataFrame([
    {
        "TMPS_Category": f"≤{TMPS_CUTOFF}", "Athletes_n": low_tmps_n,
        "Percent_of_Observed": 100 * low_tmps_n / observed_tmps_n
    },
    {
        "TMPS_Category": f">{TMPS_CUTOFF}", "Athletes_n": high_tmps_n,
        "Percent_of_Observed": 100 * high_tmps_n / observed_tmps_n
    }
])

TMPS_CEILING_SUMMARY_DF = pd.DataFrame([{
    "Observed_TMPS_n": observed_tmps_n,
    "TMPS_14_or_Less_n": low_tmps_n,
    "TMPS_14_or_Less_Percent": 100 * low_tmps_n / observed_tmps_n,
    "TMPS_Greater_Than_14_n": high_tmps_n,
    "TMPS_Greater_Than_14_Percent": 100 * high_tmps_n / observed_tmps_n,
    "Maximum_21_n": int((TMPS_VALUES == THEORETICAL_TMPS_MAXIMUM).sum()),
    "Maximum_21_Percent": 100 * (TMPS_VALUES == THEORETICAL_TMPS_MAXIMUM).sum() / observed_tmps_n
}])

display(TMPS_BINARY_SUMMARY_DF.round(3))
display(TMPS_CEILING_SUMMARY_DF.round(3))

##3.9.Descriptive Tables Export

Export all descriptive, medal-distribution, sex-comparison, and missing-data results generated in this section.

In [ ]:
# ================================================================
# 3.9 Descriptive Tables Export
# ================================================================

DESCRIPTIVE_OUTPUTS = {
    "Continuous_Characteristics": CONTINUOUS_CHARACTERISTICS_DF,
    "Categorical_Characteristics": CATEGORICAL_CHARACTERISTICS_DF,
    "Medal_Distributions": MEDAL_DISTRIBUTION_DF,
    "Sex_ChiSquare": SEX_CHI_SQUARE_DF,
    "Sex_Residuals": SEX_RESIDUALS_DF,
    "Overall_Missingness": MISSING_DATA_SUMMARY_DF,
    "MPS_Missingness": MPS_MISSINGNESS_DF,
    "MPS_Completeness": MPS_COMPLETENESS_DF,
    "MPS_Missing_Counts": MPS_MISSING_COUNT_DISTRIBUTION_DF,
    "TMPS_Summary": TMPS_SUMMARY_DF,
    "TMPS_Frequency": TMPS_FREQUENCY_DF,
    "TMPS_Binary": TMPS_BINARY_SUMMARY_DF,
    "TMPS_Ceiling": TMPS_CEILING_SUMMARY_DF
}

for name, table in DESCRIPTIVE_OUTPUTS.items():
    table.to_csv(OUTPUT_FOLDERS["descriptive"] / f"{name}.csv", index=False)

DESCRIPTIVE_EXCEL_PATH = OUTPUT_FOLDERS["tables"] / "Descriptive_and_Missing_Data_Analyses.xlsx"

with pd.ExcelWriter(DESCRIPTIVE_EXCEL_PATH, engine="xlsxwriter") as writer:
    for sheet_name, table in DESCRIPTIVE_OUTPUTS.items():
        table.to_excel(writer, sheet_name=sheet_name[:31], index=False)

print(f"Descriptive results exported to: {OUTPUT_FOLDERS['descriptive']}")
print(f"Combined workbook saved to: {DESCRIPTIVE_EXCEL_PATH}")

#4.Feature Screening and Selection

Prepare the predictor matrix used for machine-learning analyses through manuscript-aligned preprocessing, multicollinearity screening, and independent forward feature selection.

##4.1.Candidate Predictor Matrix

Construct the initial predictor matrix using all predefined candidate variables before preprocessing.

In [ ]:
# ================================================================
# 4.1 Candidate Predictor Matrix
# ================================================================

AVAILABLE_CANDIDATE_FEATURES = [c for c in CANDIDATE_FEATURES if c in df.columns]
MISSING_CANDIDATE_FEATURES = [c for c in CANDIDATE_FEATURES if c not in df.columns]

if not AVAILABLE_CANDIDATE_FEATURES:
    raise ValueError("None of the predefined candidate predictors were found in the dataset.")

X_CANDIDATE_RAW = df[AVAILABLE_CANDIDATE_FEATURES].copy()

FEATURE_MATRIX_METADATA = pd.DataFrame({
    "Feature": AVAILABLE_CANDIDATE_FEATURES,
    "Original_Dtype": [str(X_CANDIDATE_RAW[c].dtype) for c in AVAILABLE_CANDIDATE_FEATURES],
    "Observed_n": [int(X_CANDIDATE_RAW[c].notna().sum()) for c in AVAILABLE_CANDIDATE_FEATURES],
    "Missing_n": [int(X_CANDIDATE_RAW[c].isna().sum()) for c in AVAILABLE_CANDIDATE_FEATURES],
    "Unique_Observed_Values": [int(X_CANDIDATE_RAW[c].nunique(dropna=True)) for c in AVAILABLE_CANDIDATE_FEATURES]
})

display(FEATURE_MATRIX_METADATA)
print(f"Available candidate predictors: {len(AVAILABLE_CANDIDATE_FEATURES)}")
print(f"Missing candidate predictors: {len(MISSING_CANDIDATE_FEATURES)}")

if MISSING_CANDIDATE_FEATURES:
    print("Missing predictors:", MISSING_CANDIDATE_FEATURES)

##4.2.Numeric Conversion and Missing-Value Imputation

Convert candidate predictors to numeric form. Impute missing Movement Performance Score values using sport-specific medians followed by overall medians when necessary, and impute the remaining predictors using overall medians.

In [ ]:
# ================================================================
# 4.2 Numeric Conversion and Missing-Value Imputation
# ================================================================

def prepare_numeric_predictors(data: pd.DataFrame, features: list[str]) -> pd.DataFrame:
    """Convert predictors to numeric form without modifying the source dataset."""
    numeric_data = data[features].copy()
    return numeric_data.apply(pd.to_numeric, errors="coerce")


def impute_mps_by_sport(
    predictors: pd.DataFrame,
    sport_series: pd.Series,
    mps_features: list[str]
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Apply sport-specific median imputation followed by overall median imputation."""
    imputed = predictors.copy()
    rows = []

    for feature in [c for c in mps_features if c in imputed.columns]:
        original = pd.to_numeric(imputed[feature], errors="coerce")
        before_missing = int(original.isna().sum())

        sport_medians = original.groupby(sport_series).transform("median")
        after_sport = original.fillna(sport_medians)
        sport_imputed_n = before_missing - int(after_sport.isna().sum())

        overall_median = after_sport.median()
        final = after_sport.fillna(overall_median)
        global_imputed_n = int(after_sport.isna().sum()) - int(final.isna().sum())

        imputed[feature] = final
        rows.append({
            "Feature": feature,
            "Missing_Before_n": before_missing,
            "Sport_Median_Imputed_n": sport_imputed_n,
            "Global_Median_Imputed_n": global_imputed_n,
            "Remaining_Missing_n": int(final.isna().sum()),
            "Global_Median": overall_median
        })

    return imputed, pd.DataFrame(rows)


X_NUMERIC = prepare_numeric_predictors(df, AVAILABLE_CANDIDATE_FEATURES)

if "SD" not in df.columns:
    raise ValueError("The SD variable is required for sport-specific MPS imputation.")

SPORT_GROUPING_SERIES = df["SD"]
X_MPS_IMPUTED, MPS_IMPUTATION_LOG_DF = impute_mps_by_sport(
    predictors=X_NUMERIC,
    sport_series=SPORT_GROUPING_SERIES,
    mps_features=MPS_FEATURES
)

NON_MPS_FEATURES = [c for c in X_MPS_IMPUTED.columns if c not in MPS_FEATURES]
NON_MPS_MISSING_BEFORE = X_MPS_IMPUTED[NON_MPS_FEATURES].isna().sum()

median_imputer = SimpleImputer(strategy="median")
X_IMPUTED = pd.DataFrame(
    median_imputer.fit_transform(X_MPS_IMPUTED),
    columns=X_MPS_IMPUTED.columns,
    index=X_MPS_IMPUTED.index
)

NON_MPS_IMPUTATION_LOG_DF = pd.DataFrame({
    "Feature": NON_MPS_FEATURES,
    "Missing_Before_n": [int(NON_MPS_MISSING_BEFORE[c]) for c in NON_MPS_FEATURES],
    "Global_Median": [
        float(X_MPS_IMPUTED[c].median()) if X_MPS_IMPUTED[c].notna().any() else np.nan
        for c in NON_MPS_FEATURES
    ],
    "Remaining_Missing_n": [int(X_IMPUTED[c].isna().sum()) for c in NON_MPS_FEATURES]
})

if X_IMPUTED.isna().any().any():
    remaining = X_IMPUTED.columns[X_IMPUTED.isna().any()].tolist()
    raise ValueError(f"Missing values remained after imputation: {remaining}")

display(MPS_IMPUTATION_LOG_DF)
display(NON_MPS_IMPUTATION_LOG_DF)
print(f"Imputed predictor matrix: {X_IMPUTED.shape[0]:,} rows × {X_IMPUTED.shape[1]:,} columns")

##4.3.Zero-Variance Feature Removal

Remove predictors with no observed variance before standardization and variance inflation factor calculation. Binary predictors with two observed values are retained.

In [ ]:
# ================================================================
# 4.3 Zero-Variance Feature Removal
# ================================================================

FEATURE_UNIQUE_COUNTS = X_IMPUTED.nunique(dropna=False)
ZERO_VARIANCE_FEATURES = FEATURE_UNIQUE_COUNTS[FEATURE_UNIQUE_COUNTS <= 1].index.tolist()
X_NONCONSTANT = X_IMPUTED.drop(columns=ZERO_VARIANCE_FEATURES).copy()

ZERO_VARIANCE_SUMMARY_DF = pd.DataFrame({
    "Feature": ZERO_VARIANCE_FEATURES,
    "Unique_Value_Count": [int(FEATURE_UNIQUE_COUNTS[c]) for c in ZERO_VARIANCE_FEATURES]
})

print(f"Zero-variance predictors removed: {len(ZERO_VARIANCE_FEATURES)}")
print(f"Predictors retained: {X_NONCONSTANT.shape[1]}")

if ZERO_VARIANCE_FEATURES:
    display(ZERO_VARIANCE_SUMMARY_DF)

##4.4.Feature Standardization

Standardize the retained predictors using z-scores before variance inflation factor screening and forward feature selection.

In [ ]:
# ================================================================
# 4.4 Feature Standardization
# ================================================================

feature_scaler = StandardScaler()

X_STANDARDIZED = pd.DataFrame(
    feature_scaler.fit_transform(X_NONCONSTANT),
    columns=X_NONCONSTANT.columns,
    index=X_NONCONSTANT.index
)

STANDARDIZATION_SUMMARY_DF = pd.DataFrame({
    "Feature": X_NONCONSTANT.columns,
    "Original_Mean": X_NONCONSTANT.mean().values,
    "Original_SD": X_NONCONSTANT.std(ddof=0).values,
    "Standardized_Mean": X_STANDARDIZED.mean().values,
    "Standardized_SD": X_STANDARDIZED.std(ddof=0).values
})

display(STANDARDIZATION_SUMMARY_DF.round(6))

##4.5.Variance Inflation Factor Screening

Iteratively remove the predictor with the highest variance inflation factor when its value exceeds 10. Continue until all retained predictors satisfy the predefined threshold.

In [ ]:
# ================================================================
# 4.5 Variance Inflation Factor Screening
# ================================================================

def calculate_vif(predictor_matrix: pd.DataFrame) -> pd.DataFrame:
    """Calculate variance inflation factors for all columns."""
    values = predictor_matrix.to_numpy(dtype=float)

    return pd.DataFrame({
        "Feature": predictor_matrix.columns,
        "VIF": [variance_inflation_factor(values, i) for i in range(values.shape[1])]
    }).sort_values("VIF", ascending=False).reset_index(drop=True)


def iterative_vif_screening(
    predictor_matrix: pd.DataFrame,
    cutoff: float = 10.0
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Remove the highest-VIF predictor iteratively until all VIF values are acceptable."""
    retained = predictor_matrix.copy()
    removal_rows = []
    iteration = 0

    while retained.shape[1] > 1:
        iteration += 1
        current_vif = calculate_vif(retained)
        highest_feature = str(current_vif.iloc[0]["Feature"])
        highest_vif = float(current_vif.iloc[0]["VIF"])

        if np.isfinite(highest_vif) and highest_vif <= cutoff:
            break

        removal_rows.append({
            "Iteration": iteration,
            "Removed_Feature": highest_feature,
            "VIF_at_Removal": highest_vif,
            "Remaining_Feature_Count_After_Removal": retained.shape[1] - 1
        })

        retained = retained.drop(columns=highest_feature)

    final_vif = calculate_vif(retained)
    return retained, pd.DataFrame(removal_rows), final_vif


X_POST_VIF, VIF_REMOVAL_LOG_DF, FINAL_VIF_DF = iterative_vif_screening(
    predictor_matrix=X_STANDARDIZED,
    cutoff=VIF_CUTOFF
)

POST_VIF_FEATURES = X_POST_VIF.columns.tolist()

print(f"Predictors before VIF screening: {X_STANDARDIZED.shape[1]}")
print(f"Predictors removed by VIF screening: {len(VIF_REMOVAL_LOG_DF)}")
print(f"Predictors retained after VIF screening: {len(POST_VIF_FEATURES)}")
print(f"Maximum final VIF: {FINAL_VIF_DF['VIF'].max():.4f}")

display(VIF_REMOVAL_LOG_DF)
display(FINAL_VIF_DF.round(4))

if not FINAL_VIF_DF.empty and FINAL_VIF_DF["VIF"].max() > VIF_CUTOFF:
    raise RuntimeError("At least one retained predictor exceeded the VIF cutoff.")

##4.6.Forward Feature Selection Model Factory

Define the fixed algorithm configurations used during independent forward feature selection. Hyperparameter optimization is not performed during this stage.

In [ ]:
# ================================================================
# 4.6 Forward Feature Selection Model Factory
# ================================================================

def create_ffs_model(algorithm: str):
    """Return the fixed model configuration used for forward feature selection."""
    if algorithm == "CatBoost":
        return CatBoostClassifier(**FFS_MODEL_PARAMETERS["CatBoost"])

    if algorithm == "XGBoost":
        return XGBClassifier(**FFS_MODEL_PARAMETERS["XGBoost"])

    if algorithm == "LightGBM":
        return LGBMClassifier(**FFS_MODEL_PARAMETERS["LightGBM"])

    raise ValueError(f"Unsupported algorithm: {algorithm}")


FFS_MODEL_CHECK_DF = pd.DataFrame([
    {"Algorithm": algorithm, "Estimator": type(create_ffs_model(algorithm)).__name__}
    for algorithm in ALGORITHMS
])

display(FFS_MODEL_CHECK_DF)

##4.7.Independent Forward Feature Selection

Perform forward feature selection separately for each algorithm and medal outcome using repeated stratified 5-fold cross-validation with 10 repeats. At each step, add the remaining predictor that produces the highest mean ROC–AUC.

In [ ]:
# ================================================================
# 4.7 Independent Forward Feature Selection
# ================================================================

FFS_CV = RepeatedStratifiedKFold(
    n_splits=FFS_N_SPLITS,
    n_repeats=FFS_N_REPEATS,
    random_state=SEED
)


def evaluate_ffs_candidate(
    X: pd.DataFrame,
    y: pd.Series,
    selected_features: list[str],
    candidate_feature: str,
    algorithm: str,
    fixed_splits: list[tuple[np.ndarray, np.ndarray]]
) -> dict:
    """Evaluate one candidate feature added to the current selected subset."""
    candidate_subset = selected_features + [candidate_feature]
    auc_values, accuracy_values = [], []

    for train_index, validation_index in fixed_splits:
        model = clone(create_ffs_model(algorithm))
        X_train = X.iloc[train_index][candidate_subset]
        X_validation = X.iloc[validation_index][candidate_subset]
        y_train, y_validation = y.iloc[train_index], y.iloc[validation_index]

        model.fit(X_train, y_train)
        probability = model.predict_proba(X_validation)[:, 1]
        prediction = (probability >= FIXED_CLASSIFICATION_THRESHOLD).astype(int)

        auc_values.append(roc_auc_score(y_validation, probability))
        accuracy_values.append(accuracy_score(y_validation, prediction))

    return {
        "Candidate_Feature": candidate_feature,
        "ROC_AUC_Mean": float(np.mean(auc_values)),
        "ROC_AUC_SD": float(np.std(auc_values, ddof=0)),
        "Accuracy_Mean": float(np.mean(accuracy_values)),
        "Accuracy_SD": float(np.std(accuracy_values, ddof=0))
    }


def run_forward_feature_selection(
    X: pd.DataFrame,
    y: pd.Series,
    algorithm: str,
    target: str
) -> tuple[pd.DataFrame, list[str]]:
    """Generate the complete forward-selection path and retain the feature order."""
    selected_features, remaining_features, history_rows = [], list(X.columns), []
    fixed_splits = list(FFS_CV.split(X, y))

    for step in range(1, len(X.columns) + 1):
        candidate_results = [
            evaluate_ffs_candidate(
                X=X,
                y=y,
                selected_features=selected_features,
                candidate_feature=candidate,
                algorithm=algorithm,
                fixed_splits=fixed_splits
            )
            for candidate in remaining_features
        ]

        candidate_table = pd.DataFrame(candidate_results).sort_values(
            ["ROC_AUC_Mean", "Candidate_Feature"],
            ascending=[False, True]
        ).reset_index(drop=True)

        best_candidate = candidate_table.iloc[0]
        selected_feature = str(best_candidate["Candidate_Feature"])

        selected_features.append(selected_feature)
        remaining_features.remove(selected_feature)

        history_rows.append({
            "Algorithm": algorithm,
            "Target": target,
            "Num_Features": step,
            "Selected_Feature": selected_feature,
            "Selected_Features_Cumulative": " | ".join(selected_features),
            "ROC_AUC_Mean": float(best_candidate["ROC_AUC_Mean"]),
            "ROC_AUC_SD": float(best_candidate["ROC_AUC_SD"]),
            "Accuracy_Mean": float(best_candidate["Accuracy_Mean"]),
            "Accuracy_SD": float(best_candidate["Accuracy_SD"])
        })

        print(
            f"FFS | {algorithm:8s} | {target} | "
            f"{step:02d}/{len(X.columns):02d} | "
            f"{selected_feature} | "
            f"ROC–AUC={best_candidate['ROC_AUC_Mean']:.4f}"
        )

        if not remaining_features:
            break

    return pd.DataFrame(history_rows), selected_features


FFS_HISTORY = {}
FFS_SELECTION_ORDER = {}

for algorithm in ALGORITHMS:
    for target in TARGETS:
        outcome = pd.to_numeric(df[target], errors="coerce")

        valid_rows = outcome.notna()
        X_ffs = X_POST_VIF.loc[valid_rows].reset_index(drop=True)
        y_ffs = outcome.loc[valid_rows].astype(int).reset_index(drop=True)

        history, selection_order = run_forward_feature_selection(
            X=X_ffs,
            y=y_ffs,
            algorithm=algorithm,
            target=target
        )

        FFS_HISTORY[(algorithm, target)] = history
        FFS_SELECTION_ORDER[(algorithm, target)] = selection_order

##4.8.Peak Feature Subset Identification and Export

Identify the feature subset corresponding to the highest mean ROC–AUC for each algorithm and medal outcome. Export the complete forward-selection trajectories, peak feature subsets, VIF results, preprocessing logs, and summary tables.

In [ ]:
# ================================================================
# 4.8 Peak Feature Subset Identification and Export
# ================================================================

FFS_PEAK_FEATURES = {}
FFS_PEAK_SUMMARY_ROWS = []

for algorithm in ALGORITHMS:
    for target in TARGETS:
        history = FFS_HISTORY[(algorithm, target)]
        selection_order = FFS_SELECTION_ORDER[(algorithm, target)]

        peak_index = history["ROC_AUC_Mean"].idxmax()
        peak_row = history.loc[peak_index]
        peak_feature_count = int(peak_row["Num_Features"])
        peak_features = selection_order[:peak_feature_count]

        FFS_PEAK_FEATURES[(algorithm, target)] = peak_features

        FFS_PEAK_SUMMARY_ROWS.append({
            "Algorithm": algorithm,
            "Target": target,
            "Peak_Feature_Count": peak_feature_count,
            "Peak_ROC_AUC_Mean": float(peak_row["ROC_AUC_Mean"]),
            "Peak_ROC_AUC_SD": float(peak_row["ROC_AUC_SD"]),
            "Peak_Accuracy_Mean": float(peak_row["Accuracy_Mean"]),
            "Peak_Accuracy_SD": float(peak_row["Accuracy_SD"]),
            "Peak_Features": " | ".join(peak_features)
        })

        history.to_csv(
            OUTPUT_FOLDERS["ffs"] / f"{algorithm}_{target}_FFS_History.csv",
            index=False
        )

        pd.DataFrame({
            "Selection_Rank": np.arange(1, len(selection_order) + 1),
            "Feature": selection_order
        }).to_csv(
            OUTPUT_FOLDERS["ffs"] / f"{algorithm}_{target}_Full_Selection_Order.csv",
            index=False
        )

        pd.DataFrame({
            "Selection_Rank": np.arange(1, len(peak_features) + 1),
            "Feature": peak_features
        }).to_csv(
            OUTPUT_FOLDERS["ffs"] / f"{algorithm}_{target}_Peak_Features.csv",
            index=False
        )

FFS_PEAK_SUMMARY_DF = pd.DataFrame(FFS_PEAK_SUMMARY_ROWS)

FEATURE_MATRIX_METADATA.to_csv(
    OUTPUT_FOLDERS["vif"] / "Candidate_Feature_Metadata.csv",
    index=False
)
MPS_IMPUTATION_LOG_DF.to_csv(
    OUTPUT_FOLDERS["vif"] / "MPS_Imputation_Log.csv",
    index=False
)
NON_MPS_IMPUTATION_LOG_DF.to_csv(
    OUTPUT_FOLDERS["vif"] / "Non_MPS_Imputation_Log.csv",
    index=False
)
ZERO_VARIANCE_SUMMARY_DF.to_csv(
    OUTPUT_FOLDERS["vif"] / "Zero_Variance_Features.csv",
    index=False
)
STANDARDIZATION_SUMMARY_DF.to_csv(
    OUTPUT_FOLDERS["vif"] / "Standardization_Summary.csv",
    index=False
)
VIF_REMOVAL_LOG_DF.to_csv(
    OUTPUT_FOLDERS["vif"] / "VIF_Removal_Log.csv",
    index=False
)
FINAL_VIF_DF.to_csv(
    OUTPUT_FOLDERS["vif"] / "Final_VIF_Values.csv",
    index=False
)
pd.DataFrame({"Feature": POST_VIF_FEATURES}).to_csv(
    OUTPUT_FOLDERS["vif"] / "Post_VIF_Features.csv",
    index=False
)
FFS_PEAK_SUMMARY_DF.to_csv(
    OUTPUT_FOLDERS["ffs"] / "FFS_Peak_Summary_All_Models_and_Targets.csv",
    index=False
)

FFS_EXCEL_PATH = OUTPUT_FOLDERS["tables"] / "Feature_Preprocessing_and_Selection.xlsx"

with pd.ExcelWriter(FFS_EXCEL_PATH, engine="xlsxwriter") as writer:
    MPS_IMPUTATION_LOG_DF.to_excel(writer, sheet_name="MPS_Imputation", index=False)
    NON_MPS_IMPUTATION_LOG_DF.to_excel(writer, sheet_name="Other_Imputation", index=False)
    ZERO_VARIANCE_SUMMARY_DF.to_excel(writer, sheet_name="Zero_Variance", index=False)
    VIF_REMOVAL_LOG_DF.to_excel(writer, sheet_name="VIF_Removal", index=False)
    FINAL_VIF_DF.to_excel(writer, sheet_name="Final_VIF", index=False)
    FFS_PEAK_SUMMARY_DF.to_excel(writer, sheet_name="FFS_Peak_Summary", index=False)

    for algorithm in ALGORITHMS:
        for target in TARGETS:
            sheet_name = f"{algorithm[:4]}_{target}_FFS"
            FFS_HISTORY[(algorithm, target)].to_excel(
                writer,
                sheet_name=sheet_name[:31],
                index=False
            )

display(FFS_PEAK_SUMMARY_DF)
print(f"Post-VIF predictor count: {len(POST_VIF_FEATURES)}")
print(f"FFS results saved to: {OUTPUT_FOLDERS['ffs']}")
print(f"Combined workbook saved to: {FFS_EXCEL_PATH}")

#5.FFS-Based SHAP Analyses

Interpret the predictors selected through forward feature selection by refitting each algorithm on its corresponding peak feature subset and calculating global and sport-specific SHAP values.

##5.1.SHAP Analysis Configuration

Define the number of features displayed in SHAP figures and the number of top features retained for sport-specific summaries.

In [ ]:
# ================================================================
# 5.1 SHAP Analysis Configuration
# ================================================================

GLOBAL_SHAP_TOP_N = 20
SPORT_SPECIFIC_SHAP_TOP_N = 10
MINIMUM_SPORT_SAMPLE_SIZE_FOR_SHAP = 1

SHAP_ANALYSIS_SETTINGS = {
    "Global_SHAP_Top_N": GLOBAL_SHAP_TOP_N,
    "Sport_Specific_SHAP_Top_N": SPORT_SPECIFIC_SHAP_TOP_N,
    "Minimum_Sport_Sample_Size": MINIMUM_SPORT_SAMPLE_SIZE_FOR_SHAP,
    "Model_Source": "Fixed FFS model configuration",
    "Feature_Source": "FFS peak feature subset",
    "Refit_Data": "All observations with a non-missing outcome"
}

display(pd.DataFrame(
    SHAP_ANALYSIS_SETTINGS.items(),
    columns=["Setting", "Value"]
))

##5.2.FFS Peak Model Refitting

Refit each machine-learning algorithm on the full available dataset using the peak feature subset identified by independent forward feature selection.

In [ ]:
# ================================================================
# 5.2 FFS Peak Model Refitting
# ================================================================

def fit_ffs_peak_model(
    X: pd.DataFrame,
    y: pd.Series,
    algorithm: str,
    peak_features: list[str]
):
    """Refit the fixed FFS model on the full available dataset."""
    if not peak_features:
        raise ValueError(f"No FFS peak features were provided for {algorithm}.")

    missing_features = [feature for feature in peak_features if feature not in X.columns]
    if missing_features:
        raise ValueError(f"Peak features absent from the predictor matrix: {missing_features}")

    fitted_model = create_ffs_model(algorithm)
    fitted_model.fit(X[peak_features], y)
    return fitted_model


FFS_PEAK_MODELS = {}
FFS_PEAK_INPUTS = {}
FFS_PEAK_OUTCOMES = {}

for algorithm in ALGORITHMS:
    for target in TARGETS:
        outcome = pd.to_numeric(df[target], errors="coerce")
        valid_rows = outcome.notna()

        peak_features = FFS_PEAK_FEATURES[(algorithm, target)]
        X_peak = X_POST_VIF.loc[valid_rows, peak_features].copy()
        y_peak = outcome.loc[valid_rows].astype(int).copy()

        fitted_model = fit_ffs_peak_model(
            X=X_peak,
            y=y_peak,
            algorithm=algorithm,
            peak_features=peak_features
        )

        FFS_PEAK_MODELS[(algorithm, target)] = fitted_model
        FFS_PEAK_INPUTS[(algorithm, target)] = X_peak
        FFS_PEAK_OUTCOMES[(algorithm, target)] = y_peak

        print(
            f"Refitted | {algorithm:8s} | {target} | "
            f"N={len(y_peak):,} | Features={len(peak_features)}"
        )

##5.3.SHAP Value Calculation

Calculate athlete-level SHAP values for each refitted model. CatBoost native SHAP values are used for CatBoost, whereas TreeExplainer is used for XGBoost and LightGBM.

In [ ]:
# ================================================================
# 5.3 SHAP Value Calculation
# ================================================================

def calculate_shap_values(
    model,
    X: pd.DataFrame,
    algorithm: str
) -> tuple[np.ndarray, float | np.ndarray]:
    """Return feature-level SHAP values and the expected model value."""
    if algorithm == "CatBoost":
        shap_raw = np.asarray(model.get_feature_importance(X, type="ShapValues"))

        if shap_raw.shape[1] != X.shape[1] + 1:
            raise RuntimeError(
                f"Unexpected CatBoost SHAP shape: {shap_raw.shape}; "
                f"expected {X.shape[1] + 1} columns."
            )

        shap_values = shap_raw[:, :-1]
        expected_value = shap_raw[:, -1].mean()
        return shap_values, expected_value

    explainer = shap.TreeExplainer(model)
    shap_output = explainer.shap_values(X)

    if isinstance(shap_output, list):
        shap_output = shap_output[-1]

    shap_values = np.asarray(shap_output)

    if shap_values.ndim == 3:
        shap_values = shap_values[:, :, -1]

    expected_value = explainer.expected_value
    if isinstance(expected_value, (list, np.ndarray)):
        expected_value = np.asarray(expected_value).reshape(-1)[-1]

    return shap_values, expected_value


SHAP_VALUES = {}
SHAP_EXPECTED_VALUES = {}

for algorithm in ALGORITHMS:
    for target in TARGETS:
        model = FFS_PEAK_MODELS[(algorithm, target)]
        X_peak = FFS_PEAK_INPUTS[(algorithm, target)]

        shap_values, expected_value = calculate_shap_values(
            model=model,
            X=X_peak,
            algorithm=algorithm
        )

        if shap_values.shape != X_peak.shape:
            raise RuntimeError(
                f"SHAP/input mismatch for {algorithm}-{target}: "
                f"{shap_values.shape} versus {X_peak.shape}"
            )

        SHAP_VALUES[(algorithm, target)] = shap_values
        SHAP_EXPECTED_VALUES[(algorithm, target)] = expected_value

        print(
            f"SHAP calculated | {algorithm:8s} | {target} | "
            f"Shape={shap_values.shape}"
        )

##5.4.Global SHAP Summary

Rank the FFS-selected predictors using their mean absolute SHAP values across all athletes and report the corresponding variability.

In [ ]:
# ================================================================
# 5.4 Global SHAP Summary
# ================================================================

GLOBAL_SHAP_TABLES = {}
GLOBAL_SHAP_SUMMARY_ROWS = []

for algorithm in ALGORITHMS:
    for target in TARGETS:
        X_peak = FFS_PEAK_INPUTS[(algorithm, target)]
        shap_values = SHAP_VALUES[(algorithm, target)]
        absolute_shap = np.abs(shap_values)

        global_table = pd.DataFrame({
            "Algorithm": algorithm,
            "Target": target,
            "Feature": X_peak.columns,
            "Mean_Absolute_SHAP": absolute_shap.mean(axis=0),
            "SD_Absolute_SHAP": absolute_shap.std(axis=0, ddof=1),
            "Median_Absolute_SHAP": np.median(absolute_shap, axis=0),
            "Mean_Signed_SHAP": shap_values.mean(axis=0),
            "Minimum_SHAP": shap_values.min(axis=0),
            "Maximum_SHAP": shap_values.max(axis=0)
        }).sort_values(
            ["Mean_Absolute_SHAP", "Feature"],
            ascending=[False, True]
        ).reset_index(drop=True)

        global_table.insert(2, "Rank", np.arange(1, len(global_table) + 1))
        GLOBAL_SHAP_TABLES[(algorithm, target)] = global_table

        top_feature = global_table.iloc[0]
        GLOBAL_SHAP_SUMMARY_ROWS.append({
            "Algorithm": algorithm,
            "Target": target,
            "Athletes_n": len(X_peak),
            "FFS_Peak_Feature_Count": X_peak.shape[1],
            "Top_Feature": top_feature["Feature"],
            "Top_Mean_Absolute_SHAP": top_feature["Mean_Absolute_SHAP"],
            "Expected_Value": SHAP_EXPECTED_VALUES[(algorithm, target)]
        })

GLOBAL_SHAP_SUMMARY_DF = pd.DataFrame(GLOBAL_SHAP_SUMMARY_ROWS)

display(GLOBAL_SHAP_SUMMARY_DF.round(6))

##5.5.Global SHAP Figures

Generate global mean absolute SHAP bar plots and SHAP beeswarm plots for each algorithm and medal outcome using the FFS peak feature subsets.

In [ ]:
# ================================================================
# 5.5 Global SHAP Figures
# ================================================================

def save_global_shap_figures(
    X: pd.DataFrame,
    shap_values: np.ndarray,
    algorithm: str,
    target: str,
    output_folder: Path,
    top_n: int = 20
) -> None:
    """Save global SHAP bar and beeswarm plots."""
    plot_top_n = min(top_n, X.shape[1])

    shap.summary_plot(
        shap_values,
        X,
        plot_type="bar",
        max_display=plot_top_n,
        show=False
    )
    plt.title(f"{algorithm} – {target}: Mean Absolute SHAP")
    plt.tight_layout()
    plt.savefig(
        output_folder / f"{algorithm}_{target}_SHAP_Bar.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.close()

    shap.summary_plot(
        shap_values,
        X,
        max_display=plot_top_n,
        show=False
    )
    plt.title(f"{algorithm} – {target}: SHAP Distribution")
    plt.tight_layout()
    plt.savefig(
        output_folder / f"{algorithm}_{target}_SHAP_Beeswarm.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.close()


for algorithm in ALGORITHMS:
    for target in TARGETS:
        save_global_shap_figures(
            X=FFS_PEAK_INPUTS[(algorithm, target)],
            shap_values=SHAP_VALUES[(algorithm, target)],
            algorithm=algorithm,
            target=target,
            output_folder=OUTPUT_FOLDERS["figures"],
            top_n=GLOBAL_SHAP_TOP_N
        )

print(f"Global SHAP figures saved to: {OUTPUT_FOLDERS['figures']}")

##5.6.Sport-Specific SHAP Summary

Summarize mean absolute SHAP values within each sports discipline using the athlete-level SHAP values from the corresponding full-data FFS peak model.

In [ ]:
# ================================================================
# 5.6 Sport-Specific SHAP Summary
# ================================================================

def summarize_sport_specific_shap(
    X: pd.DataFrame,
    shap_values: np.ndarray,
    sport_series: pd.Series,
    algorithm: str,
    target: str,
    top_n: int = 10,
    minimum_sample_size: int = 1
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Summarize feature importance separately within each sport discipline."""
    aligned_sports = sport_series.reindex(X.index)
    all_rows, top_rows = [], []

    for sport in aligned_sports.dropna().unique():
        sport_mask = aligned_sports.eq(sport).to_numpy()
        sport_n = int(sport_mask.sum())

        if sport_n < minimum_sample_size:
            continue

        sport_shap = shap_values[sport_mask]
        absolute_sport_shap = np.abs(sport_shap)

        sport_table = pd.DataFrame({
            "Algorithm": algorithm,
            "Target": target,
            "Sport_Discipline": sport,
            "Sport_n": sport_n,
            "Feature": X.columns,
            "Mean_Absolute_SHAP": absolute_sport_shap.mean(axis=0),
            "SD_Absolute_SHAP": (
                absolute_sport_shap.std(axis=0, ddof=1)
                if sport_n > 1 else np.zeros(X.shape[1])
            ),
            "Median_Absolute_SHAP": np.median(absolute_sport_shap, axis=0),
            "Mean_Signed_SHAP": sport_shap.mean(axis=0)
        }).sort_values(
            ["Mean_Absolute_SHAP", "Feature"],
            ascending=[False, True]
        ).reset_index(drop=True)

        sport_table.insert(4, "Rank", np.arange(1, len(sport_table) + 1))
        all_rows.append(sport_table)
        top_rows.append(sport_table.head(min(top_n, len(sport_table))))

    all_sports = (
        pd.concat(all_rows, ignore_index=True)
        if all_rows else pd.DataFrame()
    )
    top_sports = (
        pd.concat(top_rows, ignore_index=True)
        if top_rows else pd.DataFrame()
    )

    return all_sports, top_sports


SPORT_SPECIFIC_SHAP_ALL = {}
SPORT_SPECIFIC_SHAP_TOP = {}

for algorithm in ALGORITHMS:
    for target in TARGETS:
        X_peak = FFS_PEAK_INPUTS[(algorithm, target)]
        sport_series = df.loc[X_peak.index, "SD"]

        all_table, top_table = summarize_sport_specific_shap(
            X=X_peak,
            shap_values=SHAP_VALUES[(algorithm, target)],
            sport_series=sport_series,
            algorithm=algorithm,
            target=target,
            top_n=SPORT_SPECIFIC_SHAP_TOP_N,
            minimum_sample_size=MINIMUM_SPORT_SAMPLE_SIZE_FOR_SHAP
        )

        SPORT_SPECIFIC_SHAP_ALL[(algorithm, target)] = all_table
        SPORT_SPECIFIC_SHAP_TOP[(algorithm, target)] = top_table

print("Sport-specific SHAP summaries were generated.")

##5.7.SHAP Results Export

Export global and sport-specific SHAP summaries, athlete-level SHAP values, refitted FFS peak models, and consolidated workbooks for reproducibility.

In [ ]:
# ================================================================
# 5.7 SHAP Results Export
# ================================================================

for algorithm in ALGORITHMS:
    for target in TARGETS:
        X_peak = FFS_PEAK_INPUTS[(algorithm, target)]
        shap_values = SHAP_VALUES[(algorithm, target)]

        GLOBAL_SHAP_TABLES[(algorithm, target)].to_csv(
            OUTPUT_FOLDERS["shap"] / f"{algorithm}_{target}_Global_SHAP.csv",
            index=False
        )

        SPORT_SPECIFIC_SHAP_ALL[(algorithm, target)].to_csv(
            OUTPUT_FOLDERS["shap"] / f"{algorithm}_{target}_Sport_Specific_SHAP_All.csv",
            index=False
        )

        SPORT_SPECIFIC_SHAP_TOP[(algorithm, target)].to_csv(
            OUTPUT_FOLDERS["shap"] / f"{algorithm}_{target}_Sport_Specific_SHAP_Top10.csv",
            index=False
        )

        athlete_level_shap = pd.DataFrame(
            shap_values,
            columns=[f"SHAP_{feature}" for feature in X_peak.columns],
            index=X_peak.index
        )
        athlete_level_shap.insert(0, "Athlete_Index", X_peak.index)
        athlete_level_shap.insert(1, "Sport_Discipline", df.loc[X_peak.index, "SD"].values)
        athlete_level_shap.insert(2, "Observed_Outcome", FFS_PEAK_OUTCOMES[(algorithm, target)].values)

        athlete_level_shap.to_csv(
            OUTPUT_FOLDERS["shap"] / f"{algorithm}_{target}_Athlete_Level_SHAP.csv",
            index=False
        )

        X_peak.to_csv(
            OUTPUT_FOLDERS["shap"] / f"{algorithm}_{target}_FFS_Peak_Input.csv",
            index=False
        )

        with open(
            OUTPUT_FOLDERS["models"] / f"{algorithm}_{target}_FFS_Peak_Model.pkl",
            "wb"
        ) as model_file:
            pickle.dump(FFS_PEAK_MODELS[(algorithm, target)], model_file)

GLOBAL_SHAP_SUMMARY_DF.to_csv(
    OUTPUT_FOLDERS["shap"] / "Global_SHAP_Summary_All_Models_and_Targets.csv",
    index=False
)

SHAP_EXCEL_PATH = OUTPUT_FOLDERS["tables"] / "FFS_Based_SHAP_Analyses.xlsx"

with pd.ExcelWriter(SHAP_EXCEL_PATH, engine="xlsxwriter") as writer:
    GLOBAL_SHAP_SUMMARY_DF.to_excel(writer, sheet_name="Global_Summary", index=False)

    for algorithm in ALGORITHMS:
        for target in TARGETS:
            prefix = f"{algorithm[:4]}_{target}"

            GLOBAL_SHAP_TABLES[(algorithm, target)].to_excel(
                writer,
                sheet_name=f"{prefix}_Global"[:31],
                index=False
            )

            SPORT_SPECIFIC_SHAP_TOP[(algorithm, target)].to_excel(
                writer,
                sheet_name=f"{prefix}_SportTop"[:31],
                index=False
            )

SHAP_CONFIGURATION_FILE = OUTPUT_FOLDERS["configuration"] / "shap_analysis_configuration.json"

with open(SHAP_CONFIGURATION_FILE, "w", encoding="utf-8") as file:
    json.dump(SHAP_ANALYSIS_SETTINGS, file, indent=2, ensure_ascii=False)

display(GLOBAL_SHAP_SUMMARY_DF.round(6))
print(f"SHAP tables saved to: {OUTPUT_FOLDERS['shap']}")
print(f"SHAP models saved to: {OUTPUT_FOLDERS['models']}")
print(f"Combined workbook saved to: {SHAP_EXCEL_PATH}")

#6.Independent Nested Cross-Validation

Evaluate CatBoost, XGBoost, and LightGBM independently from forward feature selection using five outer stratified folds and three inner stratified folds. Hyperparameter optimization, preprocessing, and SMOTE are restricted to the training data within each outer fold.

##6.1.Nested Cross-Validation Configuration

Define categorical and numeric predictors, cross-validation splitters, and the number of hyperparameter configurations evaluated for each algorithm.

In [ ]:
# ================================================================
# 6.1 Nested Cross-Validation Configuration
# ================================================================

NESTED_CV_FEATURES = [feature for feature in POST_VIF_FEATURES if feature in df.columns]

CATEGORICAL_MODEL_FEATURES = [
    feature for feature in ["Gender", "SD"]
    if feature in NESTED_CV_FEATURES
]

NUMERIC_MODEL_FEATURES = [
    feature for feature in NESTED_CV_FEATURES
    if feature not in CATEGORICAL_MODEL_FEATURES
]

OUTER_CV = StratifiedKFold(
    n_splits=OUTER_CV_SPLITS,
    shuffle=True,
    random_state=SEED
)

INNER_CV = StratifiedKFold(
    n_splits=INNER_CV_SPLITS,
    shuffle=True,
    random_state=SEED
)


def count_search_combinations(search_space: dict) -> int:
    """Return the total number of unique hyperparameter combinations."""
    return int(np.prod([len(values) for values in search_space.values()]))


def get_search_iterations(algorithm: str) -> int:
    """Evaluate up to 16 configurations, or all combinations when fewer exist."""
    return min(16, count_search_combinations(SEARCH_SPACES[algorithm]))


NESTED_CV_CONFIGURATION_DF = pd.DataFrame([
    {
        "Algorithm": algorithm,
        "Total_Search_Combinations": count_search_combinations(SEARCH_SPACES[algorithm]),
        "Configurations_Evaluated": get_search_iterations(algorithm),
        "Outer_Folds": OUTER_CV_SPLITS,
        "Inner_Folds": INNER_CV_SPLITS,
        "Post_VIF_Feature_Count": len(NESTED_CV_FEATURES)
    }
    for algorithm in ALGORITHMS
])

display(NESTED_CV_CONFIGURATION_DF)
print("Categorical predictors:", CATEGORICAL_MODEL_FEATURES)
print("Numeric predictors:", len(NUMERIC_MODEL_FEATURES))

##6.2.Preprocessing and Model Pipelines

Construct model-specific pipelines in which missing-value imputation, one-hot encoding, standardization, and SMOTE are fitted exclusively within the training data.

In [ ]:
# ================================================================
# 6.2 Preprocessing and Model Pipelines
# ================================================================

def create_one_hot_encoder() -> OneHotEncoder:
    """Create a dense one-hot encoder compatible with different sklearn versions."""
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            drop=None,
            sparse_output=False
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            drop=None,
            sparse=False
        )


def create_model_estimator(algorithm: str):
    """Create the untuned estimator used in nested cross-validation."""
    if algorithm == "CatBoost":
        return CatBoostClassifier(
            loss_function="Logloss",
            eval_metric="AUC",
            random_seed=SEED,
            verbose=0,
            allow_writing_files=False,
            thread_count=-1
        )

    if algorithm == "XGBoost":
        return XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=SEED,
            n_jobs=-1,
            tree_method="hist"
        )

    if algorithm == "LightGBM":
        return LGBMClassifier(
            objective="binary",
            random_state=SEED,
            n_jobs=-1,
            verbosity=-1
        )

    raise ValueError(f"Unsupported algorithm: {algorithm}")


def create_nested_preprocessor(
    numeric_features: list[str],
    categorical_features: list[str]
) -> ColumnTransformer:
    """Create fold-specific preprocessing for numeric and categorical variables."""
    transformers = []

    if numeric_features:
        numeric_pipeline = SklearnPipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ])
        transformers.append(("numeric", numeric_pipeline, numeric_features))

    if categorical_features:
        categorical_pipeline = SklearnPipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", create_one_hot_encoder())
        ])
        transformers.append(("categorical", categorical_pipeline, categorical_features))

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        verbose_feature_names_out=False
    )


def create_nested_pipeline(
    algorithm: str,
    numeric_features: list[str],
    categorical_features: list[str],
    use_smote: bool = True
) -> ImbalancedPipeline:
    """Create the complete training-fold pipeline."""
    steps = [
        (
            "preprocessor",
            create_nested_preprocessor(
                numeric_features=numeric_features,
                categorical_features=categorical_features
            )
        )
    ]

    if use_smote:
        steps.append((
            "smote",
            SMOTE(
                k_neighbors=SMOTE_K_NEIGHBORS,
                random_state=SEED
            )
        ))

    steps.append(("model", create_model_estimator(algorithm)))
    return ImbalancedPipeline(steps)


PIPELINE_CHECK_DF = pd.DataFrame([
    {
        "Algorithm": algorithm,
        "Pipeline": str(
            create_nested_pipeline(
                algorithm,
                NUMERIC_MODEL_FEATURES,
                CATEGORICAL_MODEL_FEATURES
            )
        )
    }
    for algorithm in ALGORITHMS
])

display(PIPELINE_CHECK_DF[["Algorithm"]])

##6.3.Classification Threshold Utilities

Define functions for the fixed 0.50 threshold and thresholds selected using the Youden index and maximum F1 score within the outer-training data.

In [ ]:
# ================================================================
# 6.3 Classification Threshold Utilities
# ================================================================

def calculate_youden_threshold(
    y_true: pd.Series,
    probabilities: np.ndarray
) -> float:
    """Select the threshold that maximizes sensitivity plus specificity minus one."""
    fpr, tpr, thresholds = roc_curve(y_true, probabilities)
    valid = np.isfinite(thresholds)

    if not valid.any():
        return FIXED_CLASSIFICATION_THRESHOLD

    youden_index = tpr[valid] - fpr[valid]
    return float(thresholds[valid][np.argmax(youden_index)])


def calculate_f1_threshold(
    y_true: pd.Series,
    probabilities: np.ndarray
) -> float:
    """Select the probability threshold that maximizes the F1 score."""
    precision, recall, thresholds = precision_recall_curve(y_true, probabilities)

    if len(thresholds) == 0:
        return FIXED_CLASSIFICATION_THRESHOLD

    f1_values = (
        2 * precision[:-1] * recall[:-1]
        / np.maximum(precision[:-1] + recall[:-1], 1e-12)
    )

    return float(thresholds[np.nanargmax(f1_values)])


def calculate_threshold_metrics(
    y_true: pd.Series,
    probabilities: np.ndarray,
    threshold: float
) -> dict:
    """Calculate threshold-dependent classification metrics."""
    predictions = (np.asarray(probabilities) >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1]
    ).ravel()

    return {
        "Threshold": float(threshold),
        "Accuracy": accuracy_score(y_true, predictions),
        "Precision": precision_score(y_true, predictions, zero_division=0),
        "Recall_Sensitivity": recall_score(y_true, predictions, zero_division=0),
        "Specificity": tn / (tn + fp) if (tn + fp) > 0 else np.nan,
        "F1": f1_score(y_true, predictions, zero_division=0),
        "Negative_Predictive_Value": tn / (tn + fn) if (tn + fn) > 0 else np.nan,
        "TP": int(tp),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn)
    }

##6.4.Calibration Utilities

Calculate calibration intercepts and slopes from pooled out-of-fold probabilities and generate calibration-curve data.

In [ ]:
# ================================================================
# 6.4 Calibration Utilities
# ================================================================

def calculate_calibration_intercept_slope(
    y_true: pd.Series,
    probabilities: pd.Series
) -> dict:
    """Estimate calibration intercept and slope using logistic recalibration."""
    y_array = np.asarray(y_true, dtype=int)
    probability_array = np.clip(
        np.asarray(probabilities, dtype=float),
        1e-6,
        1 - 1e-6
    )

    logit_probability = np.log(
        probability_array / (1 - probability_array)
    )

    design_matrix = sm.add_constant(logit_probability)

    try:
        calibration_model = sm.Logit(
            y_array,
            design_matrix
        ).fit(disp=False)

        return {
            "Calibration_Intercept": float(calibration_model.params[0]),
            "Calibration_Slope": float(calibration_model.params[1])
        }

    except Exception:
        return {
            "Calibration_Intercept": np.nan,
            "Calibration_Slope": np.nan
        }


def create_calibration_table(
    y_true: pd.Series,
    probabilities: pd.Series,
    algorithm: str,
    target: str
) -> pd.DataFrame:
    """Create quantile-based calibration points."""
    observed_fraction, mean_probability = calibration_curve(
        y_true,
        probabilities,
        n_bins=CALIBRATION_BINS,
        strategy="quantile"
    )

    return pd.DataFrame({
        "Algorithm": algorithm,
        "Target": target,
        "Calibration_Bin": np.arange(1, len(observed_fraction) + 1),
        "Mean_Predicted_Probability": mean_probability,
        "Observed_Event_Proportion": observed_fraction
    })

##6.5.Nested Cross-Validation Function

For each outer fold, tune hyperparameters using the inner folds, refit the best pipeline on the outer-training data, and evaluate predictions on the untouched outer-test fold.

In [ ]:
# ================================================================
# 6.5 Nested Cross-Validation Function
# ================================================================

def run_nested_cross_validation(
    data: pd.DataFrame,
    features: list[str],
    target: str,
    algorithm: str,
    analysis_label: str = "Main_Independent_PostVIF_NestedCV",
    outer_splits: list[tuple[np.ndarray, np.ndarray]] | None = None,
    use_smote: bool = True
) -> dict[str, pd.DataFrame]:
    """Run nested CV and return fold metrics, OOF predictions, parameters, and thresholds."""
    analysis_columns = list(dict.fromkeys(features + [target]))
    analysis_data = data[analysis_columns].copy()

    for feature in features:
        if feature in CATEGORICAL_MODEL_FEATURES:
            analysis_data[feature] = analysis_data[feature].astype("object")
        else:
            analysis_data[feature] = pd.to_numeric(
                analysis_data[feature],
                errors="coerce"
            )

    analysis_data[target] = pd.to_numeric(
        analysis_data[target],
        errors="coerce"
    )

    analysis_data = (
        analysis_data
        .dropna(subset=[target])
        .reset_index()
        .rename(columns={"index": "Original_Index"})
    )

    X = analysis_data[features].copy()
    y = analysis_data[target].astype(int).reset_index(drop=True)

    numeric_features = [
        feature for feature in features
        if feature not in CATEGORICAL_MODEL_FEATURES
    ]
    categorical_features = [
        feature for feature in features
        if feature in CATEGORICAL_MODEL_FEATURES
    ]

    if outer_splits is None:
        outer_splits = list(OUTER_CV.split(X, y))

    fold_metric_rows = []
    prediction_rows = []
    parameter_rows = []
    threshold_rows = []

    for outer_fold, (train_index, test_index) in enumerate(
        outer_splits,
        start=1
    ):
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]

        base_pipeline = create_nested_pipeline(
            algorithm=algorithm,
            numeric_features=numeric_features,
            categorical_features=categorical_features,
            use_smote=use_smote
        )

        search = RandomizedSearchCV(
            estimator=base_pipeline,
            param_distributions=SEARCH_SPACES[algorithm],
            n_iter=get_search_iterations(algorithm),
            scoring="roc_auc",
            cv=INNER_CV,
            random_state=SEED,
            n_jobs=-1,
            refit=True,
            return_train_score=False,
            error_score=np.nan
        )

        search.fit(X_train, y_train)
        best_pipeline = search.best_estimator_

        outer_test_probability = best_pipeline.predict_proba(X_test)[:, 1]

        # Generate training-only out-of-fold probabilities for threshold selection.
        inner_training_probability = cross_val_predict(
            clone(best_pipeline),
            X_train,
            y_train,
            cv=INNER_CV,
            method="predict_proba",
            n_jobs=-1
        )[:, 1]

        thresholds = {
            "Fixed_0.50": FIXED_CLASSIFICATION_THRESHOLD,
            "Inner_Youden": calculate_youden_threshold(
                y_train,
                inner_training_probability
            ),
            "Inner_F1": calculate_f1_threshold(
                y_train,
                inner_training_probability
            )
        }

        probability_metrics = {
            "ROC_AUC": roc_auc_score(y_test, outer_test_probability),
            "PR_AUC_Average_Precision": average_precision_score(
                y_test,
                outer_test_probability
            ),
            "Brier_Score": brier_score_loss(
                y_test,
                outer_test_probability
            )
        }

        for threshold_type, threshold in thresholds.items():
            fold_metric_rows.append({
                "Algorithm": algorithm,
                "Target": target,
                "Analysis": analysis_label,
                "Outer_Fold": outer_fold,
                "Threshold_Type": threshold_type,
                "Training_n": len(train_index),
                "Test_n": len(test_index),
                "Training_Positive_Prevalence": y_train.mean(),
                "Test_Positive_Prevalence": y_test.mean(),
                **probability_metrics,
                **calculate_threshold_metrics(
                    y_test,
                    outer_test_probability,
                    threshold
                )
            })

            threshold_rows.append({
                "Algorithm": algorithm,
                "Target": target,
                "Analysis": analysis_label,
                "Outer_Fold": outer_fold,
                "Threshold_Type": threshold_type,
                "Threshold": threshold
            })

        for position, row_index in enumerate(test_index):
            prediction_rows.append({
                "Algorithm": algorithm,
                "Target": target,
                "Analysis": analysis_label,
                "Outer_Fold": outer_fold,
                "Athlete_Index": int(
                    analysis_data.iloc[row_index]["Original_Index"]
                ),
                "Observed_Outcome": int(y_test.iloc[position]),
                "Predicted_Probability": float(
                    outer_test_probability[position]
                )
            })

        parameter_rows.append({
            "Algorithm": algorithm,
            "Target": target,
            "Analysis": analysis_label,
            "Outer_Fold": outer_fold,
            "Best_Inner_ROC_AUC": float(search.best_score_),
            "Best_Parameters": json.dumps(
                search.best_params_,
                sort_keys=True
            )
        })

        print(
            f"Nested CV | {algorithm:8s} | {target} | "
            f"Fold {outer_fold}/{len(outer_splits)} | "
            f"Outer ROC–AUC={probability_metrics['ROC_AUC']:.4f}"
        )

    return {
        "Fold_Metrics": pd.DataFrame(fold_metric_rows),
        "OOF_Predictions": pd.DataFrame(prediction_rows),
        "Best_Parameters": pd.DataFrame(parameter_rows),
        "Thresholds": pd.DataFrame(threshold_rows)
    }

##6.6.Main Nested Cross-Validation Execution

Run the independent nested cross-validation procedure for all three algorithms and all four medal outcomes using the complete post-VIF candidate feature set.

In [ ]:
# ================================================================
# 6.6 Main Nested Cross-Validation Execution
# ================================================================

MAIN_NESTED_CV_RESULTS = {}

for algorithm in ALGORITHMS:
    for target in TARGETS:
        MAIN_NESTED_CV_RESULTS[(algorithm, target)] = (
            run_nested_cross_validation(
                data=df,
                features=NESTED_CV_FEATURES,
                target=target,
                algorithm=algorithm,
                analysis_label="Main_Independent_PostVIF_NestedCV",
                use_smote=True
            )
        )

##6.7.Nested Cross-Validation Performance Summary

Summarize fold-level discrimination, calibration, and threshold-dependent classification performance using the fixed 0.50 decision threshold.

In [ ]:
# ================================================================
# 6.7 Nested Cross-Validation Performance Summary
# ================================================================

def summarize_nested_cv_result(
    result: dict[str, pd.DataFrame],
    feature_count: int,
    ffs_peak_count: int
) -> dict:
    """Summarize fixed-threshold outer-fold performance and pooled calibration."""
    fold_metrics = result["Fold_Metrics"]
    fixed_metrics = fold_metrics[
        fold_metrics["Threshold_Type"] == "Fixed_0.50"
    ].copy()

    predictions = (
        result["OOF_Predictions"]
        .sort_values("Athlete_Index")
        .reset_index(drop=True)
    )

    first_row = fixed_metrics.iloc[0]

    summary = {
        "Algorithm": first_row["Algorithm"],
        "Target": first_row["Target"],
        "Analysis": first_row["Analysis"],
        "Athletes_n": len(predictions),
        "Nested_CV_Feature_Count": feature_count,
        "FFS_Peak_Feature_Count_Reported_Separately": ffs_peak_count,
        "Positive_Prevalence": predictions["Observed_Outcome"].mean()
    }

    metrics = [
        "ROC_AUC",
        "PR_AUC_Average_Precision",
        "Brier_Score",
        "Accuracy",
        "Precision",
        "Recall_Sensitivity",
        "Specificity",
        "F1",
        "Negative_Predictive_Value"
    ]

    for metric in metrics:
        summary[f"{metric}_Mean"] = fixed_metrics[metric].mean()
        summary[f"{metric}_SD"] = fixed_metrics[metric].std(ddof=1)

    summary.update(
        calculate_calibration_intercept_slope(
            predictions["Observed_Outcome"],
            predictions["Predicted_Probability"]
        )
    )

    return summary


MAIN_NESTED_CV_SUMMARY_ROWS = []

for algorithm in ALGORITHMS:
    for target in TARGETS:
        result = MAIN_NESTED_CV_RESULTS[(algorithm, target)]

        MAIN_NESTED_CV_SUMMARY_ROWS.append(
            summarize_nested_cv_result(
                result=result,
                feature_count=len(NESTED_CV_FEATURES),
                ffs_peak_count=len(
                    FFS_PEAK_FEATURES[(algorithm, target)]
                )
            )
        )

MAIN_NESTED_CV_SUMMARY_DF = pd.DataFrame(
    MAIN_NESTED_CV_SUMMARY_ROWS
)

display(MAIN_NESTED_CV_SUMMARY_DF.round(4))

##6.8.Threshold Sensitivity Summary

Compare fixed, Youden-optimized, and F1-optimized classification thresholds for the outcome-specific models highlighted in the manuscript.

In [ ]:
# ================================================================
# 6.8 Threshold Sensitivity Summary
# ================================================================

THRESHOLD_SENSITIVITY_ROWS = []

for target, algorithm in SELECTED_MODELS.items():
    fold_metrics = MAIN_NESTED_CV_RESULTS[
        (algorithm, target)
    ]["Fold_Metrics"]

    for threshold_type in [
        "Fixed_0.50",
        "Inner_Youden",
        "Inner_F1"
    ]:
        subset = fold_metrics[
            fold_metrics["Threshold_Type"] == threshold_type
        ]

        THRESHOLD_SENSITIVITY_ROWS.append({
            "Algorithm": algorithm,
            "Target": target,
            "Threshold_Criterion": threshold_type,
            "Threshold_Mean": subset["Threshold"].mean(),
            "Threshold_SD": subset["Threshold"].std(ddof=1),
            "Sensitivity_Mean": subset["Recall_Sensitivity"].mean(),
            "Sensitivity_SD": subset["Recall_Sensitivity"].std(ddof=1),
            "Specificity_Mean": subset["Specificity"].mean(),
            "Specificity_SD": subset["Specificity"].std(ddof=1),
            "Precision_Mean": subset["Precision"].mean(),
            "Precision_SD": subset["Precision"].std(ddof=1),
            "F1_Mean": subset["F1"].mean(),
            "F1_SD": subset["F1"].std(ddof=1)
        })

THRESHOLD_SENSITIVITY_DF = pd.DataFrame(
    THRESHOLD_SENSITIVITY_ROWS
)

display(THRESHOLD_SENSITIVITY_DF.round(4))

##6.9.Calibration Curves

Generate calibration plots from pooled out-of-fold probabilities for each algorithm and medal outcome.

In [ ]:
# ================================================================
# 6.9 Calibration Curves
# ================================================================

CALIBRATION_TABLES = {}

for algorithm in ALGORITHMS:
    for target in TARGETS:
        predictions = (
            MAIN_NESTED_CV_RESULTS[
                (algorithm, target)
            ]["OOF_Predictions"]
            .sort_values("Athlete_Index")
            .reset_index(drop=True)
        )

        calibration_table = create_calibration_table(
            y_true=predictions["Observed_Outcome"],
            probabilities=predictions["Predicted_Probability"],
            algorithm=algorithm,
            target=target
        )

        CALIBRATION_TABLES[(algorithm, target)] = calibration_table

        plt.figure(figsize=(6, 6))
        plt.plot(
            [0, 1],
            [0, 1],
            linestyle="--",
            label="Perfect calibration"
        )
        plt.plot(
            calibration_table["Mean_Predicted_Probability"],
            calibration_table["Observed_Event_Proportion"],
            marker="o",
            label="Observed"
        )
        plt.xlabel("Mean predicted probability")
        plt.ylabel("Observed event proportion")
        plt.title(f"{algorithm} – {target}")
        plt.legend()
        plt.tight_layout()
        plt.savefig(
            OUTPUT_FOLDERS["figures"]
            / f"{algorithm}_{target}_Calibration.png",
            dpi=300,
            bbox_inches="tight"
        )
        plt.close()

print("Calibration curves were generated.")

##6.10.Nested Cross-Validation Results Export

Export fold-level results, out-of-fold probabilities, selected hyperparameters, threshold analyses, calibration data, and consolidated performance tables.

In [ ]:
# ================================================================
# 6.10 Nested Cross-Validation Results Export
# ================================================================

for algorithm in ALGORITHMS:
    for target in TARGETS:
        result = MAIN_NESTED_CV_RESULTS[(algorithm, target)]

        for result_name, result_table in result.items():
            result_table.to_csv(
                OUTPUT_FOLDERS["nested_cv"]
                / f"{algorithm}_{target}_{result_name}.csv",
                index=False
            )

        CALIBRATION_TABLES[(algorithm, target)].to_csv(
            OUTPUT_FOLDERS["nested_cv"]
            / f"{algorithm}_{target}_Calibration_Points.csv",
            index=False
        )

MAIN_NESTED_CV_SUMMARY_DF.to_csv(
    OUTPUT_FOLDERS["nested_cv"]
    / "Main_Nested_CV_Summary.csv",
    index=False
)

THRESHOLD_SENSITIVITY_DF.to_csv(
    OUTPUT_FOLDERS["nested_cv"]
    / "Threshold_Sensitivity_Selected_Models.csv",
    index=False
)

NESTED_CV_EXCEL_PATH = (
    OUTPUT_FOLDERS["tables"]
    / "Independent_Nested_Cross_Validation.xlsx"
)

with pd.ExcelWriter(
    NESTED_CV_EXCEL_PATH,
    engine="xlsxwriter"
) as writer:
    MAIN_NESTED_CV_SUMMARY_DF.to_excel(
        writer,
        sheet_name="Performance_Summary",
        index=False
    )

    THRESHOLD_SENSITIVITY_DF.to_excel(
        writer,
        sheet_name="Threshold_Sensitivity",
        index=False
    )

    NESTED_CV_CONFIGURATION_DF.to_excel(
        writer,
        sheet_name="Configuration",
        index=False
    )

    for algorithm in ALGORITHMS:
        for target in TARGETS:
            prefix = f"{algorithm[:4]}_{target}"
            result = MAIN_NESTED_CV_RESULTS[(algorithm, target)]

            result["Fold_Metrics"].to_excel(
                writer,
                sheet_name=f"{prefix}_Metrics"[:31],
                index=False
            )

            result["Best_Parameters"].to_excel(
                writer,
                sheet_name=f"{prefix}_Params"[:31],
                index=False
            )

display(MAIN_NESTED_CV_SUMMARY_DF.round(4))
print(f"Nested-CV results saved to: {OUTPUT_FOLDERS['nested_cv']}")
print(f"Combined workbook saved to: {NESTED_CV_EXCEL_PATH}")

#7.Complete-Case Sensitivity Analysis

Evaluate the robustness of the primary machine-learning models using complete-case data without Movement Performance Score imputation.

##7.1.Complete-Case Dataset Construction

Construct the complete-case dataset by excluding athletes with missing Movement Performance Score variables while retaining the same predictor definitions used in the primary analyses.

In [ ]:
# ================================================================
# 7.1 Complete-Case Cohort Construction
# ================================================================

REQUIRED_COMPLETE_CASE_MPS = [feature for feature in MPS_FEATURES if feature in df.columns]

if len(REQUIRED_COMPLETE_CASE_MPS) != len(MPS_FEATURES):
    missing_mps_columns = [feature for feature in MPS_FEATURES if feature not in df.columns]
    raise ValueError(f"Required MPS columns are missing: {missing_mps_columns}")

MPS_COMPLETE_MASK = df[REQUIRED_COMPLETE_CASE_MPS].apply(
    pd.to_numeric, errors="coerce"
).notna().all(axis=1)

COMPLETE_CASE_DF = df.loc[MPS_COMPLETE_MASK].copy()
EXCLUDED_FROM_COMPLETE_CASE_DF = df.loc[~MPS_COMPLETE_MASK].copy()

COMPLETE_CASE_COHORT_SUMMARY_DF = pd.DataFrame([{
    "Full_Cohort_n": len(df),
    "Complete_Case_Cohort_n": len(COMPLETE_CASE_DF),
    "Complete_Case_Percent": 100 * len(COMPLETE_CASE_DF) / len(df),
    "Excluded_for_Missing_MPS_n": len(EXCLUDED_FROM_COMPLETE_CASE_DF),
    "Excluded_for_Missing_MPS_Percent": 100 * len(EXCLUDED_FROM_COMPLETE_CASE_DF) / len(df),
    "Required_MPS_Variable_Count": len(REQUIRED_COMPLETE_CASE_MPS),
    "Required_MPS_Variables": " | ".join(REQUIRED_COMPLETE_CASE_MPS)
}])

COMPLETE_CASE_OUTCOME_SUMMARY_DF = pd.DataFrame([
    {
        "Target": target,
        "Complete_Case_n": int(COMPLETE_CASE_DF[target].notna().sum()),
        "Positive_n": int((pd.to_numeric(COMPLETE_CASE_DF[target], errors="coerce") == 1).sum()),
        "Positive_Percent": (
            100 * (pd.to_numeric(COMPLETE_CASE_DF[target], errors="coerce") == 1).sum()
            / pd.to_numeric(COMPLETE_CASE_DF[target], errors="coerce").notna().sum()
        )
    }
    for target in TARGETS
])

display(COMPLETE_CASE_COHORT_SUMMARY_DF.round(3))
display(COMPLETE_CASE_OUTCOME_SUMMARY_DF.round(3))

print(f"Complete-case cohort: {len(COMPLETE_CASE_DF):,} of {len(df):,} athletes")

##7.2.Complete-Case Pipeline

Construct a nested cross-validation pipeline without missing-value imputation. Standardization, one-hot encoding, SMOTE, and model fitting remain restricted to the training data within each outer fold.

In [ ]:
# ================================================================
# 7.2 Complete-Case Pipeline
# ================================================================

def create_complete_case_preprocessor(
    numeric_features: list[str],
    categorical_features: list[str]
) -> ColumnTransformer:
    """Create preprocessing without imputation for complete-case analyses."""
    transformers = []

    if numeric_features:
        numeric_pipeline = SklearnPipeline([
            ("scaler", StandardScaler())
        ])
        transformers.append(("numeric", numeric_pipeline, numeric_features))

    if categorical_features:
        categorical_pipeline = SklearnPipeline([
            ("onehot", create_one_hot_encoder())
        ])
        transformers.append(("categorical", categorical_pipeline, categorical_features))

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        verbose_feature_names_out=False
    )


def create_complete_case_pipeline(
    algorithm: str,
    numeric_features: list[str],
    categorical_features: list[str],
    use_smote: bool = True
) -> ImbalancedPipeline:
    """Create a complete-case model pipeline without imputation."""
    steps = [(
        "preprocessor",
        create_complete_case_preprocessor(
            numeric_features=numeric_features,
            categorical_features=categorical_features
        )
    )]

    if use_smote:
        steps.append((
            "smote",
            SMOTE(k_neighbors=SMOTE_K_NEIGHBORS, random_state=SEED)
        ))

    steps.append(("model", create_model_estimator(algorithm)))
    return ImbalancedPipeline(steps)


COMPLETE_CASE_MODEL_CONFIGURATION_DF = pd.DataFrame([
    {
        "Target": target,
        "Algorithm": algorithm,
        "Feature_Count": len(FFS_PEAK_FEATURES[(algorithm, target)]),
        "Features": " | ".join(FFS_PEAK_FEATURES[(algorithm, target)]),
        "Imputation": "None",
        "SMOTE_k_Neighbors": SMOTE_K_NEIGHBORS
    }
    for target, algorithm in SELECTED_MODELS.items()
])

display(COMPLETE_CASE_MODEL_CONFIGURATION_DF)

##7.3.Complete-Case Nested Cross-Validation Function

Repeat five-outer-fold and three-inner-fold nested cross-validation for the selected outcome-specific models. Athletes with missing values in any required model predictor are excluded before cross-validation, and no imputation is performed.

In [ ]:
# ================================================================
# 7.3 Complete-Case Nested Cross-Validation Function
# ================================================================

def run_complete_case_nested_cv(
    data: pd.DataFrame,
    features: list[str],
    target: str,
    algorithm: str
) -> dict[str, pd.DataFrame]:
    """Run nested CV without imputation in the complete-case cohort."""
    analysis_columns = list(dict.fromkeys(features + [target]))
    analysis_data = data[analysis_columns].copy()

    categorical_features = [
        feature for feature in features
        if feature in CATEGORICAL_MODEL_FEATURES
    ]
    numeric_features = [
        feature for feature in features
        if feature not in categorical_features
    ]

    for feature in numeric_features:
        analysis_data[feature] = pd.to_numeric(
            analysis_data[feature], errors="coerce"
        )

    for feature in categorical_features:
        analysis_data[feature] = analysis_data[feature].astype("object")

    analysis_data[target] = pd.to_numeric(
        analysis_data[target], errors="coerce"
    )

    missing_before_exclusion = analysis_data[features].isna().any(axis=1)
    excluded_for_model_features_n = int(missing_before_exclusion.sum())

    analysis_data = (
        analysis_data
        .dropna(subset=features + [target])
        .reset_index()
        .rename(columns={"index": "Original_Index"})
    )

    X = analysis_data[features].copy()
    y = analysis_data[target].astype(int).reset_index(drop=True)

    outer_splits = list(
        StratifiedKFold(
            n_splits=OUTER_CV_SPLITS,
            shuffle=True,
            random_state=SEED
        ).split(X, y)
    )

    fold_metric_rows, prediction_rows = [], []
    parameter_rows, threshold_rows = [], []

    for outer_fold, (train_index, test_index) in enumerate(outer_splits, start=1):
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]

        pipeline = create_complete_case_pipeline(
            algorithm=algorithm,
            numeric_features=numeric_features,
            categorical_features=categorical_features,
            use_smote=True
        )

        search = RandomizedSearchCV(
            estimator=pipeline,
            param_distributions=SEARCH_SPACES[algorithm],
            n_iter=get_search_iterations(algorithm),
            scoring="roc_auc",
            cv=INNER_CV,
            random_state=SEED,
            n_jobs=-1,
            refit=True,
            return_train_score=False,
            error_score=np.nan
        )

        search.fit(X_train, y_train)
        best_pipeline = search.best_estimator_
        test_probability = best_pipeline.predict_proba(X_test)[:, 1]

        inner_training_probability = cross_val_predict(
            clone(best_pipeline),
            X_train,
            y_train,
            cv=INNER_CV,
            method="predict_proba",
            n_jobs=-1
        )[:, 1]

        thresholds = {
            "Fixed_0.50": FIXED_CLASSIFICATION_THRESHOLD,
            "Inner_Youden": calculate_youden_threshold(
                y_train, inner_training_probability
            ),
            "Inner_F1": calculate_f1_threshold(
                y_train, inner_training_probability
            )
        }

        probability_metrics = {
            "ROC_AUC": roc_auc_score(y_test, test_probability),
            "PR_AUC_Average_Precision": average_precision_score(
                y_test, test_probability
            ),
            "Brier_Score": brier_score_loss(y_test, test_probability)
        }

        for threshold_type, threshold in thresholds.items():
            fold_metric_rows.append({
                "Algorithm": algorithm,
                "Target": target,
                "Analysis": "Complete_Case_No_Imputation",
                "Outer_Fold": outer_fold,
                "Threshold_Type": threshold_type,
                "Training_n": len(train_index),
                "Test_n": len(test_index),
                "Training_Positive_Prevalence": y_train.mean(),
                "Test_Positive_Prevalence": y_test.mean(),
                **probability_metrics,
                **calculate_threshold_metrics(
                    y_test, test_probability, threshold
                )
            })

            threshold_rows.append({
                "Algorithm": algorithm,
                "Target": target,
                "Analysis": "Complete_Case_No_Imputation",
                "Outer_Fold": outer_fold,
                "Threshold_Type": threshold_type,
                "Threshold": threshold
            })

        for position, local_index in enumerate(test_index):
            prediction_rows.append({
                "Algorithm": algorithm,
                "Target": target,
                "Analysis": "Complete_Case_No_Imputation",
                "Outer_Fold": outer_fold,
                "Athlete_Index": int(
                    analysis_data.iloc[local_index]["Original_Index"]
                ),
                "Observed_Outcome": int(y_test.iloc[position]),
                "Predicted_Probability": float(test_probability[position])
            })

        parameter_rows.append({
            "Algorithm": algorithm,
            "Target": target,
            "Analysis": "Complete_Case_No_Imputation",
            "Outer_Fold": outer_fold,
            "Best_Inner_ROC_AUC": float(search.best_score_),
            "Best_Parameters": json.dumps(
                search.best_params_, sort_keys=True
            )
        })

        print(
            f"Complete case | {algorithm:8s} | {target} | "
            f"Fold {outer_fold}/{OUTER_CV_SPLITS} | "
            f"ROC–AUC={probability_metrics['ROC_AUC']:.4f}"
        )

    analysis_metadata = pd.DataFrame([{
        "Algorithm": algorithm,
        "Target": target,
        "Initial_MPS_Complete_Cohort_n": len(data),
        "Excluded_for_Additional_Model_Missingness_n": excluded_for_model_features_n,
        "Final_Analysis_n": len(analysis_data),
        "Feature_Count": len(features),
        "Features": " | ".join(features)
    }])

    return {
        "Fold_Metrics": pd.DataFrame(fold_metric_rows),
        "OOF_Predictions": pd.DataFrame(prediction_rows),
        "Best_Parameters": pd.DataFrame(parameter_rows),
        "Thresholds": pd.DataFrame(threshold_rows),
        "Analysis_Metadata": analysis_metadata
    }

##7.4.Complete-Case Analysis Execution

Run the complete-case nested cross-validation procedure for the outcome-specific models reported in the manuscript using their corresponding FFS peak feature subsets.

In [ ]:
# ================================================================
# 7.4 Complete-Case Analysis Execution
# ================================================================

COMPLETE_CASE_RESULTS = {}

for target, algorithm in SELECTED_MODELS.items():
    complete_case_features = FFS_PEAK_FEATURES[(algorithm, target)]

    COMPLETE_CASE_RESULTS[(algorithm, target)] = run_complete_case_nested_cv(
        data=COMPLETE_CASE_DF,
        features=complete_case_features,
        target=target,
        algorithm=algorithm
    )

##7.5.Complete-Case Performance Summary

Summarize discrimination, calibration, and classification performance from the complete-case outer folds using the fixed 0.50 classification threshold.

In [ ]:
# ================================================================
# 7.5 Complete-Case Performance Summary
# ================================================================

COMPLETE_CASE_SUMMARY_ROWS = []

for target, algorithm in SELECTED_MODELS.items():
    result = COMPLETE_CASE_RESULTS[(algorithm, target)]
    fold_metrics = result["Fold_Metrics"]
    fixed_metrics = fold_metrics[
        fold_metrics["Threshold_Type"] == "Fixed_0.50"
    ].copy()

    predictions = (
        result["OOF_Predictions"]
        .sort_values("Athlete_Index")
        .reset_index(drop=True)
    )

    metadata = result["Analysis_Metadata"].iloc[0]

    summary = {
        "Algorithm": algorithm,
        "Target": target,
        "Analysis": "Complete_Case_No_Imputation",
        "Athletes_n": int(metadata["Final_Analysis_n"]),
        "Feature_Count": int(metadata["Feature_Count"]),
        "Positive_Prevalence": predictions["Observed_Outcome"].mean()
    }

    for metric in [
        "ROC_AUC", "PR_AUC_Average_Precision", "Brier_Score",
        "Accuracy", "Precision", "Recall_Sensitivity",
        "Specificity", "F1", "Negative_Predictive_Value"
    ]:
        summary[f"{metric}_Mean"] = fixed_metrics[metric].mean()
        summary[f"{metric}_SD"] = fixed_metrics[metric].std(ddof=1)

    summary.update(
        calculate_calibration_intercept_slope(
            predictions["Observed_Outcome"],
            predictions["Predicted_Probability"]
        )
    )

    COMPLETE_CASE_SUMMARY_ROWS.append(summary)

COMPLETE_CASE_SUMMARY_DF = pd.DataFrame(
    COMPLETE_CASE_SUMMARY_ROWS
)

display(COMPLETE_CASE_SUMMARY_DF.round(4))

##7.6.Comparison with the Primary Analysis

Compare complete-case performance with the corresponding primary imputed analysis. Performance differences are calculated as complete-case values minus primary-analysis values.

In [ ]:
# ================================================================
# 7.6 Comparison with the Primary Analysis
# ================================================================

PRIMARY_SELECTED_MODEL_SUMMARY_DF = pd.DataFrame([
    MAIN_NESTED_CV_SUMMARY_DF.loc[
        (MAIN_NESTED_CV_SUMMARY_DF["Algorithm"] == algorithm)
        & (MAIN_NESTED_CV_SUMMARY_DF["Target"] == target)
    ].iloc[0].to_dict()
    for target, algorithm in SELECTED_MODELS.items()
])

comparison_columns = [
    "Algorithm", "Target", "Athletes_n",
    "ROC_AUC_Mean", "ROC_AUC_SD",
    "PR_AUC_Average_Precision_Mean",
    "PR_AUC_Average_Precision_SD",
    "Brier_Score_Mean", "Brier_Score_SD",
    "Accuracy_Mean", "Accuracy_SD",
    "Precision_Mean", "Precision_SD",
    "Recall_Sensitivity_Mean", "Recall_Sensitivity_SD",
    "F1_Mean", "F1_SD",
    "Calibration_Intercept", "Calibration_Slope"
]

primary_comparison = PRIMARY_SELECTED_MODEL_SUMMARY_DF[
    [c for c in comparison_columns if c in PRIMARY_SELECTED_MODEL_SUMMARY_DF.columns]
].copy()

complete_comparison = COMPLETE_CASE_SUMMARY_DF[
    [c for c in comparison_columns if c in COMPLETE_CASE_SUMMARY_DF.columns]
].copy()

PRIMARY_COMPLETE_CASE_COMPARISON_DF = primary_comparison.merge(
    complete_comparison,
    on=["Algorithm", "Target"],
    suffixes=("_Primary", "_CompleteCase")
)

for metric in [
    "ROC_AUC_Mean",
    "PR_AUC_Average_Precision_Mean",
    "Brier_Score_Mean",
    "Accuracy_Mean",
    "Precision_Mean",
    "Recall_Sensitivity_Mean",
    "F1_Mean",
    "Calibration_Intercept",
    "Calibration_Slope"
]:
    primary_column = f"{metric}_Primary"
    complete_column = f"{metric}_CompleteCase"

    if primary_column in PRIMARY_COMPLETE_CASE_COMPARISON_DF.columns:
        PRIMARY_COMPLETE_CASE_COMPARISON_DF[
            f"{metric}_Difference_CompleteCase_minus_Primary"
        ] = (
            PRIMARY_COMPLETE_CASE_COMPARISON_DF[complete_column]
            - PRIMARY_COMPLETE_CASE_COMPARISON_DF[primary_column]
        )

display(PRIMARY_COMPLETE_CASE_COMPARISON_DF.round(4))

##7.7.Complete-Case Calibration Curves

Generate calibration tables and calibration plots from the pooled complete-case out-of-fold probabilities.

In [ ]:
# ================================================================
# 7.7 Complete-Case Calibration Curves
# ================================================================

COMPLETE_CASE_CALIBRATION_TABLES = {}

for target, algorithm in SELECTED_MODELS.items():
    predictions = (
        COMPLETE_CASE_RESULTS[(algorithm, target)]["OOF_Predictions"]
        .sort_values("Athlete_Index")
        .reset_index(drop=True)
    )

    calibration_table = create_calibration_table(
        y_true=predictions["Observed_Outcome"],
        probabilities=predictions["Predicted_Probability"],
        algorithm=algorithm,
        target=target
    )

    calibration_table["Analysis"] = "Complete_Case_No_Imputation"
    COMPLETE_CASE_CALIBRATION_TABLES[(algorithm, target)] = calibration_table

    plt.figure(figsize=(6, 6))
    plt.plot([0, 1], [0, 1], linestyle="--", label="Perfect calibration")
    plt.plot(
        calibration_table["Mean_Predicted_Probability"],
        calibration_table["Observed_Event_Proportion"],
        marker="o",
        label="Complete case"
    )
    plt.xlabel("Mean predicted probability")
    plt.ylabel("Observed event proportion")
    plt.title(f"Complete Case: {algorithm} – {target}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(
        OUTPUT_FOLDERS["figures"]
        / f"{algorithm}_{target}_CompleteCase_Calibration.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.close()

print("Complete-case calibration curves were generated.")

##7.8.Complete-Case Results Export

Export the complete-case cohort description, fold-level performance, out-of-fold predictions, selected hyperparameters, calibration results, and primary-versus-complete-case comparisons.

In [ ]:
# ================================================================
# 7.8 Complete-Case Results Export
# ================================================================

COMPLETE_CASE_COHORT_SUMMARY_DF.to_csv(
    OUTPUT_FOLDERS["complete_case"] / "Complete_Case_Cohort_Summary.csv",
    index=False
)

COMPLETE_CASE_OUTCOME_SUMMARY_DF.to_csv(
    OUTPUT_FOLDERS["complete_case"] / "Complete_Case_Outcome_Summary.csv",
    index=False
)

COMPLETE_CASE_MODEL_CONFIGURATION_DF.to_csv(
    OUTPUT_FOLDERS["complete_case"] / "Complete_Case_Model_Configuration.csv",
    index=False
)

COMPLETE_CASE_SUMMARY_DF.to_csv(
    OUTPUT_FOLDERS["complete_case"] / "Complete_Case_Performance_Summary.csv",
    index=False
)

PRIMARY_COMPLETE_CASE_COMPARISON_DF.to_csv(
    OUTPUT_FOLDERS["complete_case"] / "Primary_vs_Complete_Case_Comparison.csv",
    index=False
)

for target, algorithm in SELECTED_MODELS.items():
    result = COMPLETE_CASE_RESULTS[(algorithm, target)]

    for result_name, result_table in result.items():
        result_table.to_csv(
            OUTPUT_FOLDERS["complete_case"]
            / f"{algorithm}_{target}_CompleteCase_{result_name}.csv",
            index=False
        )

    COMPLETE_CASE_CALIBRATION_TABLES[(algorithm, target)].to_csv(
        OUTPUT_FOLDERS["complete_case"]
        / f"{algorithm}_{target}_CompleteCase_Calibration_Points.csv",
        index=False
    )

COMPLETE_CASE_EXCEL_PATH = (
    OUTPUT_FOLDERS["tables"]
    / "Complete_Case_Sensitivity_Analysis.xlsx"
)

with pd.ExcelWriter(COMPLETE_CASE_EXCEL_PATH, engine="xlsxwriter") as writer:
    COMPLETE_CASE_COHORT_SUMMARY_DF.to_excel(
        writer, sheet_name="Cohort_Summary", index=False
    )
    COMPLETE_CASE_OUTCOME_SUMMARY_DF.to_excel(
        writer, sheet_name="Outcome_Summary", index=False
    )
    COMPLETE_CASE_MODEL_CONFIGURATION_DF.to_excel(
        writer, sheet_name="Model_Configuration", index=False
    )
    COMPLETE_CASE_SUMMARY_DF.to_excel(
        writer, sheet_name="Performance_Summary", index=False
    )
    PRIMARY_COMPLETE_CASE_COMPARISON_DF.to_excel(
        writer, sheet_name="Primary_Comparison", index=False
    )

    for target, algorithm in SELECTED_MODELS.items():
        prefix = f"{algorithm[:4]}_{target}"
        result = COMPLETE_CASE_RESULTS[(algorithm, target)]

        result["Fold_Metrics"].to_excel(
            writer,
            sheet_name=f"{prefix}_Metrics"[:31],
            index=False
        )
        result["Best_Parameters"].to_excel(
            writer,
            sheet_name=f"{prefix}_Params"[:31],
            index=False
        )

display(COMPLETE_CASE_SUMMARY_DF.round(4))
display(PRIMARY_COMPLETE_CASE_COMPARISON_DF.round(4))

print(f"Complete-case results saved to: {OUTPUT_FOLDERS['complete_case']}")
print(f"Combined workbook saved to: {COMPLETE_CASE_EXCEL_PATH}")

#8.Sports-Discipline-Excluded Sensitivity Analysis

Evaluate whether predictive performance is retained after excluding sports discipline. Forward feature selection is repeated without sports discipline, and the resulting peak subsets are evaluated using nested cross-validation and compared with the corresponding full FFS peak models.

##8.1.Sports-Discipline-Excluded Predictor Matrix

Create the predictor matrix for the sports-discipline-excluded sensitivity analysis by removing SD from the post-VIF predictor set while retaining all other eligible predictors.

In [ ]:
# ================================================================
# 8.1 Sports-Discipline-Excluded Predictor Matrix
# ================================================================

if "SD" not in X_POST_VIF.columns:
    print(
        "Warning: SD was not present in the post-VIF predictor matrix. "
        "The no-SD analysis will otherwise use the same post-VIF predictors."
    )

NO_SD_FEATURES = [feature for feature in POST_VIF_FEATURES if feature != "SD"]
X_NO_SD = X_POST_VIF[NO_SD_FEATURES].copy()

NO_SD_PREDICTOR_SUMMARY_DF = pd.DataFrame([{
    "Full_Post_VIF_Feature_Count": len(POST_VIF_FEATURES),
    "No_SD_Post_VIF_Feature_Count": len(NO_SD_FEATURES),
    "Sports_Discipline_Removed": "SD" in POST_VIF_FEATURES,
    "Removed_Feature": "SD" if "SD" in POST_VIF_FEATURES else "Not present after VIF",
    "No_SD_Features": " | ".join(NO_SD_FEATURES)
}])

display(NO_SD_PREDICTOR_SUMMARY_DF)
print(f"No-SD predictor matrix: {X_NO_SD.shape[0]:,} rows × {X_NO_SD.shape[1]:,} columns")

##8.2.No-SD Forward Feature Selection

Repeat forward feature selection after excluding sports discipline. Selection is performed separately for each outcome-specific model using repeated stratified 5-fold cross-validation with 10 repeats.

In [ ]:
# ================================================================
# 8.2 No-SD Forward Feature Selection
# ================================================================

NO_SD_FFS_HISTORY = {}
NO_SD_FFS_SELECTION_ORDER = {}

for target, algorithm in SELECTED_MODELS.items():
    outcome = pd.to_numeric(df[target], errors="coerce")
    valid_rows = outcome.notna()

    X_no_sd_ffs = X_NO_SD.loc[valid_rows].reset_index(drop=True)
    y_no_sd_ffs = outcome.loc[valid_rows].astype(int).reset_index(drop=True)

    no_sd_history, no_sd_selection_order = run_forward_feature_selection(
        X=X_no_sd_ffs,
        y=y_no_sd_ffs,
        algorithm=algorithm,
        target=target
    )

    NO_SD_FFS_HISTORY[(algorithm, target)] = no_sd_history
    NO_SD_FFS_SELECTION_ORDER[(algorithm, target)] = no_sd_selection_order

    print(
        f"No-SD FFS completed | {algorithm:8s} | {target} | "
        f"Candidate features={X_no_sd_ffs.shape[1]}"
    )

##8.3.No-SD Peak Feature Subset Identification

Identify the no-SD feature subset corresponding to the highest mean ROC–AUC for each selected outcome-specific model.

In [ ]:
# ================================================================
# 8.3 No-SD Peak Feature Subset Identification
# ================================================================

NO_SD_FFS_PEAK_FEATURES = {}
NO_SD_FFS_PEAK_SUMMARY_ROWS = []

for target, algorithm in SELECTED_MODELS.items():
    history = NO_SD_FFS_HISTORY[(algorithm, target)]
    selection_order = NO_SD_FFS_SELECTION_ORDER[(algorithm, target)]

    peak_index = history["ROC_AUC_Mean"].idxmax()
    peak_row = history.loc[peak_index]
    peak_feature_count = int(peak_row["Num_Features"])
    peak_features = selection_order[:peak_feature_count]

    if "SD" in peak_features:
        raise RuntimeError(
            f"SD was unexpectedly retained in the no-SD feature subset for {algorithm}-{target}."
        )

    NO_SD_FFS_PEAK_FEATURES[(algorithm, target)] = peak_features

    full_peak_features = FFS_PEAK_FEATURES[(algorithm, target)]

    NO_SD_FFS_PEAK_SUMMARY_ROWS.append({
        "Algorithm": algorithm,
        "Target": target,
        "Full_FFS_Peak_Feature_Count": len(full_peak_features),
        "No_SD_FFS_Peak_Feature_Count": peak_feature_count,
        "No_SD_Peak_ROC_AUC_Mean": float(peak_row["ROC_AUC_Mean"]),
        "No_SD_Peak_ROC_AUC_SD": float(peak_row["ROC_AUC_SD"]),
        "No_SD_Peak_Accuracy_Mean": float(peak_row["Accuracy_Mean"]),
        "No_SD_Peak_Accuracy_SD": float(peak_row["Accuracy_SD"]),
        "Full_FFS_Peak_Features": " | ".join(full_peak_features),
        "No_SD_FFS_Peak_Features": " | ".join(peak_features)
    })

NO_SD_FFS_PEAK_SUMMARY_DF = pd.DataFrame(NO_SD_FFS_PEAK_SUMMARY_ROWS)

display(NO_SD_FFS_PEAK_SUMMARY_DF)

##8.4.Paired Outer-Fold Construction

Generate identical stratified outer folds for the full and no-SD models within each medal outcome. Using the same test athletes permits paired comparison of pooled out-of-fold probabilities.

In [ ]:
# ================================================================
# 8.4 Paired Outer-Fold Construction
# ================================================================

PAIRED_OUTER_SPLITS = {}
PAIRED_OUTER_FOLD_SUMMARY_ROWS = []

for target in TARGETS:
    outcome = pd.to_numeric(df[target], errors="coerce")
    valid_rows = outcome.notna()

    paired_y = outcome.loc[valid_rows].astype(int).reset_index(drop=True)
    placeholder_X = pd.DataFrame({"Placeholder": np.zeros(len(paired_y))})

    target_splits = list(
        StratifiedKFold(
            n_splits=OUTER_CV_SPLITS,
            shuffle=True,
            random_state=SEED
        ).split(placeholder_X, paired_y)
    )

    PAIRED_OUTER_SPLITS[target] = target_splits

    for outer_fold, (train_index, test_index) in enumerate(target_splits, start=1):
        PAIRED_OUTER_FOLD_SUMMARY_ROWS.append({
            "Target": target,
            "Outer_Fold": outer_fold,
            "Training_n": len(train_index),
            "Test_n": len(test_index),
            "Training_Positive_n": int(paired_y.iloc[train_index].sum()),
            "Test_Positive_n": int(paired_y.iloc[test_index].sum()),
            "Training_Positive_Prevalence": paired_y.iloc[train_index].mean(),
            "Test_Positive_Prevalence": paired_y.iloc[test_index].mean()
        })

PAIRED_OUTER_FOLD_SUMMARY_DF = pd.DataFrame(
    PAIRED_OUTER_FOLD_SUMMARY_ROWS
)

display(PAIRED_OUTER_FOLD_SUMMARY_DF.round(4))

##8.5.Full and No-SD Paired Nested Cross-Validation

Evaluate the full FFS peak subset and the corresponding no-SD peak subset using identical outer folds. Hyperparameter tuning, preprocessing, and SMOTE remain restricted to the training data.

In [ ]:
# ================================================================
# 8.5 Full and No-SD Paired Nested Cross-Validation
# ================================================================

PAIRED_FULL_RESULTS = {}
PAIRED_NO_SD_RESULTS = {}

for target, algorithm in SELECTED_MODELS.items():
    valid_rows = pd.to_numeric(df[target], errors="coerce").notna()
    paired_data = df.loc[valid_rows].reset_index(drop=True)

    full_features = FFS_PEAK_FEATURES[(algorithm, target)]
    no_sd_features = NO_SD_FFS_PEAK_FEATURES[(algorithm, target)]
    common_outer_splits = PAIRED_OUTER_SPLITS[target]

    PAIRED_FULL_RESULTS[(algorithm, target)] = run_nested_cross_validation(
        data=paired_data,
        features=full_features,
        target=target,
        algorithm=algorithm,
        analysis_label="Full_FFS_Peak_Paired_Rerun",
        outer_splits=common_outer_splits,
        use_smote=True
    )

    PAIRED_NO_SD_RESULTS[(algorithm, target)] = run_nested_cross_validation(
        data=paired_data,
        features=no_sd_features,
        target=target,
        algorithm=algorithm,
        analysis_label="Sports_Discipline_Excluded_FFS_Peak",
        outer_splits=common_outer_splits,
        use_smote=True
    )

    print(
        f"Paired analysis completed | {algorithm:8s} | {target} | "
        f"Full features={len(full_features)} | No-SD features={len(no_sd_features)}"
    )

##8.6.Full and No-SD Performance Summary

Summarize the fold-level performance and pooled calibration of the paired full and sports-discipline-excluded models.

In [ ]:
# ================================================================
# 8.6 Full and No-SD Performance Summary
# ================================================================

PAIRED_FULL_NO_SD_SUMMARY_ROWS = []

for target, algorithm in SELECTED_MODELS.items():
    full_result = PAIRED_FULL_RESULTS[(algorithm, target)]
    no_sd_result = PAIRED_NO_SD_RESULTS[(algorithm, target)]

    full_summary = summarize_nested_cv_result(
        result=full_result,
        feature_count=len(FFS_PEAK_FEATURES[(algorithm, target)]),
        ffs_peak_count=len(FFS_PEAK_FEATURES[(algorithm, target)])
    )
    full_summary["Analysis"] = "Full_FFS_Peak_Paired_Rerun"
    full_summary["Sports_Discipline_Included"] = True

    no_sd_summary = summarize_nested_cv_result(
        result=no_sd_result,
        feature_count=len(NO_SD_FFS_PEAK_FEATURES[(algorithm, target)]),
        ffs_peak_count=len(NO_SD_FFS_PEAK_FEATURES[(algorithm, target)])
    )
    no_sd_summary["Analysis"] = "Sports_Discipline_Excluded_FFS_Peak"
    no_sd_summary["Sports_Discipline_Included"] = False

    PAIRED_FULL_NO_SD_SUMMARY_ROWS.extend([
        full_summary,
        no_sd_summary
    ])

PAIRED_FULL_NO_SD_SUMMARY_DF = pd.DataFrame(
    PAIRED_FULL_NO_SD_SUMMARY_ROWS
)

display(PAIRED_FULL_NO_SD_SUMMARY_DF.round(4))

##8.7.Paired Out-of-Fold Predictions

Merge the full and no-SD out-of-fold probabilities by athlete to verify one-to-one correspondence before paired bootstrap comparison.

In [ ]:
# ================================================================
# 8.7 Paired Out-of-Fold Predictions
# ================================================================

PAIRED_OOF_PREDICTIONS = {}
PAIRED_PREDICTION_CHECK_ROWS = []

for target, algorithm in SELECTED_MODELS.items():
    full_predictions = (
        PAIRED_FULL_RESULTS[(algorithm, target)]["OOF_Predictions"]
        .sort_values("Athlete_Index")
        .reset_index(drop=True)
    )

    no_sd_predictions = (
        PAIRED_NO_SD_RESULTS[(algorithm, target)]["OOF_Predictions"]
        .sort_values("Athlete_Index")
        .reset_index(drop=True)
    )

    paired_predictions = full_predictions[
        [
            "Athlete_Index", "Observed_Outcome",
            "Outer_Fold", "Predicted_Probability"
        ]
    ].merge(
        no_sd_predictions[
            [
                "Athlete_Index", "Observed_Outcome",
                "Outer_Fold", "Predicted_Probability"
            ]
        ],
        on=["Athlete_Index", "Observed_Outcome", "Outer_Fold"],
        how="inner",
        suffixes=("_Full", "_NoSD"),
        validate="one_to_one"
    )

    if len(paired_predictions) != len(full_predictions):
        raise RuntimeError(
            f"Incomplete pairing for {algorithm}-{target}: "
            f"{len(paired_predictions)} paired versus {len(full_predictions)} expected."
        )

    PAIRED_OOF_PREDICTIONS[(algorithm, target)] = paired_predictions

    PAIRED_PREDICTION_CHECK_ROWS.append({
        "Algorithm": algorithm,
        "Target": target,
        "Full_OOF_n": len(full_predictions),
        "No_SD_OOF_n": len(no_sd_predictions),
        "Paired_OOF_n": len(paired_predictions),
        "Observed_Outcomes_Identical": bool(
            np.array_equal(
                full_predictions["Observed_Outcome"].to_numpy(),
                no_sd_predictions["Observed_Outcome"].to_numpy()
            )
        ),
        "Outer_Folds_Identical": bool(
            np.array_equal(
                full_predictions["Outer_Fold"].to_numpy(),
                no_sd_predictions["Outer_Fold"].to_numpy()
            )
        )
    })

PAIRED_PREDICTION_CHECK_DF = pd.DataFrame(
    PAIRED_PREDICTION_CHECK_ROWS
)

display(PAIRED_PREDICTION_CHECK_DF)

##8.8.Paired Bootstrap Comparison

Estimate differences in ROC–AUC, average precision, and Brier score between the no-SD and full models using 2,000 paired bootstrap resamples of the pooled out-of-fold predictions.

In [ ]:
# ================================================================
# 8.8 Paired Bootstrap Comparison
# ================================================================

def calculate_probability_metric(
    y_true: np.ndarray,
    probabilities: np.ndarray,
    metric: str
) -> float:
    """Calculate a threshold-independent prediction metric."""
    if metric == "ROC_AUC":
        return float(roc_auc_score(y_true, probabilities))

    if metric == "PR_AUC_Average_Precision":
        return float(average_precision_score(y_true, probabilities))

    if metric == "Brier_Score":
        return float(brier_score_loss(y_true, probabilities))

    raise ValueError(f"Unsupported bootstrap metric: {metric}")


def paired_bootstrap_model_comparison(
    y_true: pd.Series,
    full_probabilities: pd.Series,
    no_sd_probabilities: pd.Series,
    metric: str,
    repetitions: int = 2000,
    random_seed: int = 42
) -> dict:
    """Compare paired out-of-fold probabilities using bootstrap resampling."""
    y_array = np.asarray(y_true, dtype=int)
    full_array = np.asarray(full_probabilities, dtype=float)
    no_sd_array = np.asarray(no_sd_probabilities, dtype=float)

    if not (
        len(y_array) == len(full_array) == len(no_sd_array)
    ):
        raise ValueError("Paired bootstrap inputs must have identical lengths.")

    full_metric = calculate_probability_metric(
        y_array, full_array, metric
    )
    no_sd_metric = calculate_probability_metric(
        y_array, no_sd_array, metric
    )
    observed_difference = no_sd_metric - full_metric

    random_generator = np.random.default_rng(random_seed)
    bootstrap_differences = []

    while len(bootstrap_differences) < repetitions:
        sampled_indices = random_generator.integers(
            low=0,
            high=len(y_array),
            size=len(y_array)
        )

        bootstrap_y = y_array[sampled_indices]

        # ROC–AUC and average precision require both outcome classes.
        if len(np.unique(bootstrap_y)) < 2:
            continue

        bootstrap_full = calculate_probability_metric(
            bootstrap_y,
            full_array[sampled_indices],
            metric
        )
        bootstrap_no_sd = calculate_probability_metric(
            bootstrap_y,
            no_sd_array[sampled_indices],
            metric
        )

        bootstrap_differences.append(
            bootstrap_no_sd - bootstrap_full
        )

    bootstrap_differences = np.asarray(
        bootstrap_differences,
        dtype=float
    )

    return {
        "Metric": metric,
        "Full_Model_Value": full_metric,
        "No_SD_Model_Value": no_sd_metric,
        "Difference_NoSD_minus_Full": observed_difference,
        "Bootstrap_Mean_Difference": bootstrap_differences.mean(),
        "CI_Lower_95": np.percentile(bootstrap_differences, 2.5),
        "CI_Upper_95": np.percentile(bootstrap_differences, 97.5),
        "Bootstrap_Replicates": len(bootstrap_differences),
        "Two_Sided_Bootstrap_P_Value": min(
            1.0,
            2 * min(
                np.mean(bootstrap_differences <= 0),
                np.mean(bootstrap_differences >= 0)
            )
        )
    }


PAIRED_BOOTSTRAP_ROWS = []

for target, algorithm in SELECTED_MODELS.items():
    paired_predictions = PAIRED_OOF_PREDICTIONS[(algorithm, target)]

    for metric in [
        "ROC_AUC",
        "PR_AUC_Average_Precision",
        "Brier_Score"
    ]:
        comparison = paired_bootstrap_model_comparison(
            y_true=paired_predictions["Observed_Outcome"],
            full_probabilities=paired_predictions[
                "Predicted_Probability_Full"
            ],
            no_sd_probabilities=paired_predictions[
                "Predicted_Probability_NoSD"
            ],
            metric=metric,
            repetitions=BOOTSTRAP_REPETITIONS,
            random_seed=SEED
        )

        comparison.update({
            "Algorithm": algorithm,
            "Target": target,
            "Reference_Model": "Full_FFS_Peak_Paired_Rerun",
            "Comparison_Model": "Sports_Discipline_Excluded_FFS_Peak"
        })

        PAIRED_BOOTSTRAP_ROWS.append(comparison)

FULL_VS_NO_SD_BOOTSTRAP_DF = pd.DataFrame(
    PAIRED_BOOTSTRAP_ROWS
)

display(FULL_VS_NO_SD_BOOTSTRAP_DF.round(4))

##8.9.No-SD Performance Difference Summary

Create a direct summary of the fold-level performance differences between the no-SD and paired full models.

In [ ]:
# ================================================================
# 8.9 No-SD Performance Difference Summary
# ================================================================

FULL_NO_SD_WIDE_DF = (
    PAIRED_FULL_NO_SD_SUMMARY_DF
    .pivot(
        index=["Algorithm", "Target"],
        columns="Analysis"
    )
)

FULL_NO_SD_DIFFERENCE_ROWS = []

for target, algorithm in SELECTED_MODELS.items():
    full_row = PAIRED_FULL_NO_SD_SUMMARY_DF.loc[
        (PAIRED_FULL_NO_SD_SUMMARY_DF["Algorithm"] == algorithm)
        & (PAIRED_FULL_NO_SD_SUMMARY_DF["Target"] == target)
        & (
            PAIRED_FULL_NO_SD_SUMMARY_DF["Analysis"]
            == "Full_FFS_Peak_Paired_Rerun"
        )
    ].iloc[0]

    no_sd_row = PAIRED_FULL_NO_SD_SUMMARY_DF.loc[
        (PAIRED_FULL_NO_SD_SUMMARY_DF["Algorithm"] == algorithm)
        & (PAIRED_FULL_NO_SD_SUMMARY_DF["Target"] == target)
        & (
            PAIRED_FULL_NO_SD_SUMMARY_DF["Analysis"]
            == "Sports_Discipline_Excluded_FFS_Peak"
        )
    ].iloc[0]

    difference_row = {
        "Algorithm": algorithm,
        "Target": target,
        "Full_Feature_Count": full_row["Nested_CV_Feature_Count"],
        "No_SD_Feature_Count": no_sd_row["Nested_CV_Feature_Count"]
    }

    for metric in [
        "ROC_AUC_Mean",
        "PR_AUC_Average_Precision_Mean",
        "Brier_Score_Mean",
        "Accuracy_Mean",
        "Precision_Mean",
        "Recall_Sensitivity_Mean",
        "Specificity_Mean",
        "F1_Mean",
        "Calibration_Intercept",
        "Calibration_Slope"
    ]:
        if metric in full_row.index and metric in no_sd_row.index:
            difference_row[f"Full_{metric}"] = full_row[metric]
            difference_row[f"No_SD_{metric}"] = no_sd_row[metric]
            difference_row[
                f"Difference_NoSD_minus_Full_{metric}"
            ] = no_sd_row[metric] - full_row[metric]

    FULL_NO_SD_DIFFERENCE_ROWS.append(difference_row)

FULL_NO_SD_DIFFERENCE_DF = pd.DataFrame(
    FULL_NO_SD_DIFFERENCE_ROWS
)

display(FULL_NO_SD_DIFFERENCE_DF.round(4))

##8.10.Sports-Discipline-Excluded Results Export

Export no-SD forward-selection paths, selected feature subsets, paired nested cross-validation results, out-of-fold predictions, bootstrap comparisons, and consolidated supplementary tables.

In [ ]:
# ================================================================
# 8.10 Sports-Discipline-Excluded Results Export
# ================================================================

NO_SD_PREDICTOR_SUMMARY_DF.to_csv(
    OUTPUT_FOLDERS["no_sd"] / "No_SD_Predictor_Summary.csv",
    index=False
)

NO_SD_FFS_PEAK_SUMMARY_DF.to_csv(
    OUTPUT_FOLDERS["no_sd"] / "No_SD_FFS_Peak_Summary.csv",
    index=False
)

PAIRED_OUTER_FOLD_SUMMARY_DF.to_csv(
    OUTPUT_FOLDERS["no_sd"] / "Paired_Outer_Fold_Summary.csv",
    index=False
)

PAIRED_FULL_NO_SD_SUMMARY_DF.to_csv(
    OUTPUT_FOLDERS["no_sd"] / "Full_and_No_SD_Performance_Summary.csv",
    index=False
)

PAIRED_PREDICTION_CHECK_DF.to_csv(
    OUTPUT_FOLDERS["no_sd"] / "Paired_Prediction_Integrity_Check.csv",
    index=False
)

FULL_VS_NO_SD_BOOTSTRAP_DF.to_csv(
    OUTPUT_FOLDERS["no_sd"] / "Full_vs_No_SD_Paired_Bootstrap.csv",
    index=False
)

FULL_NO_SD_DIFFERENCE_DF.to_csv(
    OUTPUT_FOLDERS["no_sd"] / "Full_vs_No_SD_Performance_Differences.csv",
    index=False
)

for target, algorithm in SELECTED_MODELS.items():
    no_sd_history = NO_SD_FFS_HISTORY[(algorithm, target)]
    no_sd_order = NO_SD_FFS_SELECTION_ORDER[(algorithm, target)]
    no_sd_peak = NO_SD_FFS_PEAK_FEATURES[(algorithm, target)]

    no_sd_history.to_csv(
        OUTPUT_FOLDERS["no_sd"]
        / f"{algorithm}_{target}_NoSD_FFS_History.csv",
        index=False
    )

    pd.DataFrame({
        "Selection_Rank": np.arange(1, len(no_sd_order) + 1),
        "Feature": no_sd_order
    }).to_csv(
        OUTPUT_FOLDERS["no_sd"]
        / f"{algorithm}_{target}_NoSD_Full_Selection_Order.csv",
        index=False
    )

    pd.DataFrame({
        "Selection_Rank": np.arange(1, len(no_sd_peak) + 1),
        "Feature": no_sd_peak
    }).to_csv(
        OUTPUT_FOLDERS["no_sd"]
        / f"{algorithm}_{target}_NoSD_Peak_Features.csv",
        index=False
    )

    paired_predictions = PAIRED_OOF_PREDICTIONS[(algorithm, target)]
    paired_predictions.to_csv(
        OUTPUT_FOLDERS["no_sd"]
        / f"{algorithm}_{target}_Paired_OOF_Predictions.csv",
        index=False
    )

    for analysis_name, analysis_result in [
        ("Full", PAIRED_FULL_RESULTS[(algorithm, target)]),
        ("NoSD", PAIRED_NO_SD_RESULTS[(algorithm, target)])
    ]:
        for result_name, result_table in analysis_result.items():
            result_table.to_csv(
                OUTPUT_FOLDERS["no_sd"]
                / f"{algorithm}_{target}_{analysis_name}_{result_name}.csv",
                index=False
            )

NO_SD_EXCEL_PATH = (
    OUTPUT_FOLDERS["tables"]
    / "Sports_Discipline_Excluded_Sensitivity_Analysis.xlsx"
)

with pd.ExcelWriter(NO_SD_EXCEL_PATH, engine="xlsxwriter") as writer:
    NO_SD_PREDICTOR_SUMMARY_DF.to_excel(
        writer, sheet_name="Predictor_Summary", index=False
    )
    NO_SD_FFS_PEAK_SUMMARY_DF.to_excel(
        writer, sheet_name="NoSD_FFS_Peaks", index=False
    )
    PAIRED_FULL_NO_SD_SUMMARY_DF.to_excel(
        writer, sheet_name="Performance_Summary", index=False
    )
    FULL_NO_SD_DIFFERENCE_DF.to_excel(
        writer, sheet_name="Performance_Differences", index=False
    )
    FULL_VS_NO_SD_BOOTSTRAP_DF.to_excel(
        writer, sheet_name="Paired_Bootstrap", index=False
    )
    PAIRED_PREDICTION_CHECK_DF.to_excel(
        writer, sheet_name="Pairing_Check", index=False
    )

    for target, algorithm in SELECTED_MODELS.items():
        prefix = f"{algorithm[:4]}_{target}"

        NO_SD_FFS_HISTORY[(algorithm, target)].to_excel(
            writer,
            sheet_name=f"{prefix}_NoSD_FFS"[:31],
            index=False
        )

        PAIRED_FULL_RESULTS[(algorithm, target)]["Fold_Metrics"].to_excel(
            writer,
            sheet_name=f"{prefix}_Full_Metrics"[:31],
            index=False
        )

        PAIRED_NO_SD_RESULTS[(algorithm, target)]["Fold_Metrics"].to_excel(
            writer,
            sheet_name=f"{prefix}_NoSD_Metrics"[:31],
            index=False
        )

display(NO_SD_FFS_PEAK_SUMMARY_DF)
display(FULL_VS_NO_SD_BOOTSTRAP_DF.round(4))

print(f"No-SD results saved to: {OUTPUT_FOLDERS['no_sd']}")
print(f"Combined workbook saved to: {NO_SD_EXCEL_PATH}")

#9.Additional Supplementary Analyses

Conduct additional analyses reported in the Supplementary Materials, including prevalence-referenced precision–recall performance, continuous-versus-dichotomized TMPS comparisons, and sport-specific medal residual analyses with multiple-testing correction.

In [ ]:
# ================================================================
# 9.1 Threshold-Independent Performance Summary
# ================================================================

THRESHOLD_INDEPENDENT_ROWS = []

for algorithm in ALGORITHMS:
    for target in TARGETS:
        predictions = (
            MAIN_NESTED_CV_RESULTS[(algorithm, target)]["OOF_Predictions"]
            .sort_values("Athlete_Index")
            .reset_index(drop=True)
        )

        y_true = predictions["Observed_Outcome"].astype(int)
        probabilities = predictions["Predicted_Probability"].astype(float)
        prevalence = float(y_true.mean())
        average_precision = float(average_precision_score(y_true, probabilities))

        THRESHOLD_INDEPENDENT_ROWS.append({
            "Algorithm": algorithm,
            "Target": target,
            "Athletes_n": len(predictions),
            "Event_n": int(y_true.sum()),
            "Event_Prevalence": prevalence,
            "ROC_AUC_Pooled_OOF": roc_auc_score(y_true, probabilities),
            "PR_AUC_Average_Precision_Pooled_OOF": average_precision,
            "Average_Precision_minus_Prevalence": average_precision - prevalence,
            "Brier_Score_Pooled_OOF": brier_score_loss(y_true, probabilities)
        })

THRESHOLD_INDEPENDENT_PERFORMANCE_DF = pd.DataFrame(
    THRESHOLD_INDEPENDENT_ROWS
).sort_values(["Target", "Algorithm"]).reset_index(drop=True)

display(THRESHOLD_INDEPENDENT_PERFORMANCE_DF.round(4))

##9.2.Fold-Level PR–AUC and Brier Score Summary

Report mean and standard deviation values for ROC–AUC, average precision, and Brier score across the five outer folds of the main nested cross-validation analysis.

In [ ]:
# ================================================================
# 9.2 Fold-Level PR–AUC and Brier Score Summary
# ================================================================

FOLD_LEVEL_PROBABILITY_ROWS = []

for algorithm in ALGORITHMS:
    for target in TARGETS:
        fold_metrics = MAIN_NESTED_CV_RESULTS[
            (algorithm, target)
        ]["Fold_Metrics"]

        # Probability-based metrics are repeated for each threshold type,
        # so only one threshold row per outer fold is retained.
        probability_folds = (
            fold_metrics.loc[
                fold_metrics["Threshold_Type"] == "Fixed_0.50",
                [
                    "Outer_Fold",
                    "ROC_AUC",
                    "PR_AUC_Average_Precision",
                    "Brier_Score"
                ]
            ]
            .sort_values("Outer_Fold")
            .reset_index(drop=True)
        )

        FOLD_LEVEL_PROBABILITY_ROWS.append({
            "Target": target,
            "Algorithm": algorithm,
            "Outer_Folds": len(probability_folds),
            "ROC_AUC_Mean": probability_folds["ROC_AUC"].mean(),
            "ROC_AUC_SD": probability_folds["ROC_AUC"].std(ddof=1),
            "PR_AUC_Mean": probability_folds[
                "PR_AUC_Average_Precision"
            ].mean(),
            "PR_AUC_SD": probability_folds[
                "PR_AUC_Average_Precision"
            ].std(ddof=1),
            "Brier_Score_Mean": probability_folds["Brier_Score"].mean(),
            "Brier_Score_SD": probability_folds["Brier_Score"].std(ddof=1)
        })

FOLD_LEVEL_PROBABILITY_PERFORMANCE_DF = pd.DataFrame(
    FOLD_LEVEL_PROBABILITY_ROWS
).sort_values(["Target", "Algorithm"]).reset_index(drop=True)

display(FOLD_LEVEL_PROBABILITY_PERFORMANCE_DF.round(4))

##9.3.TMPS Analysis Dataset

Construct the observed-TMPS dataset and define both the continuous TMPS variable and the prespecified dichotomous indicator for scores of 14 or less.

In [ ]:
# ================================================================
# 9.3 TMPS Analysis Dataset
# ================================================================

if "TMPS" not in df.columns:
    raise ValueError("TMPS is required for the continuous-versus-binary analysis.")

TMPS_ANALYSIS_DATA = pd.DataFrame({
    "Athlete_Index": df.index,
    "TMPS_Continuous": pd.to_numeric(df["TMPS"], errors="coerce")
})

TMPS_ANALYSIS_DATA["TMPS_14_or_Less"] = np.where(
    TMPS_ANALYSIS_DATA["TMPS_Continuous"].notna(),
    (
        TMPS_ANALYSIS_DATA["TMPS_Continuous"] <= TMPS_CUTOFF
    ).astype(int),
    np.nan
)

for target in TARGETS:
    TMPS_ANALYSIS_DATA[target] = pd.to_numeric(
        df[target],
        errors="coerce"
    )

TMPS_ANALYSIS_SUMMARY_DF = pd.DataFrame([{
    "Full_Cohort_n": len(df),
    "Observed_TMPS_n": int(
        TMPS_ANALYSIS_DATA["TMPS_Continuous"].notna().sum()
    ),
    "Missing_TMPS_n": int(
        TMPS_ANALYSIS_DATA["TMPS_Continuous"].isna().sum()
    ),
    "TMPS_14_or_Less_n": int(
        (TMPS_ANALYSIS_DATA["TMPS_14_or_Less"] == 1).sum()
    ),
    "TMPS_Greater_Than_14_n": int(
        (TMPS_ANALYSIS_DATA["TMPS_14_or_Less"] == 0).sum()
    ),
    "TMPS_Cutoff": TMPS_CUTOFF
}])

display(TMPS_ANALYSIS_SUMMARY_DF)

##9.4.Continuous-versus-Dichotomized TMPS Comparison

Compare continuous TMPS with the TMPS ≤14 indicator using logistic regression and identical stratified five-fold cross-validation splits for each medal outcome.

In [ ]:
# ================================================================
# 9.4 Continuous-versus-Dichotomized TMPS Comparison
# ================================================================

def evaluate_single_predictor_cv(
    data: pd.DataFrame,
    predictor: str,
    target: str,
    outer_splits: list[tuple[np.ndarray, np.ndarray]]
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Evaluate one TMPS representation using fixed five-fold splits."""
    X = data[[predictor]].copy()
    y = data[target].astype(int).reset_index(drop=True)

    if predictor == "TMPS_Continuous":
        model = SklearnPipeline([
            ("scaler", StandardScaler()),
            (
                "model",
                LogisticRegression(
                    solver="liblinear",
                    random_state=SEED
                )
            )
        ])
    else:
        model = LogisticRegression(
            solver="liblinear",
            random_state=SEED
        )

    fold_rows, prediction_rows = [], []

    for fold_number, (train_index, test_index) in enumerate(
        outer_splits,
        start=1
    ):
        fitted_model = clone(model)
        fitted_model.fit(
            X.iloc[train_index],
            y.iloc[train_index]
        )

        probabilities = fitted_model.predict_proba(
            X.iloc[test_index]
        )[:, 1]

        fold_rows.append({
            "Target": target,
            "Predictor_Representation": predictor,
            "Outer_Fold": fold_number,
            "Training_n": len(train_index),
            "Test_n": len(test_index),
            "Test_Prevalence": y.iloc[test_index].mean(),
            "ROC_AUC": roc_auc_score(
                y.iloc[test_index],
                probabilities
            ),
            "PR_AUC_Average_Precision": average_precision_score(
                y.iloc[test_index],
                probabilities
            ),
            "Brier_Score": brier_score_loss(
                y.iloc[test_index],
                probabilities
            )
        })

        for position, local_index in enumerate(test_index):
            prediction_rows.append({
                "Target": target,
                "Predictor_Representation": predictor,
                "Outer_Fold": fold_number,
                "Athlete_Index": int(
                    data.iloc[local_index]["Athlete_Index"]
                ),
                "Observed_Outcome": int(
                    y.iloc[test_index].iloc[position]
                ),
                "Predicted_Probability": float(
                    probabilities[position]
                )
            })

    return pd.DataFrame(fold_rows), pd.DataFrame(prediction_rows)


TMPS_CV_FOLD_RESULTS = {}
TMPS_CV_OOF_PREDICTIONS = {}
TMPS_COMPARISON_SUMMARY_ROWS = []

for target in TARGETS:
    target_data = (
        TMPS_ANALYSIS_DATA[
            [
                "Athlete_Index",
                "TMPS_Continuous",
                "TMPS_14_or_Less",
                target
            ]
        ]
        .dropna()
        .reset_index(drop=True)
    )

    y_target = target_data[target].astype(int)

    common_splits = list(
        StratifiedKFold(
            n_splits=5,
            shuffle=True,
            random_state=SEED
        ).split(target_data, y_target)
    )

    for representation in [
        "TMPS_Continuous",
        "TMPS_14_or_Less"
    ]:
        fold_table, prediction_table = evaluate_single_predictor_cv(
            data=target_data,
            predictor=representation,
            target=target,
            outer_splits=common_splits
        )

        TMPS_CV_FOLD_RESULTS[(target, representation)] = fold_table
        TMPS_CV_OOF_PREDICTIONS[(target, representation)] = prediction_table

        TMPS_COMPARISON_SUMMARY_ROWS.append({
            "Target": target,
            "Predictor_Representation": representation,
            "Athletes_n": len(target_data),
            "Positive_n": int(y_target.sum()),
            "Positive_Prevalence": y_target.mean(),
            "ROC_AUC_Mean": fold_table["ROC_AUC"].mean(),
            "ROC_AUC_SD": fold_table["ROC_AUC"].std(ddof=1),
            "PR_AUC_Mean": fold_table[
                "PR_AUC_Average_Precision"
            ].mean(),
            "PR_AUC_SD": fold_table[
                "PR_AUC_Average_Precision"
            ].std(ddof=1),
            "Brier_Score_Mean": fold_table["Brier_Score"].mean(),
            "Brier_Score_SD": fold_table["Brier_Score"].std(ddof=1)
        })

TMPS_CONTINUOUS_BINARY_SUMMARY_DF = pd.DataFrame(
    TMPS_COMPARISON_SUMMARY_ROWS
).sort_values(
    ["Target", "Predictor_Representation"]
).reset_index(drop=True)

display(TMPS_CONTINUOUS_BINARY_SUMMARY_DF.round(4))

##9.5.Paired TMPS Representation Differences

Calculate direct fold-level differences between continuous and dichotomized TMPS using the identical cross-validation folds.

In [ ]:
# ================================================================
# 9.5 Paired TMPS Representation Differences
# ================================================================

TMPS_REPRESENTATION_DIFFERENCE_ROWS = []

for target in TARGETS:
    continuous_folds = TMPS_CV_FOLD_RESULTS[
        (target, "TMPS_Continuous")
    ].copy()

    binary_folds = TMPS_CV_FOLD_RESULTS[
        (target, "TMPS_14_or_Less")
    ].copy()

    paired_folds = continuous_folds.merge(
        binary_folds,
        on=["Target", "Outer_Fold"],
        suffixes=("_Continuous", "_Binary"),
        validate="one_to_one"
    )

    difference_row = {
        "Target": target,
        "Athletes_n": int(
            TMPS_CONTINUOUS_BINARY_SUMMARY_DF.loc[
                TMPS_CONTINUOUS_BINARY_SUMMARY_DF["Target"] == target,
                "Athletes_n"
            ].iloc[0]
        )
    }

    for metric in [
        "ROC_AUC",
        "PR_AUC_Average_Precision",
        "Brier_Score"
    ]:
        fold_differences = (
            paired_folds[f"{metric}_Continuous"]
            - paired_folds[f"{metric}_Binary"]
        )

        difference_row[f"Continuous_{metric}_Mean"] = paired_folds[
            f"{metric}_Continuous"
        ].mean()

        difference_row[f"Binary_{metric}_Mean"] = paired_folds[
            f"{metric}_Binary"
        ].mean()

        difference_row[
            f"Difference_Continuous_minus_Binary_{metric}_Mean"
        ] = fold_differences.mean()

        difference_row[
            f"Difference_Continuous_minus_Binary_{metric}_SD"
        ] = fold_differences.std(ddof=1)

    TMPS_REPRESENTATION_DIFFERENCE_ROWS.append(difference_row)

TMPS_REPRESENTATION_DIFFERENCE_DF = pd.DataFrame(
    TMPS_REPRESENTATION_DIFFERENCE_ROWS
)

display(TMPS_REPRESENTATION_DIFFERENCE_DF.round(4))

##9.6.Sport-Discipline Labels

Preserve the original sports-discipline values for analysis and optionally apply a verified code-to-label mapping for exported tables and figures.

In [ ]:
# ================================================================
# 9.6 Sport-Discipline Labels
# ================================================================

# Keep this dictionary empty unless the mapping has been verified
# directly against the source dataset or data dictionary.
SPORT_LABEL_MAP = {}

SPORT_DISCIPLINE_SERIES = df["SD"].copy()

if SPORT_LABEL_MAP:
    SPORT_DISCIPLINE_LABELS = SPORT_DISCIPLINE_SERIES.map(
        SPORT_LABEL_MAP
    ).fillna(
        SPORT_DISCIPLINE_SERIES.astype(str)
    )
else:
    SPORT_DISCIPLINE_LABELS = SPORT_DISCIPLINE_SERIES.astype(str)

SPORT_DISCIPLINE_LABEL_SUMMARY_DF = pd.DataFrame({
    "Original_SD_Value": SPORT_DISCIPLINE_SERIES,
    "Exported_Sport_Label": SPORT_DISCIPLINE_LABELS
}).drop_duplicates().sort_values(
    "Original_SD_Value",
    key=lambda series: series.astype(str)
).reset_index(drop=True)

display(SPORT_DISCIPLINE_LABEL_SUMMARY_DF)

##9.7.Benjamini–Hochberg Correction Utility

Define the Benjamini–Hochberg procedure used to control the false discovery rate within each medal-outcome family of sport-specific comparisons.

In [ ]:
# ================================================================
# 9.7 Benjamini–Hochberg Correction Utility
# ================================================================

def benjamini_hochberg_adjustment(
    p_values: pd.Series | np.ndarray
) -> np.ndarray:
    """Return monotonic Benjamini–Hochberg adjusted p-values."""
    p_array = np.asarray(p_values, dtype=float)
    number_of_tests = len(p_array)

    if number_of_tests == 0:
        return np.array([], dtype=float)

    order = np.argsort(p_array)
    ordered_p = p_array[order]

    adjusted_ordered = (
        ordered_p
        * number_of_tests
        / np.arange(1, number_of_tests + 1)
    )

    adjusted_ordered = np.minimum.accumulate(
        adjusted_ordered[::-1]
    )[::-1]

    adjusted = np.empty(number_of_tests, dtype=float)
    adjusted[order] = np.clip(adjusted_ordered, 0, 1)

    return adjusted

##9.8.Sport-Specific Omnibus Chi-Square Tests

Test the overall association between sports discipline and each medal outcome using omnibus chi-square tests.

In [ ]:
# ================================================================
# 9.8 Sport-Specific Omnibus Chi-Square Tests
# ================================================================

SPORT_CHI_SQUARE_OMNIBUS_ROWS = []
SPORT_CONTINGENCY_TABLES = {}
SPORT_EXPECTED_TABLES = {}

for target in TARGETS:
    analysis_data = pd.DataFrame({
        "Sport_Discipline": SPORT_DISCIPLINE_LABELS,
        "Outcome": pd.to_numeric(df[target], errors="coerce")
    }).dropna()

    contingency = pd.crosstab(
        analysis_data["Sport_Discipline"],
        analysis_data["Outcome"].astype(int)
    ).reindex(columns=[0, 1], fill_value=0)

    chi_square, p_value, degrees_of_freedom, expected = chi2_contingency(
        contingency,
        correction=False
    )

    expected_table = pd.DataFrame(
        expected,
        index=contingency.index,
        columns=contingency.columns
    )

    SPORT_CONTINGENCY_TABLES[target] = contingency
    SPORT_EXPECTED_TABLES[target] = expected_table

    SPORT_CHI_SQUARE_OMNIBUS_ROWS.append({
        "Target": target,
        "Athletes_n": int(contingency.to_numpy().sum()),
        "Sports_Discipline_Count": contingency.shape[0],
        "Chi_Square": chi_square,
        "Degrees_of_Freedom": degrees_of_freedom,
        "P_Value": p_value,
        "Minimum_Expected_Cell_Count": expected_table.min().min(),
        "Cells_with_Expected_Count_Below_5_n": int(
            (expected_table < 5).to_numpy().sum()
        )
    })

SPORT_CHI_SQUARE_OMNIBUS_DF = pd.DataFrame(
    SPORT_CHI_SQUARE_OMNIBUS_ROWS
)

display(SPORT_CHI_SQUARE_OMNIBUS_DF.round(4))

##9.9.Adjusted Standardized Residuals

Calculate adjusted standardized residuals for medal achievement within each sports discipline. Report the prespecified unadjusted thresholds of |Z| ≥1.96 and |Z| ≥2.58.

In [ ]:
# ================================================================
# 9.9 Adjusted Standardized Residuals
# ================================================================

SPORT_RESIDUAL_TABLES = []

for target in TARGETS:
    contingency = SPORT_CONTINGENCY_TABLES[target]
    expected_table = SPORT_EXPECTED_TABLES[target]

    total_n = contingency.to_numpy().sum()
    row_totals = contingency.sum(axis=1)
    column_totals = contingency.sum(axis=0)

    residual_rows = []

    for sport in contingency.index:
        observed_positive = float(contingency.loc[sport, 1])
        expected_positive = float(expected_table.loc[sport, 1])

        row_proportion = row_totals.loc[sport] / total_n
        positive_column_proportion = column_totals.loc[1] / total_n

        denominator = np.sqrt(
            expected_positive
            * (1 - row_proportion)
            * (1 - positive_column_proportion)
        )

        adjusted_residual = (
            (observed_positive - expected_positive) / denominator
            if denominator > 0 else np.nan
        )

        raw_p_value = (
            2 * norm.sf(abs(adjusted_residual))
            if np.isfinite(adjusted_residual) else np.nan
        )

        residual_rows.append({
            "Target": target,
            "Sport_Discipline": sport,
            "Athletes_n": int(row_totals.loc[sport]),
            "Observed_No_Medal_n": int(contingency.loc[sport, 0]),
            "Observed_Medal_n": int(observed_positive),
            "Observed_Medal_Percent": (
                100 * observed_positive / row_totals.loc[sport]
                if row_totals.loc[sport] > 0 else np.nan
            ),
            "Expected_Medal_n": expected_positive,
            "Adjusted_Standardized_Residual": adjusted_residual,
            "Raw_Two_Sided_P_Value": raw_p_value,
            "Absolute_Z_At_Least_1_96": (
                abs(adjusted_residual) >= 1.96
                if np.isfinite(adjusted_residual) else False
            ),
            "Absolute_Z_At_Least_2_58": (
                abs(adjusted_residual) >= 2.58
                if np.isfinite(adjusted_residual) else False
            )
        })

    SPORT_RESIDUAL_TABLES.append(
        pd.DataFrame(residual_rows)
    )

SPORT_ADJUSTED_RESIDUALS_DF = pd.concat(
    SPORT_RESIDUAL_TABLES,
    ignore_index=True
)

display(
    SPORT_ADJUSTED_RESIDUALS_DF.sort_values(
        ["Target", "Adjusted_Standardized_Residual"],
        ascending=[True, False]
    ).round(4)
)

##9.10.Multiple-Testing Correction

Apply Benjamini–Hochberg false discovery rate and Bonferroni corrections separately within each medal outcome. These corrected analyses are treated as sensitivity analyses to the prespecified unadjusted residual thresholds.

In [ ]:
# ================================================================
# 9.10 Multiple-Testing Correction
# ================================================================

SPORT_MULTIPLE_TESTING_TABLES = []

for target in TARGETS:
    target_table = SPORT_ADJUSTED_RESIDUALS_DF.loc[
        SPORT_ADJUSTED_RESIDUALS_DF["Target"] == target
    ].copy()

    valid_p_mask = target_table[
        "Raw_Two_Sided_P_Value"
    ].notna()

    number_of_tests = int(valid_p_mask.sum())

    target_table["Comparisons_in_Outcome_Family"] = number_of_tests
    target_table["BH_FDR_Adjusted_P"] = np.nan
    target_table["Bonferroni_Adjusted_P"] = np.nan

    target_table.loc[
        valid_p_mask,
        "BH_FDR_Adjusted_P"
    ] = benjamini_hochberg_adjustment(
        target_table.loc[
            valid_p_mask,
            "Raw_Two_Sided_P_Value"
        ]
    )

    target_table.loc[
        valid_p_mask,
        "Bonferroni_Adjusted_P"
    ] = np.minimum(
        target_table.loc[
            valid_p_mask,
            "Raw_Two_Sided_P_Value"
        ] * number_of_tests,
        1.0
    )

    target_table["BH_FDR_Significant_0_05"] = (
        target_table["BH_FDR_Adjusted_P"] < 0.05
    )

    target_table["Bonferroni_Significant_0_05"] = (
        target_table["Bonferroni_Adjusted_P"] < 0.05
    )

    SPORT_MULTIPLE_TESTING_TABLES.append(target_table)

SPORT_RESIDUALS_CORRECTED_DF = pd.concat(
    SPORT_MULTIPLE_TESTING_TABLES,
    ignore_index=True
)

SPORT_MULTIPLE_TESTING_SUMMARY_DF = (
    SPORT_RESIDUALS_CORRECTED_DF
    .groupby("Target", as_index=False)
    .agg(
        Comparisons=("Comparisons_in_Outcome_Family", "max"),
        Unadjusted_Z_1_96_n=(
            "Absolute_Z_At_Least_1_96",
            "sum"
        ),
        Unadjusted_Z_2_58_n=(
            "Absolute_Z_At_Least_2_58",
            "sum"
        ),
        BH_FDR_Significant_n=(
            "BH_FDR_Significant_0_05",
            "sum"
        ),
        Bonferroni_Significant_n=(
            "Bonferroni_Significant_0_05",
            "sum"
        )
    )
)

display(SPORT_MULTIPLE_TESTING_SUMMARY_DF)

##9.11.Top Sport Residual Tables

Identify the ten sports disciplines with the highest positive adjusted standardized residuals for each medal outcome and append the prespecified significance symbols.

In [ ]:
# ================================================================
# 9.11 Top Sport Residual Tables
# ================================================================

def residual_significance_symbol(z_value: float) -> str:
    """Return the prespecified unadjusted residual symbol."""
    if not np.isfinite(z_value):
        return ""

    if abs(z_value) >= 2.58:
        return "**"

    if abs(z_value) >= 1.96:
        return "*"

    return ""


SPORT_TOP10_RESIDUAL_TABLES = {}
SPORT_TOP10_RESIDUAL_ROWS = []

for target in TARGETS:
    target_top10 = (
        SPORT_RESIDUALS_CORRECTED_DF.loc[
            SPORT_RESIDUALS_CORRECTED_DF["Target"] == target
        ]
        .sort_values(
            "Adjusted_Standardized_Residual",
            ascending=False
        )
        .head(10)
        .copy()
    )

    target_top10["Residual_Symbol"] = target_top10[
        "Adjusted_Standardized_Residual"
    ].apply(residual_significance_symbol)

    target_top10["Sport_Label_for_Figure"] = (
        target_top10["Sport_Discipline"].astype(str)
        + target_top10["Residual_Symbol"].apply(
            lambda symbol: f" {symbol}" if symbol else ""
        )
    )

    target_top10.insert(
        1,
        "Positive_Residual_Rank",
        np.arange(1, len(target_top10) + 1)
    )

    SPORT_TOP10_RESIDUAL_TABLES[target] = target_top10
    SPORT_TOP10_RESIDUAL_ROWS.append(target_top10)

SPORT_TOP10_RESIDUALS_DF = pd.concat(
    SPORT_TOP10_RESIDUAL_ROWS,
    ignore_index=True
)

display(
    SPORT_TOP10_RESIDUALS_DF[
        [
            "Target",
            "Positive_Residual_Rank",
            "Sport_Discipline",
            "Adjusted_Standardized_Residual",
            "Residual_Symbol",
            "BH_FDR_Adjusted_P",
            "Bonferroni_Adjusted_P"
        ]
    ].round(4)
)

##9.12.Sport Residual Figures

Generate separate horizontal bar plots for the ten highest positive adjusted standardized residuals in each medal outcome.

In [ ]:
# ================================================================
# 9.12 Sport Residual Figures
# ================================================================

SPORT_RESIDUAL_FIGURE_PATHS = {}

for target in TARGETS:
    plot_data = (
        SPORT_TOP10_RESIDUAL_TABLES[target]
        .sort_values(
            "Adjusted_Standardized_Residual",
            ascending=True
        )
        .copy()
    )

    plt.figure(figsize=(9, 6))

    plt.barh(
        plot_data["Sport_Label_for_Figure"],
        plot_data["Adjusted_Standardized_Residual"]
    )

    plt.axvline(
        1.96,
        linestyle="--",
        linewidth=1,
        label="|Z| = 1.96"
    )

    plt.axvline(
        2.58,
        linestyle=":",
        linewidth=1,
        label="|Z| = 2.58"
    )

    plt.xlabel("Adjusted standardized residual")
    plt.ylabel("Sports discipline")
    plt.title(target)
    plt.legend()
    plt.tight_layout()

    figure_path = (
        OUTPUT_FOLDERS["figures"]
        / f"{target}_Top10_Adjusted_Standardized_Residuals.png"
    )

    plt.savefig(
        figure_path,
        dpi=300,
        bbox_inches="tight"
    )
    plt.close()

    SPORT_RESIDUAL_FIGURE_PATHS[target] = figure_path

print("Sport residual figures were generated.")

##9.13.Supplementary Analysis Export

Export threshold-independent model performance, TMPS comparisons, omnibus chi-square tests, adjusted standardized residuals, multiple-testing corrections, and top-sport summaries.

In [ ]:
# ================================================================
# 9.13 Supplementary Analysis Export
# ================================================================

THRESHOLD_INDEPENDENT_PERFORMANCE_DF.to_csv(
    OUTPUT_FOLDERS["supplementary"]
    / "Threshold_Independent_Pooled_OOF_Performance.csv",
    index=False
)

FOLD_LEVEL_PROBABILITY_PERFORMANCE_DF.to_csv(
    OUTPUT_FOLDERS["supplementary"]
    / "Outer_Fold_PR_AUC_Brier_Performance.csv",
    index=False
)

TMPS_ANALYSIS_SUMMARY_DF.to_csv(
    OUTPUT_FOLDERS["supplementary"]
    / "TMPS_Analysis_Sample_Summary.csv",
    index=False
)

TMPS_CONTINUOUS_BINARY_SUMMARY_DF.to_csv(
    OUTPUT_FOLDERS["supplementary"]
    / "TMPS_Continuous_vs_Binary_Performance.csv",
    index=False
)

TMPS_REPRESENTATION_DIFFERENCE_DF.to_csv(
    OUTPUT_FOLDERS["supplementary"]
    / "TMPS_Continuous_vs_Binary_Differences.csv",
    index=False
)

SPORT_DISCIPLINE_LABEL_SUMMARY_DF.to_csv(
    OUTPUT_FOLDERS["supplementary"]
    / "Sport_Discipline_Label_Mapping.csv",
    index=False
)

SPORT_CHI_SQUARE_OMNIBUS_DF.to_csv(
    OUTPUT_FOLDERS["supplementary"]
    / "Sport_ChiSquare_Omnibus_Tests.csv",
    index=False
)

SPORT_RESIDUALS_CORRECTED_DF.to_csv(
    OUTPUT_FOLDERS["supplementary"]
    / "Sport_Adjusted_Residuals_FDR_Bonferroni.csv",
    index=False
)

SPORT_MULTIPLE_TESTING_SUMMARY_DF.to_csv(
    OUTPUT_FOLDERS["supplementary"]
    / "Sport_Multiple_Testing_Summary.csv",
    index=False
)

SPORT_TOP10_RESIDUALS_DF.to_csv(
    OUTPUT_FOLDERS["supplementary"]
    / "Sport_Top10_Positive_Adjusted_Residuals.csv",
    index=False
)

for target in TARGETS:
    for representation in [
        "TMPS_Continuous",
        "TMPS_14_or_Less"
    ]:
        TMPS_CV_FOLD_RESULTS[
            (target, representation)
        ].to_csv(
            OUTPUT_FOLDERS["supplementary"]
            / f"{target}_{representation}_Fold_Performance.csv",
            index=False
        )

        TMPS_CV_OOF_PREDICTIONS[
            (target, representation)
        ].to_csv(
            OUTPUT_FOLDERS["supplementary"]
            / f"{target}_{representation}_OOF_Predictions.csv",
            index=False
        )

    SPORT_CONTINGENCY_TABLES[target].to_csv(
        OUTPUT_FOLDERS["supplementary"]
        / f"{target}_Sport_Contingency_Table.csv"
    )

    SPORT_EXPECTED_TABLES[target].to_csv(
        OUTPUT_FOLDERS["supplementary"]
        / f"{target}_Sport_Expected_Counts.csv"
    )

    SPORT_TOP10_RESIDUAL_TABLES[target].to_csv(
        OUTPUT_FOLDERS["supplementary"]
        / f"{target}_Top10_Adjusted_Residuals.csv",
        index=False
    )


SUPPLEMENTARY_EXCEL_PATH = (
    OUTPUT_FOLDERS["tables"]
    / "Additional_Supplementary_Analyses.xlsx"
)

with pd.ExcelWriter(
    SUPPLEMENTARY_EXCEL_PATH,
    engine="xlsxwriter"
) as writer:
    THRESHOLD_INDEPENDENT_PERFORMANCE_DF.to_excel(
        writer,
        sheet_name="Pooled_OOF_Performance",
        index=False
    )

    FOLD_LEVEL_PROBABILITY_PERFORMANCE_DF.to_excel(
        writer,
        sheet_name="Fold_PR_AUC_Brier",
        index=False
    )

    TMPS_ANALYSIS_SUMMARY_DF.to_excel(
        writer,
        sheet_name="TMPS_Sample",
        index=False
    )

    TMPS_CONTINUOUS_BINARY_SUMMARY_DF.to_excel(
        writer,
        sheet_name="TMPS_Performance",
        index=False
    )

    TMPS_REPRESENTATION_DIFFERENCE_DF.to_excel(
        writer,
        sheet_name="TMPS_Differences",
        index=False
    )

    SPORT_CHI_SQUARE_OMNIBUS_DF.to_excel(
        writer,
        sheet_name="Sport_Omnibus",
        index=False
    )

    SPORT_MULTIPLE_TESTING_SUMMARY_DF.to_excel(
        writer,
        sheet_name="Multiple_Testing",
        index=False
    )

    SPORT_TOP10_RESIDUALS_DF.to_excel(
        writer,
        sheet_name="Top10_Residuals",
        index=False
    )

    for target in TARGETS:
        target_residuals = SPORT_RESIDUALS_CORRECTED_DF.loc[
            SPORT_RESIDUALS_CORRECTED_DF["Target"] == target
        ]

        target_residuals.to_excel(
            writer,
            sheet_name=f"{target}_Residuals",
            index=False
        )

        SPORT_CONTINGENCY_TABLES[target].to_excel(
            writer,
            sheet_name=f"{target}_Observed"
        )

        SPORT_EXPECTED_TABLES[target].to_excel(
            writer,
            sheet_name=f"{target}_Expected"
        )

display(FOLD_LEVEL_PROBABILITY_PERFORMANCE_DF.round(4))
display(TMPS_CONTINUOUS_BINARY_SUMMARY_DF.round(4))
display(SPORT_MULTIPLE_TESTING_SUMMARY_DF)

print(
    f"Supplementary results saved to: "
    f"{OUTPUT_FOLDERS['supplementary']}"
)
print(
    f"Combined workbook saved to: "
    f"{SUPPLEMENTARY_EXCEL_PATH}"
)

#10.Consolidated Outputs and Reproducibility Checks

Consolidate the principal analysis outputs, document the computational environment, verify manuscript-related values, generate repository metadata, and prepare a privacy-safe release package for GitHub and Zenodo.

##10.1.Public Release Configuration

Define the repository metadata, manuscript information, and privacy rules used to prepare the public code release. Athlete-level data and individual-level model outputs are excluded from the release package.

In [ ]:
# ================================================================
# 10.1 Public Release Configuration
# ================================================================

import hashlib
import importlib.metadata
import platform
import shutil
import subprocess
import sys
import zipfile

from datetime import date, datetime

REPOSITORY_NAME = "olympic-medal-outcome-ml"
SOFTWARE_TITLE = (
    "Code for Predicting Olympic Medal Outcomes in 1,011 Elite "
    "Athletes Using Multimodal Machine Learning"
)
MANUSCRIPT_TITLE = (
    "Predicting Olympic medal outcomes in 1,011 elite athletes "
    "using multimodal machine learning"
)

RELEASE_VERSION = "1.0.0"
RELEASE_DATE = date.today().isoformat()

AUTHOR_GIVEN_NAME = "Hyoungjoo"
AUTHOR_FAMILY_NAME = "Choi"

GITHUB_REPOSITORY_URL = "ADD_GITHUB_REPOSITORY_URL_AFTER_CREATION"
ZENODO_DOI = "ADD_ZENODO_DOI_AFTER_ARCHIVING"

PUBLIC_RELEASE_DIR = BASE_DIR / "Olympic_Medal_ML_Public_Release"
PUBLIC_METADATA_DIR = PUBLIC_RELEASE_DIR / "metadata"
PUBLIC_DOCUMENTATION_DIR = PUBLIC_RELEASE_DIR / "documentation"
PUBLIC_AGGREGATE_RESULTS_DIR = PUBLIC_RELEASE_DIR / "aggregate_results"

for folder in [
    PUBLIC_RELEASE_DIR,
    PUBLIC_METADATA_DIR,
    PUBLIC_DOCUMENTATION_DIR,
    PUBLIC_AGGREGATE_RESULTS_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

# Files containing athlete-level or potentially identifiable outputs
# must not be copied into the public repository.
PRIVATE_FILE_PATTERNS = [
    "International_Sports_Data",
    "Athlete_Level",
    "OOF_Predictions",
    "Paired_OOF_Predictions",
    "FFS_Peak_Input",
    "Saved_Models",
    ".pkl",
    ".pickle"
]

PUBLIC_RELEASE_CONFIGURATION = {
    "repository_name": REPOSITORY_NAME,
    "software_title": SOFTWARE_TITLE,
    "manuscript_title": MANUSCRIPT_TITLE,
    "release_version": RELEASE_VERSION,
    "release_date": RELEASE_DATE,
    "github_repository_url": GITHUB_REPOSITORY_URL,
    "zenodo_doi": ZENODO_DOI,
    "athlete_level_dataset_included": False,
    "athlete_level_predictions_included": False,
    "serialized_fitted_models_included": False,
    "privacy_rule": (
        "Only code, documentation, metadata, and aggregate results "
        "may be included in the public release."
    )
}

display(pd.DataFrame(
    PUBLIC_RELEASE_CONFIGURATION.items(),
    columns=["Setting", "Value"]
))

##10.2.Consolidated Aggregate Results

Consolidate the principal aggregate results generated across the descriptive, feature-selection, nested cross-validation, complete-case, no-SD, and supplementary analyses.

In [ ]:
# ================================================================
# 10.2 Consolidated Aggregate Results
# ================================================================

CONSOLIDATED_AGGREGATE_TABLES = {
    "Medal_Outcomes": MEDAL_DISTRIBUTION_DF,
    "MPS_Missingness": MPS_MISSINGNESS_DF,
    "MPS_Completeness": MPS_COMPLETENESS_DF,
    "TMPS_Summary": TMPS_SUMMARY_DF,
    "VIF_Removal": VIF_REMOVAL_LOG_DF,
    "Final_VIF": FINAL_VIF_DF,
    "FFS_Peak_Summary": FFS_PEAK_SUMMARY_DF,
    "Global_SHAP_Summary": GLOBAL_SHAP_SUMMARY_DF,
    "Main_Nested_CV": MAIN_NESTED_CV_SUMMARY_DF,
    "Threshold_Sensitivity": THRESHOLD_SENSITIVITY_DF,
    "Complete_Case": COMPLETE_CASE_SUMMARY_DF,
    "Primary_vs_CompleteCase": PRIMARY_COMPLETE_CASE_COMPARISON_DF,
    "NoSD_FFS_Peaks": NO_SD_FFS_PEAK_SUMMARY_DF,
    "Full_vs_NoSD": FULL_NO_SD_DIFFERENCE_DF,
    "NoSD_Bootstrap": FULL_VS_NO_SD_BOOTSTRAP_DF,
    "PR_AUC_and_Brier": FOLD_LEVEL_PROBABILITY_PERFORMANCE_DF,
    "TMPS_Continuous_vs_Binary": TMPS_CONTINUOUS_BINARY_SUMMARY_DF,
    "Sport_Omnibus_Tests": SPORT_CHI_SQUARE_OMNIBUS_DF,
    "Sport_Multiple_Testing": SPORT_MULTIPLE_TESTING_SUMMARY_DF
}

CONSOLIDATED_RESULTS_PATH = (
    OUTPUT_FOLDERS["tables"] / "Consolidated_Aggregate_Results.xlsx"
)

with pd.ExcelWriter(CONSOLIDATED_RESULTS_PATH, engine="xlsxwriter") as writer:
    for sheet_name, table in CONSOLIDATED_AGGREGATE_TABLES.items():
        table.to_excel(writer, sheet_name=sheet_name[:31], index=False)

for table_name, table in CONSOLIDATED_AGGREGATE_TABLES.items():
    table.to_csv(
        PUBLIC_AGGREGATE_RESULTS_DIR / f"{table_name}.csv",
        index=False
    )

CONSOLIDATED_TABLE_SUMMARY_DF = pd.DataFrame([
    {
        "Table_Name": name,
        "Rows": len(table),
        "Columns": table.shape[1],
        "Public_Aggregate_Output": True
    }
    for name, table in CONSOLIDATED_AGGREGATE_TABLES.items()
])

display(CONSOLIDATED_TABLE_SUMMARY_DF)
print(f"Consolidated workbook saved to: {CONSOLIDATED_RESULTS_PATH}")

##10.3.Required Object Validation

Verify that all principal analysis objects were created before preparing the repository release.

In [ ]:
# ================================================================
# 10.3 Required Object Validation
# ================================================================

REQUIRED_ANALYSIS_OBJECTS = [
    "df",
    "MPS_MISSINGNESS_DF",
    "MPS_COMPLETENESS_DF",
    "TMPS_SUMMARY_DF",
    "FINAL_VIF_DF",
    "POST_VIF_FEATURES",
    "FFS_PEAK_FEATURES",
    "FFS_PEAK_SUMMARY_DF",
    "GLOBAL_SHAP_SUMMARY_DF",
    "MAIN_NESTED_CV_RESULTS",
    "MAIN_NESTED_CV_SUMMARY_DF",
    "COMPLETE_CASE_RESULTS",
    "COMPLETE_CASE_SUMMARY_DF",
    "NO_SD_FFS_PEAK_FEATURES",
    "FULL_VS_NO_SD_BOOTSTRAP_DF",
    "TMPS_CONTINUOUS_BINARY_SUMMARY_DF",
    "SPORT_RESIDUALS_CORRECTED_DF"
]

OBJECT_VALIDATION_DF = pd.DataFrame([
    {
        "Object": object_name,
        "Available": object_name in globals(),
        "Object_Type": (
            type(globals()[object_name]).__name__
            if object_name in globals() else "Missing"
        )
    }
    for object_name in REQUIRED_ANALYSIS_OBJECTS
])

display(OBJECT_VALIDATION_DF)

missing_objects = OBJECT_VALIDATION_DF.loc[
    ~OBJECT_VALIDATION_DF["Available"],
    "Object"
].tolist()

if missing_objects:
    raise RuntimeError(
        "Required analysis objects are missing: "
        + ", ".join(missing_objects)
    )

##10.4.Manuscript-Related Value Checks

Compare key cohort and missing-data values generated by the notebook with the values reported in the current manuscript and Supplementary Materials.

In [ ]:
# ================================================================
# 10.4 Manuscript-Related Value Checks
# ================================================================

MANUSCRIPT_EXPECTED_VALUES = {
    "Full_Cohort_n": 1011,
    "MPS_Complete_n": 815,
    "MPS_Incomplete_n": 196,
    "OMA_Positive_n": 470
}

observed_oma_positive = int(
    (pd.to_numeric(df["OMA"], errors="coerce") == 1).sum()
)

MANUSCRIPT_OBSERVED_VALUES = {
    "Full_Cohort_n": len(df),
    "MPS_Complete_n": int(MPS_COMPLETE_MASK.sum()),
    "MPS_Incomplete_n": int((~MPS_COMPLETE_MASK).sum()),
    "OMA_Positive_n": observed_oma_positive
}

MANUSCRIPT_VALUE_CHECK_DF = pd.DataFrame([
    {
        "Check": check_name,
        "Expected_Manuscript_Value": expected_value,
        "Observed_Code_Value": MANUSCRIPT_OBSERVED_VALUES[check_name],
        "Difference": (
            MANUSCRIPT_OBSERVED_VALUES[check_name] - expected_value
        ),
        "Passed": (
            MANUSCRIPT_OBSERVED_VALUES[check_name] == expected_value
        )
    }
    for check_name, expected_value in MANUSCRIPT_EXPECTED_VALUES.items()
])

display(MANUSCRIPT_VALUE_CHECK_DF)

MANUSCRIPT_VALUE_CHECK_DF.to_csv(
    OUTPUT_FOLDERS["configuration"]
    / "Manuscript_Related_Value_Checks.csv",
    index=False
)

if not MANUSCRIPT_VALUE_CHECK_DF["Passed"].all():
    print(
        "Warning: At least one generated value differs from the "
        "current manuscript. Review the source data, preprocessing, "
        "and manuscript tables before public release."
    )

##10.5.Analysis Structure Audit

Document the implemented workflow and verify that forward feature selection, SHAP interpretation, and primary nested cross-validation remain methodologically distinct.

In [ ]:
# ================================================================
# 10.5 Analysis Structure Audit
# ================================================================

ANALYSIS_STRUCTURE_AUDIT_DF = pd.DataFrame([
    {
        "Analysis_Component": "VIF screening",
        "Input_Features": len(X_STANDARDIZED.columns),
        "Output_Features": len(POST_VIF_FEATURES),
        "Role": "Multicollinearity screening before FFS"
    },
    {
        "Analysis_Component": "Forward feature selection",
        "Input_Features": len(POST_VIF_FEATURES),
        "Output_Features": "Algorithm- and target-specific peak subsets",
        "Role": "Independent feature-selection analysis"
    },
    {
        "Analysis_Component": "SHAP",
        "Input_Features": "FFS peak subset",
        "Output_Features": "Global and sport-specific SHAP rankings",
        "Role": "Interpretation after full-data model refitting"
    },
    {
        "Analysis_Component": "Primary nested CV",
        "Input_Features": len(NESTED_CV_FEATURES),
        "Output_Features": "Outer-fold performance estimates",
        "Role": "Independent post-VIF model evaluation"
    },
    {
        "Analysis_Component": "Complete-case sensitivity",
        "Input_Features": "Selected-model FFS peak subsets",
        "Output_Features": "Complete-case nested-CV estimates",
        "Role": "Missing-data sensitivity analysis"
    },
    {
        "Analysis_Component": "No-SD sensitivity",
        "Input_Features": "Post-VIF features excluding SD",
        "Output_Features": "No-SD FFS peaks and paired nested CV",
        "Role": "Sports-discipline exclusion analysis"
    }
])

display(ANALYSIS_STRUCTURE_AUDIT_DF)

ANALYSIS_STRUCTURE_AUDIT_DF.to_csv(
    PUBLIC_METADATA_DIR / "analysis_structure_audit.csv",
    index=False
)

##10.6.Computational Environment and Package Versions

Record the Python runtime, operating system, and package versions used for the final execution.

In [ ]:
# ================================================================
# 10.6 Computational Environment and Package Versions
# ================================================================

REQUIRED_PACKAGES = [
    "numpy",
    "pandas",
    "scipy",
    "scikit-learn",
    "imbalanced-learn",
    "statsmodels",
    "matplotlib",
    "shap",
    "catboost",
    "xgboost",
    "lightgbm",
    "openpyxl",
    "xlsxwriter"
]


def get_installed_version(package_name: str) -> str:
    """Return the installed package version or a missing-package flag."""
    try:
        return importlib.metadata.version(package_name)
    except importlib.metadata.PackageNotFoundError:
        return "Not installed"


PACKAGE_VERSION_DF = pd.DataFrame([
    {
        "Package": package_name,
        "Version": get_installed_version(package_name)
    }
    for package_name in REQUIRED_PACKAGES
])

RUNTIME_INFORMATION = {
    "Execution_Timestamp": datetime.now().isoformat(),
    "Python_Version": sys.version.replace("\n", " "),
    "Python_Executable": sys.executable,
    "Operating_System": platform.platform(),
    "Processor": platform.processor(),
    "Machine": platform.machine(),
    "Random_Seed": SEED
}

RUNTIME_INFORMATION_DF = pd.DataFrame(
    RUNTIME_INFORMATION.items(),
    columns=["Setting", "Value"]
)

display(RUNTIME_INFORMATION_DF)
display(PACKAGE_VERSION_DF)

RUNTIME_INFORMATION_DF.to_csv(
    PUBLIC_METADATA_DIR / "runtime_information.csv",
    index=False
)
PACKAGE_VERSION_DF.to_csv(
    PUBLIC_METADATA_DIR / "package_versions.csv",
    index=False
)

##10.7.Requirements File Generation

Generate a requirements file using the package versions installed during the validated final execution.

In [ ]:
# ================================================================
# 10.7 Requirements File Generation
# ================================================================

requirements_lines = [
    f"{row.Package}=={row.Version}"
    for row in PACKAGE_VERSION_DF.itertuples(index=False)
    if row.Version != "Not installed"
]

REQUIREMENTS_PATH = PUBLIC_RELEASE_DIR / "requirements.txt"

REQUIREMENTS_PATH.write_text(
    "\n".join(requirements_lines) + "\n",
    encoding="utf-8"
)

print(REQUIREMENTS_PATH.read_text(encoding="utf-8"))

##10.8.Data Dictionary Generation

Generate a code-level data dictionary describing the outcome variables, predictor groups, expected variable names, and public-data availability status.

In [ ]:
# ================================================================
# 10.8 Data Dictionary Generation
# ================================================================

FEATURE_GROUPS = {
    "Demographic": DEMOGRAPHIC_FEATURES,
    "Sports discipline": SPORT_FEATURES,
    "Physiotherapy counts": PHYSIOTHERAPY_COUNT_FEATURES,
    "Treatment purposes": TREATMENT_PURPOSE_FEATURES,
    "Treatment causes": TREATMENT_CAUSE_FEATURES,
    "Treated body areas": TREATED_AREA_FEATURES,
    "Taped body areas": TAPED_AREA_FEATURES,
    "Movement Performance Scores": MPS_FEATURES
}

DATA_DICTIONARY_ROWS = []

for feature_group, feature_list in FEATURE_GROUPS.items():
    for feature in feature_list:
        DATA_DICTIONARY_ROWS.append({
            "Variable": feature,
            "Variable_Role": "Predictor",
            "Variable_Group": feature_group,
            "Available_in_Source_Data": feature in df.columns,
            "Observed_Dtype": (
                str(df[feature].dtype) if feature in df.columns else "Not available"
            ),
            "Public_Athlete_Level_Data": False
        })

for target in TARGETS:
    DATA_DICTIONARY_ROWS.append({
        "Variable": target,
        "Variable_Role": "Binary outcome",
        "Variable_Group": "Medal achievement",
        "Available_in_Source_Data": target in df.columns,
        "Observed_Dtype": (
            str(df[target].dtype) if target in df.columns else "Not available"
        ),
        "Public_Athlete_Level_Data": False
    })

DATA_DICTIONARY_DF = (
    pd.DataFrame(DATA_DICTIONARY_ROWS)
    .drop_duplicates(subset=["Variable", "Variable_Role"])
    .reset_index(drop=True)
)

DATA_DICTIONARY_PATH = (
    PUBLIC_DOCUMENTATION_DIR / "data_dictionary.csv"
)

DATA_DICTIONARY_DF.to_csv(DATA_DICTIONARY_PATH, index=False)

display(DATA_DICTIONARY_DF)

##10.9.Aggregate Output Privacy Audit

Screen the proposed public-release files for filenames associated with athlete-level data, predictions, fitted models, or the restricted source dataset.

In [ ]:
# ================================================================
# 10.9 Aggregate Output Privacy Audit
# ================================================================

def contains_private_pattern(file_path: Path) -> bool:
    """Return True when a filename matches a restricted-output pattern."""
    filename = file_path.name.lower()

    return any(
        private_pattern.lower() in filename
        for private_pattern in PRIVATE_FILE_PATTERNS
    )


PUBLIC_RELEASE_FILE_CANDIDATES = [
    path for path in PUBLIC_RELEASE_DIR.rglob("*")
    if path.is_file()
]

PRIVACY_AUDIT_DF = pd.DataFrame([
    {
        "Relative_Path": str(path.relative_to(PUBLIC_RELEASE_DIR)),
        "Private_Pattern_Detected": contains_private_pattern(path),
        "Eligible_for_Public_Release": not contains_private_pattern(path)
    }
    for path in PUBLIC_RELEASE_FILE_CANDIDATES
])

display(PRIVACY_AUDIT_DF)

privacy_violations = PRIVACY_AUDIT_DF.loc[
    PRIVACY_AUDIT_DF["Private_Pattern_Detected"],
    "Relative_Path"
].tolist()

if privacy_violations:
    raise RuntimeError(
        "Potentially private files were detected in the public release folder: "
        + ", ".join(privacy_violations)
    )

##10.10.Repository README Generation

Generate a README describing the study workflow, execution requirements, privacy restrictions, output structure, and citation instructions.

In [ ]:
# ================================================================
# 10.10 Repository README Generation
# ================================================================

README_TEXT = f"""# {SOFTWARE_TITLE}

## Associated manuscript

**{MANUSCRIPT_TITLE}**

## Overview

This repository contains the public reproducibility code for the associated
machine-learning study of medal outcomes in 1,011 elite athletes.

The implemented workflow includes:

1. Descriptive and missing-data analyses
2. Iterative variance inflation factor screening
3. Independent forward feature selection
4. FFS-peak global and sport-specific SHAP analyses
5. Independent five-outer-fold and three-inner-fold nested cross-validation
6. Complete-case sensitivity analyses
7. Sports-discipline-excluded sensitivity analyses
8. TMPS and sport-specific supplementary analyses

## Methodological structure

Forward feature selection and the primary nested cross-validation are separate
analyses.

- FFS peak subsets are used for feature-selection summaries and SHAP.
- Primary nested cross-validation independently evaluates the post-VIF
  candidate feature set.
- The sports-discipline-excluded sensitivity repeats FFS without SD and
  evaluates the resulting no-SD peak subsets.

## Data availability

The athlete-level source dataset is not included because it contains
institutionally restricted athlete health, physiotherapy-service, and
performance data.

The repository does not contain:

- athlete-level source data;
- athlete-level out-of-fold predictions;
- athlete-level SHAP values;
- serialized fitted models; or
- identifying information.

Researchers with authorized access to a dataset containing the required
variables can execute the notebook using the accompanying data dictionary.

## Reproducibility status

The notebook must be fully executed using the original authorized dataset
before version {RELEASE_VERSION} is released. Generated results must be
compared with the manuscript and Supplementary Materials before archiving.

## Execution

1. Open `Olympic_Medal_ML_Public_Reproducibility_Code.ipynb` in Google Colab.
2. Mount Google Drive.
3. Update `BASE_DIR` and `DATA_PATH`.
4. Install packages from `requirements.txt`.
5. Run all cells from top to bottom.
6. Review all reproducibility and privacy checks in Section 10.

## Repository structure

```text
{REPOSITORY_NAME}/
├── Olympic_Medal_ML_Public_Reproducibility_Code.ipynb
├── README.md
├── requirements.txt
├── CITATION.cff
├── LICENSE
├── documentation/
│   ├── data_dictionary.csv
│   ├── CODE_AVAILABILITY.txt
│   └── DATA_AVAILABILITY.txt
├── metadata/
│   ├── runtime_information.csv
│   ├── package_versions.csv
│   ├── analysis_structure_audit.csv
│   └── file_manifest_sha256.csv
└── aggregate_results/

## 10.11 Citation and Availability Statements — Markdown 셀

```markdown
##10.11.Citation and Availability Statements

Generate CITATION.cff metadata and draft Code Availability and Data Availability statements for the manuscript and repository.

In [ ]:
# ================================================================
# 10.11 Citation and Availability Statements
# ================================================================

CITATION_TEXT = f"""cff-version: 1.2.0
message: "Please cite this software and the associated research article."
title: "{SOFTWARE_TITLE}"
type: software
authors:
  - family-names: "{AUTHOR_FAMILY_NAME}"
    given-names: "{AUTHOR_GIVEN_NAME}"
version: "{RELEASE_VERSION}"
date-released: "{RELEASE_DATE}"
repository-code: "{GITHUB_REPOSITORY_URL}"
doi: "{ZENODO_DOI}"
"""

CODE_AVAILABILITY_TEXT = f"""Code Availability

The analysis code used in this study is publicly available through GitHub
({GITHUB_REPOSITORY_URL}) and has been archived in Zenodo
({ZENODO_DOI}).
"""

DATA_AVAILABILITY_TEXT = """Data Availability

The athlete-level datasets generated and/or analyzed during the current study
are not publicly available because of institutional policies concerning
athlete health data but are available from the corresponding author upon
reasonable request and subject to institutional approval.
"""

CITATION_PATH = PUBLIC_RELEASE_DIR / "CITATION.cff"
CODE_AVAILABILITY_PATH = (
    PUBLIC_DOCUMENTATION_DIR / "CODE_AVAILABILITY.txt"
)
DATA_AVAILABILITY_PATH = (
    PUBLIC_DOCUMENTATION_DIR / "DATA_AVAILABILITY.txt"
)

CITATION_PATH.write_text(CITATION_TEXT, encoding="utf-8")
CODE_AVAILABILITY_PATH.write_text(
    CODE_AVAILABILITY_TEXT,
    encoding="utf-8"
)
DATA_AVAILABILITY_PATH.write_text(
    DATA_AVAILABILITY_TEXT,
    encoding="utf-8"
)

print(f"CITATION.cff generated: {CITATION_PATH}")
print(f"Code Availability draft: {CODE_AVAILABILITY_PATH}")
print(f"Data Availability draft: {DATA_AVAILABILITY_PATH}")

##10.12.Software License

Create a license-selection notice. A software license should be selected and added before the first GitHub release.

In [ ]:
# ================================================================
# 10.12 Software License
# ================================================================

LICENSE_NOTICE_TEXT = """Software License Required

Before creating the first public GitHub release, select an appropriate
open-source software license and save the full license text as a file named
LICENSE in the repository root.

A commonly used permissive option for academic software is the MIT License.
The final choice should reflect the authors' and institution's requirements.

Do not release the repository until the LICENSE file has been added and
reviewed.
"""

LICENSE_NOTICE_PATH = (
    PUBLIC_DOCUMENTATION_DIR / "LICENSE_SELECTION_REQUIRED.txt"
)

LICENSE_NOTICE_PATH.write_text(
    LICENSE_NOTICE_TEXT,
    encoding="utf-8"
)

print(f"License notice generated: {LICENSE_NOTICE_PATH}")

##10.13.GitHub and Zenodo Release Checklist

Generate a release checklist covering execution, numerical validation, privacy review, repository creation, GitHub release tagging, Zenodo archiving, and manuscript updating.

In [ ]:
# ================================================================
# 10.13 GitHub and Zenodo Release Checklist
# ================================================================

RELEASE_CHECKLIST_TEXT = f"""# Public Release Checklist

## Code completion

- [ ] All notebook sections are present.
- [ ] All cells run from top to bottom without errors.
- [ ] No undefined variables remain.
- [ ] The final Colab notebook is saved as:
      `Olympic_Medal_ML_Public_Reproducibility_Code.ipynb`.

## Numerical validation

- [ ] Cohort counts match the manuscript.
- [ ] MPS missingness matches the Supplementary Materials.
- [ ] VIF-excluded variables match the reported table.
- [ ] FFS peak feature counts and feature lists match the manuscript.
- [ ] Table 2 nested-CV performance values match.
- [ ] PR-AUC and Brier scores match.
- [ ] Calibration intercepts and slopes match.
- [ ] Complete-case results match.
- [ ] No-SD results and paired bootstrap estimates match.
- [ ] TMPS continuous-versus-binary results match.
- [ ] Sport residual, BH-FDR, and Bonferroni results match.
- [ ] Global and sport-specific SHAP rankings match.

## Privacy review

- [ ] The original athlete-level dataset is excluded.
- [ ] Athlete-level SHAP files are excluded.
- [ ] Athlete-level out-of-fold predictions are excluded.
- [ ] Serialized fitted models are excluded unless institutionally approved.
- [ ] Google Drive paths containing personal information are removed.
- [ ] All aggregate tables have been manually inspected.
- [ ] No small-cell or identifying information is inadvertently disclosed.

## Repository preparation

- [ ] README.md is complete.
- [ ] requirements.txt reflects the validated environment.
- [ ] data_dictionary.csv is complete.
- [ ] CITATION.cff is complete.
- [ ] A LICENSE file has been selected and added.
- [ ] GitHub repository URL replaces its placeholder.
- [ ] The final notebook has been added to the repository root.

## GitHub and Zenodo

- [ ] Save the Colab notebook to a standard GitHub repository, not a Gist.
- [ ] Review the repository before making it public.
- [ ] Create GitHub release `v{RELEASE_VERSION}`.
- [ ] Enable the repository in Zenodo.
- [ ] Archive GitHub release `v{RELEASE_VERSION}` in Zenodo.
- [ ] Record the concept DOI and version DOI.
- [ ] Replace the DOI placeholder in README.md and CITATION.cff.
- [ ] Update the manuscript Code Availability section.
- [ ] Confirm that the archived Zenodo record opens correctly.
"""

RELEASE_CHECKLIST_PATH = (
    PUBLIC_DOCUMENTATION_DIR / "RELEASE_CHECKLIST.md"
)

RELEASE_CHECKLIST_PATH.write_text(
    RELEASE_CHECKLIST_TEXT,
    encoding="utf-8"
)

print(f"Release checklist generated: {RELEASE_CHECKLIST_PATH}")

##10.14.File Manifest and SHA-256 Checksums

Create a manifest containing the relative path, file size, modification time, and SHA-256 checksum of each public-release file.

In [ ]:
# ================================================================
# 10.14 File Manifest and SHA-256 Checksums
# ================================================================

def calculate_sha256(file_path: Path) -> str:
    """Calculate the SHA-256 checksum of a file."""
    sha256 = hashlib.sha256()

    with open(file_path, "rb") as file:
        for block in iter(lambda: file.read(1024 * 1024), b""):
            sha256.update(block)

    return sha256.hexdigest()


def build_file_manifest(root_directory: Path) -> pd.DataFrame:
    """Create a manifest for all files contained in a directory."""
    rows = []

    for file_path in sorted(root_directory.rglob("*")):
        if not file_path.is_file():
            continue

        rows.append({
            "Relative_Path": str(file_path.relative_to(root_directory)),
            "File_Size_Bytes": file_path.stat().st_size,
            "Modified_Time": datetime.fromtimestamp(
                file_path.stat().st_mtime
            ).isoformat(),
            "SHA256": calculate_sha256(file_path)
        })

    return pd.DataFrame(rows)


PUBLIC_FILE_MANIFEST_DF = build_file_manifest(PUBLIC_RELEASE_DIR)

MANIFEST_PATH = (
    PUBLIC_METADATA_DIR / "file_manifest_sha256.csv"
)

PUBLIC_FILE_MANIFEST_DF.to_csv(
    MANIFEST_PATH,
    index=False
)

display(PUBLIC_FILE_MANIFEST_DF)

##10.15.Final Privacy and Placeholder Audit

Verify that no restricted filenames are present and identify metadata placeholders that must be replaced before release.

In [ ]:
# ================================================================
# 10.15 Final Privacy and Placeholder Audit
# ================================================================

PUBLIC_RELEASE_FILES = [
    path for path in PUBLIC_RELEASE_DIR.rglob("*")
    if path.is_file()
]

FINAL_PRIVACY_AUDIT_DF = pd.DataFrame([
    {
        "Relative_Path": str(path.relative_to(PUBLIC_RELEASE_DIR)),
        "Restricted_Filename_Pattern": contains_private_pattern(path),
        "File_Size_Bytes": path.stat().st_size
    }
    for path in PUBLIC_RELEASE_FILES
])

TEXT_FILE_SUFFIXES = {
    ".md", ".txt", ".cff", ".json", ".csv"
}

PLACEHOLDER_SEARCH_TERMS = [
    "ADD_GITHUB_REPOSITORY_URL_AFTER_CREATION",
    "ADD_ZENODO_DOI_AFTER_ARCHIVING"
]

PLACEHOLDER_AUDIT_ROWS = []

for path in PUBLIC_RELEASE_FILES:
    if path.suffix.lower() not in TEXT_FILE_SUFFIXES:
        continue

    try:
        file_text = path.read_text(encoding="utf-8")
    except UnicodeDecodeError:
        continue

    for placeholder in PLACEHOLDER_SEARCH_TERMS:
        PLACEHOLDER_AUDIT_ROWS.append({
            "Relative_Path": str(path.relative_to(PUBLIC_RELEASE_DIR)),
            "Placeholder": placeholder,
            "Present": placeholder in file_text
        })

PLACEHOLDER_AUDIT_DF = pd.DataFrame(
    PLACEHOLDER_AUDIT_ROWS
)

display(FINAL_PRIVACY_AUDIT_DF)
display(
    PLACEHOLDER_AUDIT_DF.loc[
        PLACEHOLDER_AUDIT_DF["Present"]
    ]
)

if FINAL_PRIVACY_AUDIT_DF[
    "Restricted_Filename_Pattern"
].any():
    raise RuntimeError(
        "Restricted file patterns remain in the public release folder."
    )

print(
    "Placeholders are expected before GitHub and Zenodo creation. "
    "They must be replaced before the final public release."
)

##10.16.Public Release Archive

Create a ZIP archive containing the privacy-screened repository documentation, metadata, requirements, and aggregate results.

In [ ]:
# ================================================================
# 10.16 Public Release Archive
# ================================================================

PUBLIC_RELEASE_ZIP = (
    BASE_DIR
    / f"{REPOSITORY_NAME}_v{RELEASE_VERSION}_release-files.zip"
)

if PUBLIC_RELEASE_ZIP.exists():
    PUBLIC_RELEASE_ZIP.unlink()

with zipfile.ZipFile(
    PUBLIC_RELEASE_ZIP,
    mode="w",
    compression=zipfile.ZIP_DEFLATED
) as archive:

    for file_path in sorted(PUBLIC_RELEASE_DIR.rglob("*")):
        if not file_path.is_file():
            continue

        if contains_private_pattern(file_path):
            continue

        archive.write(
            file_path,
            arcname=(
                Path(REPOSITORY_NAME)
                / file_path.relative_to(PUBLIC_RELEASE_DIR)
            )
        )

print(f"Public release archive created: {PUBLIC_RELEASE_ZIP}")
print(f"Archive size: {PUBLIC_RELEASE_ZIP.stat().st_size / 1024**2:.2f} MB")

##10.17.Final Reproducibility Status

Summarize the completion status of the computational workflow and identify the remaining actions required before GitHub and Zenodo release.

In [ ]:
# ================================================================
# 10.17 Final Reproducibility Status
# ================================================================

FINAL_REPRODUCIBILITY_STATUS_DF = pd.DataFrame([
    {
        "Item": "Required analysis objects available",
        "Status": bool(OBJECT_VALIDATION_DF["Available"].all()),
        "Action_Required": (
            "None" if OBJECT_VALIDATION_DF["Available"].all()
            else "Run or correct missing analysis sections"
        )
    },
    {
        "Item": "Key manuscript counts matched",
        "Status": bool(MANUSCRIPT_VALUE_CHECK_DF["Passed"].all()),
        "Action_Required": (
            "None" if MANUSCRIPT_VALUE_CHECK_DF["Passed"].all()
            else "Resolve differences before release"
        )
    },
    {
        "Item": "Restricted filenames absent from public folder",
        "Status": not bool(
            FINAL_PRIVACY_AUDIT_DF[
                "Restricted_Filename_Pattern"
            ].any()
        ),
        "Action_Required": "Perform manual content-level privacy review"
    },
    {
        "Item": "GitHub URL finalized",
        "Status": (
            GITHUB_REPOSITORY_URL
            != "ADD_GITHUB_REPOSITORY_URL_AFTER_CREATION"
        ),
        "Action_Required": "Create repository and replace placeholder"
    },
    {
        "Item": "Zenodo DOI finalized",
        "Status": ZENODO_DOI != "ADD_ZENODO_DOI_AFTER_ARCHIVING",
        "Action_Required": "Create GitHub release, archive it, and replace DOI"
    },
    {
        "Item": "Software license added",
        "Status": (PUBLIC_RELEASE_DIR / "LICENSE").exists(),
        "Action_Required": "Select and add a LICENSE file"
    },
    {
        "Item": "Notebook added to public release",
        "Status": (
            PUBLIC_RELEASE_DIR
            / "Olympic_Medal_ML_Public_Reproducibility_Code.ipynb"
        ).exists(),
        "Action_Required": (
            "Save the final Colab notebook to the GitHub repository"
        )
    }
])

display(FINAL_REPRODUCIBILITY_STATUS_DF)

FINAL_REPRODUCIBILITY_STATUS_DF.to_csv(
    PUBLIC_METADATA_DIR / "final_reproducibility_status.csv",
    index=False
)

print("=" * 80)
print("PUBLIC REPRODUCIBILITY NOTEBOOK CODE COMPLETE")
print("=" * 80)
print(f"Analysis output directory: {OUTPUT_DIR}")
print(f"Public release directory: {PUBLIC_RELEASE_DIR}")
print(f"Release archive: {PUBLIC_RELEASE_ZIP}")
print()
print("The repository must not be released until:")
print("1. the notebook has been fully executed without errors;")
print("2. all numerical results have been compared with the manuscript;")
print("3. all public files have passed a manual privacy review;")
print("4. a software license has been added; and")
print("5. GitHub and Zenodo placeholders have been replaced.")
print("=" * 80)